# EXP-016 GPU A/B — physical sampling (treatment)

唯一变量: `--sampling-mode physical`（vs EXP-013 control **0.716** legacy）。
固定: EXP-013 多源标签 + conf-weight loss + Axial fold0 + n_clips=4 + 3ep + batch8 + seed42。
GPU 本周仅 ~4h → 只跑 Axial（S+C+A 会超预算）。**不自动提交。**


In [ ]:
# 提交模式:禁网(enable_internet:false),torch 2.2.0+cu118 + nvidia-cu11 依赖全部离线安装
# 坑1:Kaggle 挂载把文件名中 '+' 去掉(2.2.0cu118=非法 PEP440)→ 复制为合法名
# 坑2:torch 2.2.0+cu118 的 CUDA 库在 nvidia-*-cu11 依赖包中(不在 torch wheel 内),
#     只装 torch 会缺 libcudart(v8/v9 实证)→ 必须显式安装全部 13 个 wheel
# 坑3:/kaggle/working 会被收集为 kernel 输出 → 装完立即删除
import subprocess, sys, os, glob, shutil
wh = sorted(glob.glob('/kaggle/input/**/*.whl', recursive=True))
print('挂载 wheel 共', len(wh), '个')
for w in wh: print('  ', os.path.basename(w))
assert len(wh) >= 13, 'FAIL: 离线 wheel 不足(应 13: torch+torchvision+11 nvidia-cu11)'
fixed = []
for w in wh:
    base = os.path.basename(w)
    dst = os.path.join('/kaggle/working', base.replace('cu118', '+cu118', 1) if 'cu118' in base else base)
    shutil.copy(w, dst)
    fixed.append(dst)
print('安装 wheel:', [os.path.basename(f) for f in fixed])
subprocess.check_call([sys.executable,"-m","pip","uninstall","-y","-q","torch","torchvision","torchaudio"])
# 显式安装全部 wheel(--no-deps --no-index: 无版本解析/无联网)
subprocess.check_call([sys.executable,"-m","pip","install","-q","--no-deps","--no-index"] + fixed)
for f in fixed: os.remove(f)
import numpy, torch, torchvision, sklearn, pandas
print("torch", torch.__version__, "cuda", torch.version.cuda, "| numpy", numpy.__version__)
assert torch.cuda.is_available()
print("GPU:", torch.cuda.get_device_name(0))


In [ ]:
# preflight: competition mount + Axial coverage
# Correct layout: test_series/<StudyInstanceUID>/<SeriesInstanceUID>/*.dcm
import os, csv
from collections import defaultdict

base = '/kaggle/input'

def _find(b, depth=0, max_depth=3):
    if not os.path.isdir(b):
        return None
    try:
        entries = sorted(os.listdir(b))
    except OSError:
        return None
    for e in entries:
        p = os.path.join(b, e)
        if not os.path.isdir(p):
            continue
        if e in ('train_series', 'test_series'):
            continue
        if os.path.isfile(os.path.join(p, 'test.csv')) and os.path.isdir(os.path.join(p, 'test_series')):
            return p
        if depth < max_depth - 1:
            r = _find(p, depth + 1)
            if r:
                return r
    return None

root = _find(base)
if root is None:
    raise SystemExit('FAIL: 未找到含 test_series+test.csv 的比赛挂载')
print('root =', root)

def count_dcm(dirpath):
    if not os.path.isdir(dirpath):
        return -1
    return sum(1 for f in os.listdir(dirpath) if f.lower().endswith('.dcm'))

def plane_stats(series_csv, series_dir, label):
    if not os.path.isfile(series_csv):
        print(label, 'series csv missing', series_csv)
        return
    with open(series_csv, newline='', encoding='utf-8') as f:
        rows = list(csv.DictReader(f))
    if not rows:
        raise SystemExit(f'FAIL: {label} series csv empty')
    plane_key = None
    for k in rows[0].keys():
        if 'plane' in k.lower() or 'anatom' in k.lower():
            plane_key = k
            break
    uid_key = 'StudyInstanceUID'
    ser_key = 'SeriesInstanceUID'
    series_counts = defaultdict(int)
    study_sets = defaultdict(set)
    axial_series_rows = []
    for r in rows:
        pl = r.get(plane_key, '?') if plane_key else '?'
        series_counts[pl] += 1
        if uid_key in r:
            study_sets[pl].add(r[uid_key])
        if pl == 'Axial' and ser_key in r:
            axial_series_rows.append(r)
    print(label, 'plane series counts', dict(series_counts))
    print(label, 'plane study counts', {k: len(v) for k, v in study_sets.items()})
    print(label, 'Axial studies', len(study_sets.get('Axial', set())))
    assert len(study_sets.get('Axial', set())) > 0, f'FAIL: {label} 无 Axial study'

    # Walk both possible layouts
    n_dcm = 0
    hits = 0
    misses = []
    for r in axial_series_rows:
        suid = r[uid_key]
        seid = r[ser_key]
        candidates = [
            os.path.join(series_dir, suid, seid),
            os.path.join(series_dir, seid),
            os.path.join(series_dir, suid, 'series', seid),
        ]
        found = None
        for c in candidates:
            k = count_dcm(c)
            if k and k > 0:
                found = c
                n_dcm += k
                hits += 1
                print(f'  axial OK study={suid[:14]} series={seid[:16]} dcm={k} path={c}')
                break
        if found is None:
            # fallback: list study dir
            sdir = os.path.join(series_dir, suid)
            if os.path.isdir(sdir):
                children = sorted(os.listdir(sdir))[:8]
                print(f'  axial miss study={suid[:14]} series={seid[:16]} study_children={children}')
            else:
                print(f'  axial miss study={suid[:14]} series={seid[:16]} NO_STUDY_DIR {sdir}')
            misses.append(seid)
    print(label, f'axial series hits={hits}/{len(axial_series_rows)} dcm_total={n_dcm}')
    assert hits > 0 and n_dcm > 0, f'FAIL: {label} Axial DICOM 为空 (hits={hits}, dcm={n_dcm})'

    # also sample sagittal for sanity
    sag = [r for r in rows if (r.get(plane_key) if plane_key else '') == 'Sagittal']
    if sag:
        r0 = sag[0]
        for c in [os.path.join(series_dir, r0[uid_key], r0[ser_key]), os.path.join(series_dir, r0[ser_key])]:
            k = count_dcm(c)
            if k and k > 0:
                print(label, 'sagittal sanity', r0[ser_key][:16], 'dcm', k)
                break

plane_stats(os.path.join(root, 'test_series.csv'), os.path.join(root, 'test_series'), 'test')
plane_stats(os.path.join(root, 'train_series.csv'), os.path.join(root, 'train_series'), 'train')
print('preflight axial: OK')


In [ ]:
import base64, os
# embed rsna_v3.py
parts=[]
parts.append("IyAtKi0gY29kaW5nOiB1dGYtOCAtKi0NCiIiIg0KUlNOQSBLbmVlIEFibm9ybWFsaXR5IERldGVjdGlvbiDigJQgdjIo5oyJIEFuYXRvbWljYWxfUGxhbmUg5YiG5bmz6Z2i54us56uL5bu65qihICsg6J6N5ZCIKQ0KPT09PT09PT09PT09PT09PT09PT09PT09PT09")
parts.append("PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT0NCuebuOWvuSBiYXNlbGluZShyc25hX2Jhc2VsaW5lLnB5KeeahOaguOW/g+aUueWKqDoNCi0gYmFzZWxpbmUg5oqKIHN0dWR5IOWGheaJgOacieW5s+mdoueahCBzZXJp")
parts.append("ZXMg57KX5pq05ou85oiQ6YCa6YGTO3YyIOaMiSBBbmF0b21pY2FsX1BsYW5lDQogIChBeGlhbCAvIFNhZ2l0dGFsIC8gQ29yb25hbCnlkITorq3kuIDkuKrni6znq4sgUmVzTmV0MzQs5bmz6Z2i5YaF6YOo5YaN5YGaIDIuNUQgY2xpcDsNCi0gdGVzdCDmjqjnkIbm")
parts.append("l7bmr4/kuKrlubPpnaLmqKHlnovni6znq4vpooTmtYss5pyA57uI5oyJIOWQhOW5s+mdoiBPT0Yg5a6P5bmz5Z2HIEFVQyDliqDmnYPono3lkIgNCiAgKOadg+mHjeWPryAtLWZ1c2Ugd2VpZ2h0IOeUqCBBVUMg5Yqg5p2DLOaIliAtLWZ1c2UgbWVhbiDnroDljZXl")
parts.append("ubPlnYcpOw0KLSDmn5Agc3R1ZHkg57y65p+Q5bmz6Z2i5pe2LOiejeWQiOWPquWvueivpSBzdHVkeSDlrp7pmYXlj6/nlKjnmoTlubPpnaLlvZLkuIDljJbmnYPph43jgIINCg0K5YW25L2Z6YC76L6RKOi9r+agh+etvuWhq+WFheOAgUdyb3VwS0ZvbGTjgIF2YWxp")
parts.append("ZCDmjqnnoIHmjZ/lpLHjgIFESUNPTSDpooTlpITnkIYp5LiOIGJhc2VsaW5lIOS4gOiHtOOAgg0KDQrov5DooYznjq/looM6S2FnZ2xlIE5vdGVib29rKEdQVSzmjILovb3mr5TotZvmlbDmja4gKyDnpL7ljLrova/moIfnrb7mlbDmja7pm4Yp44CCDQrnlKjms5U6")
parts.append("DQogICAgIXB5dGhvbiByc25hX3YyLnB5IC0tZXBvY2hzIDMgLS1iYXRjaCAxNiAtLW4tY2xpcHMgMiAtLXBsYW5lcyBzYWdpdHRhbCxjb3JvbmFsDQogICAgIXB5dGhvbiByc25hX3YyLnB5IC0tZXBvY2hzIDMgLS1iYXRjaCAxNiAtLWZ1c2Ugd2VpZ2h0IC0tcHNl")
parts.append("dWRvLWNvbmYgMC4zDQrovpPlh7o6IHN1Ym1pc3Npb24uY3N2KFN0dWR5SW5zdGFuY2VVSUQgKyAxMiDliJfnva7kv6HliIbmlbApDQoiIiINCmltcG9ydCBhcmdwYXJzZQ0KaW1wb3J0IGpzb24NCmltcG9ydCBvcw0KaW1wb3J0IHN5cw0KaW1wb3J0IHRpbWUNCmlt")
parts.append("cG9ydCB3YXJuaW5ncw0KZnJvbSBwYXRobGliIGltcG9ydCBQYXRoDQoNCmltcG9ydCBudW1weSBhcyBucA0KaW1wb3J0IHBhbmRhcyBhcyBwZA0KDQp0cnk6DQogICAgaW1wb3J0IHRvcmNoDQogICAgaW1wb3J0IHRvcmNoLm5uIGFzIG5uDQogICAgZnJvbSB0b3Jj")
parts.append("aC51dGlscy5kYXRhIGltcG9ydCBEYXRhc2V0LCBEYXRhTG9hZGVyDQogICAgZnJvbSB0b3JjaC5vcHRpbSBpbXBvcnQgQWRhbVcNCiAgICBmcm9tIHRvcmNoLm9wdGltLmxyX3NjaGVkdWxlciBpbXBvcnQgQ29zaW5lQW5uZWFsaW5nTFINCiAgICBIQVNfVE9SQ0gg")
parts.append("PSBUcnVlDQpleGNlcHQgSW1wb3J0RXJyb3I6DQogICAgSEFTX1RPUkNIID0gRmFsc2UNCg0KdHJ5Og0KICAgIGltcG9ydCB0b3JjaHZpc2lvbi5tb2RlbHMgYXMgdHZtDQogICAgSEFTX1RWID0gVHJ1ZQ0KZXhjZXB0IEltcG9ydEVycm9yOg0KICAgIEhBU19UViA9")
parts.append("IEZhbHNlDQoNCnRyeToNCiAgICBpbXBvcnQgcHlkaWNvbQ0KICAgIEhBU19QWURJQ09NID0gVHJ1ZQ0KZXhjZXB0IEltcG9ydEVycm9yOg0KICAgIEhBU19QWURJQ09NID0gRmFsc2UNCg0KdHJ5Og0KICAgIGltcG9ydCByc25hX3JlcG9ydF9sYWJlbHMgYXMgUkwg")
parts.append("ICAjIOaKpeWRiuKGkjEyIOeXheeBtuW8seagh+etvijkuI4gcnNuYV92My5weSDlkIznm67lvZUpDQogICAgSEFTX1JMID0gVHJ1ZQ0KZXhjZXB0IEltcG9ydEVycm9yOg0KICAgIEhBU19STCA9IEZhbHNlDQoNCnRyeToNCiAgICBpbXBvcnQgc2FtcGxpbmdfZ2Vv")
parts.append("bWV0cnkgYXMgU0cgICMgRVhQLTAxNiBwaHlzaWNhbCBzb3J0IC8gMi41RCBzYW1wbGluZw0KICAgIEhBU19TRyA9IFRydWUNCmV4Y2VwdCBJbXBvcnRFcnJvcjoNCiAgICBIQVNfU0cgPSBGYWxzZQ0KDQpmcm9tIHNrbGVhcm4ubW9kZWxfc2VsZWN0aW9uIGltcG9y")
parts.append("dCBHcm91cEtGb2xkDQpmcm9tIHNrbGVhcm4ubWV0cmljcyBpbXBvcnQgcm9jX2F1Y19zY29yZQ0KDQp3YXJuaW5ncy5maWx0ZXJ3YXJuaW5ncygiaWdub3JlIikNCg0KTEFCRUxTID0gWyJBQ0wiLCAiTUNMIiwgIk1lZGlhbCBNZW5pc2N1cyIsICJMYXRlcmFsIE1l")
parts.append("bmlzY3VzIiwNCiAgICAgICAgICAiTWVkaWFsIE9BIiwgIkxhdGVyYWwgT0EiLCAiUEYgT0EiLCAiRWZmdXNpb24iLCAiU3lub3ZpdGlzIiwNCiAgICAgICAgICAiQmFrZXIncyIsICJDb250dXNpb24iLCAiRnJhY3R1cmUiXQ0KTEFCRUxTX0VOID0gWyJBQ0wiLCAi")
parts.append("TUNMIiwgIk1lZGlhbE1lbmlzY3VzIiwgIkxhdGVyYWxNZW5pc2N1cyIsDQogICAgICAgICAgICAgIk1lZGlhbE9BIiwgIkxhdGVyYWxPQSIsICJQRk9BIiwgIkVmZnVzaW9uIiwgIlN5bm92aXRpcyIsDQogICAgICAgICAgICAgIkJha2VycyIsICJDb250dXNpb24i")
parts.append("LCAiRnJhY3R1cmUiXSAgIyDliJflkI3lronlhajljJYo5YaZ5o+Q5Lqk55SoKQ0KUExBTkVTID0gWyJBeGlhbCIsICJTYWdpdHRhbCIsICJDb3JvbmFsIl0NCg0KSU1HX1NJWkUgPSAyMjQNCkNMSVAgPSAzICAgICAgICAgICMg5q+PIGNsaXAg5YiH54mH5pWwKHBy")
parts.append("ZXYvY2VudGVyL25leHQpDQpNQVhfU0VSSUVTID0gNCAgICAjIOavjyBzdHVkeSDmr4/kuKrlubPpnaLmnIDlpJrlj5bnmoQgc2VyaWVzIOaVsCh2MiDmjInlubPpnaLlj5Ys5Y2V5bmz6Z2iIHNlcmllcyDmm7TlsJEsNCDotrPlpJ8pDQpGT0xEUyA9IDUNClNFRUQg")
parts.append("PSA0Mg0KDQoNCmRlZiByZXNvbHZlX3BhdGhzKCk6DQogICAgIiIi5pWw5o2u5a6a5L2N6aG65bqPOjEpIC9rYWdnbGUvaW5wdXQg5oyC6L29OzIpIGthZ2dsZWh1YiDoh6rliqjkuIvovb07Mykg5pys5ZywIHJzbmFfa25lZS8g55uu5b2V44CCIiIiDQogICAgIyAx")
parts.append("KSBLYWdnbGUg5oyC6L29KOWFvOWuueWkmuWxguaMgui9vTova2FnZ2xlL2lucHV0LzzmlbDmja7lkI0+LyDmiJYgL2thZ2dsZS9pbnB1dC885YiG57G7Pi885pWw5o2u5ZCNPi8pDQogICAgaWYgb3MucGF0aC5pc2RpcigiL2thZ2dsZS9pbnB1dCIpOg0KICAgICAg")
parts.append("ICBmb3Igc3ViIGluIHNvcnRlZChvcy5saXN0ZGlyKCIva2FnZ2xlL2lucHV0IikpOg0KICAgICAgICAgICAgZCA9IG9zLnBhdGguam9pbigiL2thZ2dsZS9pbnB1dCIsIHN1YikNCiAgICAgICAgICAgIGZvciBjYW5kIGluIChkLCBvcy5wYXRoLmpvaW4oZCwgc3Vi")
parts.append("KSk6ICAjIOWkhOeQhiBLYWdnbGUg5Y+M5bGC5oyC6L29DQogICAgICAgICAgICAgICAgaWYgb3MucGF0aC5leGlzdHMob3MucGF0aC5qb2luKGNhbmQsICJ0cmFpbi5jc3YiKSk6DQogICAgICAgICAgICAgICAgICAgIHJldHVybiBjYW5kLCAiL2thZ2dsZS93b3Jr")
parts.append("aW5nIg0KICAgICAgICAgICAgaWYgb3MucGF0aC5pc2RpcihkKTogICMg5YaN5b6A5LiL5LiA5bGC5om+KOWmgiAva2FnZ2xlL2lucHV0L2NvbXBldGl0aW9ucy88bmFtZT4vKQ0KICAgICAgICAgICAgICAgIGZvciBzdWIyIGluIHNvcnRlZChvcy5saXN0ZGlyKGQp")
parts.append("KToNCiAgICAgICAgICAgICAgICAgICAgY2FuZCA9IG9zLnBhdGguam9pbihkLCBzdWIyKQ0KICAgICAgICAgICAgICAgICAgICBpZiBvcy5wYXRoLmV4aXN0cyhvcy5wYXRoLmpvaW4oY2FuZCwgInRyYWluLmNzdiIpKToNCiAgICAgICAgICAgICAgICAgICAgICAg")
parts.append("IHJldHVybiBjYW5kLCAiL2thZ2dsZS93b3JraW5nIg0KICAgICAgICBwcmludCgiW+iviuaWrV0gL2thZ2dsZS9pbnB1dCDlrZjlnKgs5L2G5rKh5om+5Yiw5ZCrIHRyYWluLmNzdiDnmoTnm67lvZXjgILlrp7pmYXmjILovb3lhoXlrrnlpoLkuIs6IiwgZmx1c2g9")
parts.append("VHJ1ZSkNCiAgICAgICAgZm9yIHN1YiBpbiBzb3J0ZWQob3MubGlzdGRpcigiL2thZ2dsZS9pbnB1dCIpKToNCiAgICAgICAgICAgIHAgPSBvcy5wYXRoLmpvaW4oIi9rYWdnbGUvaW5wdXQiLCBzdWIpDQogICAgICAgICAgICB0cnk6DQogICAgICAgICAgICAgICAg")
parts.append("aW5uZXIgPSBvcy5saXN0ZGlyKHApDQogICAgICAgICAgICBleGNlcHQgT1NFcnJvcjoNCiAgICAgICAgICAgICAgICBpbm5lciA9IFsiKOaXoOazleivu+WPlikiXQ0KICAgICAgICAgICAgcHJpbnQoZiIgIC9rYWdnbGUvaW5wdXQve3N1Yn0vICAtPiAge2lubmVy")
parts.append("Wzo4XX0iLCBmbHVzaD1UcnVlKQ0KICAgICAgICBwcmludCgiW+iviuaWrV0g5aaC5p6c5L2g5oyC55qE5pivICfkvZzogIXmlbDmja7ljIUv5Yir5Lq655qEIGtlcm5lbCDovpPlh7onLCDph4zpnaLmsqHmnIkgdHJhaW4uY3N2IOWSjCB0cmFpbl9zZXJpZXMvLCIs")
parts.append("DQogICAgICAgICAgICAgICLlv4XpobvmjILmr5TotZvmlbDmja7mnKzkvZMgcnNuYS1rbmVlLWFibm9ybWFsaXR5LWRldGVjdGlvbiAoQWRkIElucHV0IC0+IENvbXBldGl0aW9ucyDmoIfnrb7pobUpIiwgZmx1c2g9VHJ1ZSkNCiAgICAjIDIpIGthZ2dsZWh1YiDl")
parts.append("hZzlupU65pyq5oyC6L295pe26Ieq5Yqo5LiL6L295q+U6LWb5pWw5o2uKEthZ2dsZSBOb3RlYm9vayDlhoXnva7lh63mja4pDQogICAgdHJ5Og0KICAgICAgICBpbXBvcnQga2FnZ2xlaHViDQogICAgICAgIGQgPSBrYWdnbGVodWIuY29tcGV0aXRpb25fZG93bmxv")
parts.append("YWQoInJzbmEta25lZS1hYm5vcm1hbGl0eS1kZXRlY3Rpb24iKQ0KICAgICAgICBpZiBvcy5wYXRoLmV4aXN0cyhvcy5wYXRoLmpvaW4oZCwgInRyYWluLmNzdiIpKToNCiAgICAgICAgICAgIHByaW50KGYiW+i3r+W+hF0g5pyq5oyC6L29LOW3sueUqCBrYWdnbGVo")
parts.append("dWIg6I635Y+W5pWw5o2uOiB7ZH0iLCBmbHVzaD1UcnVlKQ0KICAgICAgICAgICAgcmV0dXJuIGQsICIva2FnZ2xlL3dvcmtpbmciDQogICAgZXhjZXB0IEltcG9ydEVycm9yOg0KICAgICAgICBwYXNzICAjIOacrOWcsOacquijhSBrYWdnbGVodWINCiAgICBleGNl")
parts.append("cHQgRXhjZXB0aW9uIGFzIGU6DQogICAgICAgIHByaW50KGYiW+aPkOekul0ga2FnZ2xlaHViIOS4i+i9veaVsOaNruWksei0pToge2V9IiwgZmx1c2g9VHJ1ZSkNCiAgICAjIDMpIOacrOWcsOWbnumAgCjohJrmnKznm67lvZUgLyBjd2QgLyBjd2Qg5LiL55qEIHJz")
parts.append("bmFfa25lZSDlrZDnm67lvZXpg73mib4pDQogICAgY2FuZHMgPSBbXQ0KICAgIHRyeToNCiAgICAgICAgY2FuZHMuYXBwZW5kKG9zLnBhdGguZGlybmFtZShvcy5wYXRoLmFic3BhdGgoX19maWxlX18pKSkNCiAgICBleGNlcHQgTmFtZUVycm9yOg0KICAgICAgICBw")
parts.append("YXNzDQogICAgY2FuZHMuYXBwZW5kKG9zLmdldGN3ZCgpKQ0KICAgIGZvciBiYXNlIGluIGxpc3QoY2FuZHMpOg0KICAgICAgICBjYW5kcy5hcHBlbmQob3MucGF0aC5qb2luKGJhc2UsICJyc25hX2tuZWUiKSkNCiAgICBzZWVuID0gc2V0KCkNCiAgICBmb3IgYmFz")
parts.append("ZSBpbiBjYW5kczoNCiAgICAgICAgaWYgYmFzZSBpbiBzZWVuOg0KICAgICAgICAgICAgY29udGludWUNCiAgICAgICAgc2Vlbi5hZGQoYmFzZSkNCiAgICAgICAgaWYgb3MucGF0aC5leGlzdHMob3MucGF0aC5qb2luKGJhc2UsICJ0cmFpbi5jc3YiKSk6DQogICAg")
parts.append("ICAgICAgICByZXR1cm4gYmFzZSwgYmFzZQ0KICAgIHN5cy5leGl0KCLmib7kuI3liLDmlbDmja466K+35YWI5Zyo5q+U6LWb6aG15Yqg5YWl5q+U6LWb5bm25ZCM5oSP5pWw5o2u6KeE5YiZLOWGjSBBZGQgSW5wdXQg5oyC6L29ICINCiAgICAgICAgICAgICAicnNu")
parts.append("YS1rbmVlLWFibm9ybWFsaXR5LWRldGVjdGlvbjvmnKzlnLDor7flnKggcnNuYV9rbmVlLyDkuIvov5DooYzjgIIiKQ0KDQoNCl9QU0VVRE9fRklMRVMgPSAoInRyYWluX2ZvbGRzX3dpdGhfcHNldWRvLmNzdiIsICJ0cmFpbl93aXRoX3BzZXVkby5jc3YiLA0KICAg")
parts.append("ICAgICAgICAgICAgICAidHJhaW5fd2l0aF9zb2Z0X2xhYmVscy5jc3YiLCAidHJhaW5fcHNldWRvLmNzdiIpDQpfU0tJUF9ESVJTID0geyJ0cmFpbl9zZXJpZXMiLCAidGVzdF9zZXJpZXMifSAgIyBESUNPTSDlpKfnm67lvZUs57ud5LiN5Y676YGN5Y6GDQpfV0VJ")
parts.append("R0hUX0VYVCA9ICgiLnB0aCIsICIucHQiLCAiLmNrcHQiLCAiLnB0aC50YXIiKSAgIyDnprvnur/pooTorq3nu4PmnYPph43mlofku7bmianlsZXlkI0NCg0KIyAxMiDmoIfnrb7lpJror63oqIDlhbPplK7or43or43lhbgo6IuxL+ilvy/ojbcv5b63L+azlSnigJTi")
parts.append("gJTpm7bmjILovb3ova/moIfnrb7lhZzlupUNCiMg5ZG95LitIOKGkiDpq5jnva7kv6Es5pyq5ZG95LitIOKGkiDkvY7nva7kv6Eo6L2v5qCH562+LOS4jee7meehrCAwLzEs6YG/5YWN6ZSZ6K+N6K+v5a+8KQ0KX1JFUE9SVF9LRVlXT1JEUyA9IHsNCiAgICAiQUNM")
parts.append("IjogWyJhY2wiLCAiYW50ZXJpb3IgY3J1Y2lhdGUiLCAiY3J1Y2lhZG8gYW50ZXJpb3IiLCAidm9vcnN0ZSBrcnVpc2JhbmQiLA0KICAgICAgICAgICAgInZvcmRlcmVuIGtyZXV6YmFuZCIsICJsY2EiLCAibGlnYW1lbnQgY3JvaXPDqSBhbnTDqXJpZXVyIl0sDQog")
parts.append("ICAgIk1DTCI6IFsibWNsIiwgIm1lZGlhbCBjb2xsYXRlcmFsIiwgImNvbGF0ZXJhbCBtZWRpYWwiLCAibWVkaWFsZSBjb2xsYXRlcmFsZSIsDQogICAgICAgICAgICAiaW5uZXJlcyBzZWl0ZW5iYW5kIiwgImxpZ2FtZW50IGNvbGxhdMOpcmFsIG3DqWRpYWwiXSwN")
parts.append("CiAgICAiTWVkaWFsIE1lbmlzY3VzIjogWyJtZWRpYWwgbWVuaXNjdXMiLCAibWVuaXNjbyBpbnRlcm5vIiwgIm1lbmlzY3VzIG1lZGlhbGlzIiwNCiAgICAgICAgICAgICAgICAgICAgICAgICJtZW5pc2t1cyBtZWRpYWwiLCAibWVuaXNxdWUgbcOpZGlhbCIsICJp")
parts.append("bm5lciBtZW5pc2N1cyIsICJtZW5pc2NvIG1lZGlhbCJdLA0KICAgICJMYXRlcmFsIE1lbmlzY3VzIjogWyJsYXRlcmFsIG1lbmlzY3VzIiwgIm1lbmlzY28gZXh0ZXJubyIsICJtZW5pc2N1cyBsYXRlcmFsaXMiLA0KICAgICAgICAgICAgICAgICAgICAgICAgICJt")
parts.append("ZW5pc2t1cyBsYXRlcmFsIiwgIm1lbmlzcXVlIGxhdMOpcmFsIiwgIm91dGVyIG1lbmlzY3VzIl0sDQogICAgIk1lZGlhbCBPQSI6IFsibWVkaWFsIG9zdGVvYXJ0aHJpdGlzIiwgImFydHJvc2lzIGZlbW9yb3RpYmlhbCBtZWRpYWwiLA0KICAgICAgICAgICAgICAg")
parts.append("ICAgIm1lZGlhbGUgZ29uYXJ0cm9zZSIsICJtZWRpYWxlIGFydGhyb3NlIiwgIm9zdGVvYXJ0cml0aXMgbWVkaWFsIiwNCiAgICAgICAgICAgICAgICAgICJtZWRpYWwgY29tcGFydG1lbnQgbmFycm93aW5nIiwgIm1lZGlhbCBqb2ludCBzcGFjZSBuYXJyb3dpbmci")
parts.append("XSwNCiAgICAiTGF0ZXJhbCBPQSI6IFsibGF0ZXJhbCBvc3Rlb2FydGhyaXRpcyIsICJhcnRyb3NpcyBmZW1vcm90aWJpYWwgbGF0ZXJhbCIsDQogICAgICAgICAgICAgICAgICAgImxhdGVyYWxlIGdvbmFydHJvc2UiLCAibGF0ZXJhbGUgYXJ0aHJvc2UiLCAibGF0")
parts.append("ZXJhbCBqb2ludCBzcGFjZSBuYXJyb3dpbmciXSwNCiAgICAiUEYgT0EiOiBbInBhdGVsbG9mZW1vcmFsIG9zdGVvYXJ0aHJpdGlzIiwgImFydHJvc2lzIHBhdGVsb2ZlbW9yYWwiLA0KICAgICAgICAgICAgICAicGF0ZWxsb2ZlbW9yYWxlIGFydHJvc2UiLCAicGF0")
parts.append("ZWxsb2ZlbW9yYWwgYXJ0aHJvc2UiLCAicGYgb2EiLA0KICAgICAgICAgICAgICAicGF0ZWxsb2ZlbW9yYWwgbmFycm93aW5nIiwgInBhdGVsbG9mZW1vcmFsIGpvaW50IHNwYWNlIl0sDQogICAgIkVmZnVzaW9uIjogWyJlZmZ1c2lvbiIsICJkZXJyYW1lIiwgInZv")
parts.append("Y2h0IiwgImVyZ3VzcyIsICLDqXBhbmNoZW1lbnQiLCAiam9pbnQgZWZmdXNpb24iXSwNCiAgICAiU3lub3ZpdGlzIjogWyJzeW5vdml0aXMiLCAic2lub3ZpdGlzIiwgInN5bm92aWFsZSIsICJzeW5vdmlhbCB0aGlja2VuaW5nIiwgInN5bm92aXRpcyJdLA0KICAg")
parts.append("ICJCYWtlcidzIjogWyJiYWtlciIsICJwb3BsaXRlYWwgY3lzdCIsICJxdWlzdGUgZGUgYmFrZXIiLCAiYmFrZXIgY3lzdCJdLA0KICAgICJDb250dXNpb24iOiBbImNvbnR1c2lvbiIsICJib25lIGJydWlzZSIsICJjb250dXNpw7NuIiwgImtuZXV6aW5nIiwgImtv")
parts.append("bnR1c2lvbiIsDQogICAgICAgICAgICAgICAgICAibWFycm93IGVkZW1hIiwgIsOzc2VvIiwgImJvbmUgYnJ1aXNpbmciXSwNCiAgICAiRnJhY3R1cmUiOiBbImZyYWN0dXJlIiwgImZyYWN0dXJhIiwgImZyYWN0dXVyIiwgImZyYWt0dXIiLCAiZnJhY3R1cmUgbGlu")
parts.append("ZSIsDQogICAgICAgICAgICAgICAgICJmcmFjdHVyZSBsaW5lIiwgInN1YmNob25kcmFsIGZyYWN0dXJlIl0sDQp9DQoNCg0KZGVmIF9wc2V1ZG9fZnJvbV9yZXBvcnRzKHRyYWluX2NzdiwgY29uZl9oaXQ9MC45LCBjb25mX21pc3M9MC4xKToNCiAgICAiIiLku47m")
parts.append("lL7lsITmiqXlkYrmlofmnKznlKjlpJror63oqIDlhbPplK7or43nlJ/miJAgMTIg5qCH562+6L2v5qCH562+KOmbtuaMgui9veWFnOW6lSnjgIINCg0KICAgIOi/lOWbniBEYXRhRnJhbWUoU3R1ZHlJbnN0YW5jZVVJRCArIHBzZXVkb18qIOWIlynjgILlhbPplK7o")
parts.append("r43lkb3kuK3ihpJjb25mX2hpdCwNCiAgICDmnKrlkb3kuK3ihpJjb25mX21pc3Mo5LiN57uZ56GsIDAvMSzpmY3kvY7plJnor43or6/lr7wp44CC5a6Y5pa55bey5pyJ5qCH5rOo55qEIHN0dWR5DQogICAg5LuN5Lul5a6Y5pa55Li65YeGKGxvYWRfbGFiZWxzIOea")
parts.append("hOWhq+WFhemAu+i+keS/neivgeWumOaWueS8mOWFiCnjgIINCiAgICAiIiINCiAgICBpZiAiUmVwb3J0IiBub3QgaW4gdHJhaW5fY3N2LmNvbHVtbnM6DQogICAgICAgIHJldHVybiBOb25lDQogICAgcmVwID0gdHJhaW5fY3N2WyJSZXBvcnQiXS5maWxsbmEoIiIp")
parts.append("DQogICAgcm93cyA9IHt9DQogICAgZm9yIGxhYiwga3dzIGluIF9SRVBPUlRfS0VZV09SRFMuaXRlbXMoKToNCiAgICAgICAgbG93ID0gW2subG93ZXIoKSBmb3IgayBpbiBrd3NdDQogICAgICAgIHJvd3NbZiJwc2V1ZG9fe2xhYn0iXSA9IHJlcC5hcHBseSgNCiAg")
parts.append("ICAgICAgICAgIGxhbWJkYSB0OiBjb25mX2hpdCBpZiBhbnkoayBpbiBzdHIodCkubG93ZXIoKSBmb3IgayBpbiBsb3cpIGVsc2UgY29uZl9taXNzKQ0KICAgIGRmID0gcGQuRGF0YUZyYW1lKHsiU3R1ZHlJbnN0YW5jZVVJRCI6IHRyYWluX2NzdlsiU3R1ZHlJbnN0")
parts.append("YW5jZVVJRCJdLCAqKnJvd3N9KQ0KICAgIHJldHVybiBkZg0KDQoNCmRlZiBfcHJpbnRfbW91bnRfdHJlZShiYXNlPSIva2FnZ2xlL2lucHV0IiwgdGFnPSLmlbDmja4iLCBtYXhfZmlsZXM9Nik6DQogICAgIiIi5omT5Y2w5oyC6L2955uu5b2V5qCRKOacgOWkmuS4")
parts.append("pOWxgizmr4/nm67lvZXliY3lh6DkuKrmlofku7YpLOW4rueUqOaIt+ehruiupOaMgui9veWGheWuueOAgiIiIg0KICAgIGlmIG5vdCBvcy5wYXRoLmlzZGlyKGJhc2UpOg0KICAgICAgICBwcmludChmIlvor4rmlq0te3RhZ31dIOebruW9leS4jeWtmOWcqDoge2Jh")
parts.append("c2V9IiwgZmx1c2g9VHJ1ZSkNCiAgICAgICAgcmV0dXJuDQogICAgcHJpbnQoZiJb6K+K5patLXt0YWd9XSAva2FnZ2xlL2lucHV0IOWunumZheaMgui9veWGheWuuToiLCBmbHVzaD1UcnVlKQ0KICAgIGZvciBzdWIgaW4gc29ydGVkKG9zLmxpc3RkaXIoYmFzZSkp")
parts.append("Og0KICAgICAgICBwID0gb3MucGF0aC5qb2luKGJhc2UsIHN1YikNCiAgICAgICAgdHJ5Og0KICAgICAgICAgICAgaWYgb3MucGF0aC5pc2RpcihwKToNCiAgICAgICAgICAgICAgICBpbm5lciA9IHNvcnRlZChvcy5saXN0ZGlyKHApKVs6bWF4X2ZpbGVzXQ0KICAg")
parts.append("ICAgICAgICAgICAgIHByaW50KGYiICB7c3VifS8gLT4ge2lubmVyfSIsIGZsdXNoPVRydWUpDQogICAgICAgICAgICBlbHNlOg0KICAgICAgICAgICAgICAgIHByaW50KGYiICB7c3VifSAo5paH5Lu2KSIsIGZsdXNoPVRydWUpDQogICAgICAgIGV4Y2VwdCBPU0Vy")
parts.append("cm9yOg0KICAgICAgICAgICAgcHJpbnQoZiIgIHtzdWJ9ICjml6Dms5Xor7vlj5YpIiwgZmx1c2g9VHJ1ZSkNCg0KDQpkZWYgX3ByaW50X2RjbV90cmVlKHJvb3QsIHRhZz0iRElDT00iLCBtYXhfc2hvdz01KToNCiAgICAiIiLmiZPljbAgdHJhaW5fc2VyaWVzIOeb")
parts.append("ruW9leWJjeWHoOWxguecn+Wunue7k+aehCArIOaKveagtyAuZGNtIOiuoeaVsCznoa7orqQgRElDT00g5biD5bGA44CCDQoNCiAgICBSU05BIOW4uOingeW4g+WxgDp0cmFpbl9zZXJpZXMvPFN0dWR5SW5zdGFuY2VVSUQ+LzxTZXJpZXNJbnN0YW5jZVVJRD4vKi5k")
parts.append("Y20o5Lik5bGCKTsNCiAgICDkuZ/lhbzlrrnkuIDlsYIoPFNlcmllc0luc3RhbmNlVUlEPi8qLmRjbSnmiJbmm7Tmt7HlsYIs6L+Z6YeM5omT5Y2w5a6e6ZmF55yL5Yiw55qE44CCDQogICAgIiIiDQogICAgZGNtID0gb3MucGF0aC5qb2luKHJvb3QsICJ0cmFpbl9z")
parts.append("ZXJpZXMiKQ0KICAgIGlmIG5vdCBvcy5wYXRoLmlzZGlyKGRjbSk6DQogICAgICAgIHByaW50KGYiW+iviuaWrS17dGFnfV0ge2RjbX0g5LiN5a2Y5ZyoISIsIGZsdXNoPVRydWUpDQogICAgICAgIHJldHVybg0KICAgIHRyeToNCiAgICAgICAgc3Vic19hbGwgPSBz")
parts.append("b3J0ZWQob3MubGlzdGRpcihkY20pKQ0KICAgIGV4Y2VwdCBPU0Vycm9yOg0KICAgICAgICBzdWJzX2FsbCA9IFtdDQogICAgc3VicyA9IHN1YnNfYWxsWzptYXhfc2hvd10NCiAgICBwcmludChmIlvor4rmlq0te3RhZ31dIHtkY219IiwgZmx1c2g9VHJ1ZSkNCiAg")
parts.append("ICBwcmludChmIlvor4rmlq0te3RhZ31dIOWtkOebruW9lS/mlofku7bmlbA6IHtsZW4oc3Vic19hbGwpfSzmoLfmnKw6IiwgZmx1c2g9VHJ1ZSkNCiAgICBmb3IgcyBpbiBzdWJzOg0KICAgICAgICBzcCA9IG9zLnBhdGguam9pbihkY20sIHMpDQogICAgICAgIGlm")
parts.append("IG9zLnBhdGguaXNkaXIoc3ApOg0KICAgICAgICAgICAgdHJ5Og0KICAgICAgICAgICAgICAgIGlubmVyID0gc29ydGVkKG9zLmxpc3RkaXIoc3ApKVs6bWF4X3Nob3ddDQogICAgICAgICAgICBleGNlcHQgT1NFcnJvcjoNCiAgICAgICAgICAgICAgICBpbm5lciA9")
parts.append("IFtdDQogICAgICAgICAgICBwcmludChmIiAge3N9LyAtPiB7aW5uZXJ9IiwgZmx1c2g9VHJ1ZSkNCiAgICAgICAgICAgIGZvciBpdCBpbiBpbm5lcls6Ml06DQogICAgICAgICAgICAgICAgaXRwID0gb3MucGF0aC5qb2luKHNwLCBpdCkNCiAgICAgICAgICAgICAg")
parts.append("ICBpZiBvcy5wYXRoLmlzZGlyKGl0cCk6DQogICAgICAgICAgICAgICAgICAgIHRyeToNCiAgICAgICAgICAgICAgICAgICAgICAgIGZpbGVzID0gc29ydGVkKG9zLmxpc3RkaXIoaXRwKSlbOjNdDQogICAgICAgICAgICAgICAgICAgIGV4Y2VwdCBPU0Vycm9yOg0K")
parts.append("ICAgICAgICAgICAgICAgICAgICAgICAgZmlsZXMgPSBbXQ0KICAgICAgICAgICAgICAgICAgICBwcmludChmIiAgICAgIHtpdH0vIC0+IHtmaWxlc30iLCBmbHVzaD1UcnVlKQ0KICAgICAgICBlbHNlOg0KICAgICAgICAgICAgcHJpbnQoZiIgIHtzfSAo5paH5Lu2")
parts.append("KSIsIGZsdXNoPVRydWUpDQogICAgIyDmir3moLcgLmRjbSDorqHmlbA65YmNIDMgc3R1ZHkgw5cg5YmNIDMgc2VyaWVzDQogICAgbl9kY20gPSAwDQogICAgdHJ5Og0KICAgICAgICBmb3IgcyBpbiBzdWJzX2FsbFs6M106DQogICAgICAgICAgICBzcCA9IG9zLnBh")
parts.append("dGguam9pbihkY20sIHMpDQogICAgICAgICAgICBpZiBub3Qgb3MucGF0aC5pc2RpcihzcCk6DQogICAgICAgICAgICAgICAgY29udGludWUNCiAgICAgICAgICAgIGZvciBpdCBpbiBzb3J0ZWQob3MubGlzdGRpcihzcCkpWzozXToNCiAgICAgICAgICAgICAgICBp")
parts.append("dHAgPSBvcy5wYXRoLmpvaW4oc3AsIGl0KQ0KICAgICAgICAgICAgICAgIGlmIG9zLnBhdGguaXNkaXIoaXRwKToNCiAgICAgICAgICAgICAgICAgICAgbl9kY20gKz0gc3VtKDEgZm9yIGYgaW4gb3MubGlzdGRpcihpdHApIGlmIGYubG93ZXIoKS5lbmRzd2l0aCgi")
parts.append("LmRjbSIpKQ0KICAgIGV4Y2VwdCBPU0Vycm9yOg0KICAgICAgICBwYXNzDQogICAgcHJpbnQoZiJb6K+K5patLXt0YWd9XSDmir3moLcgLmRjbSDmlofku7bmlbA6IHtuX2RjbX0o5YmNIDMgc3R1ZHkgw5cg5YmNIDMgc2VyaWVzKSIsIGZsdXNoPVRydWUpDQoNCg0K")
parts.append("ZGVmIF9zY2FuX29mZmxpbmVfd2VpZ2h0cyhiYXNlPSIva2FnZ2xlL2lucHV0IiwgZGVwdGg9MCwgbWF4X2RlcHRoPTQpOg0KICAgICIiIumAkuW9kuaJq+aPj+aMgui9veebruW9leS4i+eahOmihOiuree7g+adg+mHjeaWh+S7tigucHRoLy5wdCDnrYkpLOi3s+i/")
parts.append("hyBESUNPTSDlpKfnm67lvZXjgIINCg0KICAgIOi/lOWbniBbKOe7neWvuei3r+W+hCwg55u45a+5IC9rYWdnbGUvaW5wdXQg55qE5o+P6L+wKV075om+5LiN5Yiw6L+U5Zue56m65YiX6KGo44CCDQogICAg56S+5Yy65omT5YyF55qE56a757q/5p2D6YeN5pWw5o2u")
parts.append("6ZuGKOWmgiBtYXR0ZW9uYWNjYXJhdG8vcmVzbmV0MzQtaW1hZ2VuZXQtcHRo44CBDQogICAgbWFyY3NoYWRlL3Jlc25ldC1wcmV0cmFpbmVkLXdlaWdodHMtaW1hZ2VuZXTjgIFweXRvcmNoL3Jlc25ldDM0KeaMgui9veWQjuWNs+WPr+iiq+aJq+WIsCwNCiAgICDm")
parts.append("l6DpnIDogZTnvZHjgIINCiAgICAiIiINCiAgICBoaXRzID0gW10NCiAgICBpZiBkZXB0aCA+IG1heF9kZXB0aCBvciBub3Qgb3MucGF0aC5pc2RpcihiYXNlKToNCiAgICAgICAgcmV0dXJuIGhpdHMNCiAgICB0cnk6DQogICAgICAgIGVudHJpZXMgPSBzb3J0ZWQo")
parts.append("b3MubGlzdGRpcihiYXNlKSkNCiAgICBleGNlcHQgT1NFcnJvcjoNCiAgICAgICAgcmV0dXJuIGhpdHMNCiAgICBmb3IgZm4gaW4gZW50cmllczoNCiAgICAgICAgcCA9IG9zLnBhdGguam9pbihiYXNlLCBmbikNCiAgICAgICAgaWYgb3MucGF0aC5pc2ZpbGUocCkg")
parts.append("YW5kIGZuLmxvd2VyKCkuZW5kc3dpdGgoX1dFSUdIVF9FWFQpOg0KICAgICAgICAgICAgaGl0cy5hcHBlbmQoKHAsIG9zLnBhdGgucmVscGF0aChwLCAiL2thZ2dsZS9pbnB1dCIpKSkNCiAgICAgICAgZWxpZiBvcy5wYXRoLmlzZGlyKHApIGFuZCBmbiBub3QgaW4g")
parts.append("X1NLSVBfRElSUzoNCiAgICAgICAgICAgIGhpdHMuZXh0ZW5kKF9zY2FuX29mZmxpbmVfd2VpZ2h0cyhwLCBkZXB0aCArIDEsIG1heF9kZXB0aCkpDQogICAgcmV0dXJuIGhpdHMNCg0KDQpkZWYgX2xvYWRfc3RhdGVfZGljdF9mcm9tX2ZpbGUocGF0aCk6DQogICAg")
parts.append("IiIi5LuO5p2D6YeN5paH5Lu26K+7IHN0YXRlX2RpY3Qo5YW85a65OuebtOaOpSBzdGF0ZV9kaWN0IC8g5YyF5ZyoIGNoZWNrcG9pbnQg6YeMIC8g5ZCE56eN5YmN57yAKeOAgg0KDQogICAg6K+75Y+W5aSx6LSl5oiW5qC85byP5LiN5a+56L+U5ZueIE5vbmXjgIIN")
parts.append("CiAgICAiIiINCiAgICB0cnk6DQogICAgICAgIG9iaiA9IHRvcmNoLmxvYWQocGF0aCwgbWFwX2xvY2F0aW9uPSJjcHUiLCB3ZWlnaHRzX29ubHk9VHJ1ZSkNCiAgICBleGNlcHQgVHlwZUVycm9yOiAgIyDml6fniYggdG9yY2gg5pegIHdlaWdodHNfb25seSDlj4Lm")
parts.append("lbANCiAgICAgICAgb2JqID0gdG9yY2gubG9hZChwYXRoLCBtYXBfbG9jYXRpb249ImNwdSIpDQogICAgZXhjZXB0IEV4Y2VwdGlvbiBhcyBlOg0KICAgICAgICBwcmludChmIlvmnYPph41dIHtwYXRofSDor7vlj5blpLHotKU6IHtlfSIsIGZsdXNoPVRydWUpDQog")
parts.append("ICAgICAgIHJldHVybiBOb25lDQogICAgaWYgaXNpbnN0YW5jZShvYmosIGRpY3QpIGFuZCAic3RhdGVfZGljdCIgaW4gb2JqOg0KICAgICAgICBvYmogPSBvYmpbInN0YXRlX2RpY3QiXQ0KICAgIGVsaWYgaXNpbnN0YW5jZShvYmosIGRpY3QpIGFuZCAibW9kZWxf")
parts.append("c3RhdGVfZGljdCIgaW4gb2JqOg0KICAgICAgICBvYmogPSBvYmpbIm1vZGVsX3N0YXRlX2RpY3QiXQ0KICAgIGVsaWYgaXNpbnN0YW5jZShvYmosIGRpY3QpIGFuZCAibW9kZWwiIGluIG9iaiBhbmQgaXNpbnN0YW5jZShvYmpbIm1vZGVsIl0sIGRpY3QpOg0KICAg")
parts.append("ICAgICBvYmogPSBvYmpbIm1vZGVsIl0NCiAgICBpZiBub3QgaXNpbnN0YW5jZShvYmosIGRpY3QpOg0KICAgICAgICByZXR1cm4gTm9uZQ0KICAgICMg5Yml5bi46KeB5YmN57yAOm1vZHVsZS4gLyBtb2RlbC4gLyBiYWNrYm9uZS4o6Iul5pW05L2T5bim5YmN57yA")
parts.append("LOe7n+S4gOi/mOWOnykNCiAgICBvdXQgPSB7fQ0KICAgIGZvciBrLCB2IGluIG9iai5pdGVtcygpOg0KICAgICAgICBmb3IgcHJlIGluICgibW9kdWxlLiIsICJtb2RlbC4iLCAiYmFja2JvbmUuIik6DQogICAgICAgICAgICBpZiBrLnN0YXJ0c3dpdGgocHJlKToN")
parts.append("CiAgICAgICAgICAgICAgICBrID0ga1tsZW4ocHJlKTpdDQogICAgICAgICAgICAgICAgYnJlYWsNCiAgICAgICAgb3V0W2tdID0gdg0KICAgIHJldHVybiBvdXQNCg0KDQpkZWYgX3RyeV9sb2FkX29mZmxpbmUobmV0LCBwYXRocyk6DQogICAgIiIi5oyJ5bqP5bCd")
parts.append("6K+V5oqK5YCZ6YCJ56a757q/5p2D6YeN5Yqg6L296L+bIG5ldChzdGF0ZV9kaWN0IOe7k+aehOmhu+S4jiBSZXNOZXQzNCDljLnphY0p44CCDQoNCiAgICDpgILphY3op4TliJko6Z2i5ZCR56S+5Yy6IFJTTkEg5p2D6YeNLOWmgiBiYXJ1bjIxMDQg55qEIGJlc3Rf")
parts.append("cmVzbmV0MzRfZm9sZDAucHRoKToNCiAgICAtIGNvbnYxLndlaWdodCDpgJrpgZPmlbDkuI3lkIzml7Y6cmVzaGFwZSttZWFuIOmAgumFjeWIsCBuZXQg55qEIGNvbnYxIOmAmumBk+aVsCjlpoIgMjTihpIzLA0KICAgICAgYmFydW4yMTA0IOaYryA4IGNsaXAgw5cg")
parts.append("MyDpgJrpgZPnmoQgMi41RCDmqKHlnospOw0KICAgIC0gZmMuKiAvIGhlYWQuKiAvIGNsYXNzaWZpZXIuKiDkuIDlvovliaXnprso5oiR5Lus55qE5qih5Z6L5Lya5pu/5o2i5oiQIDEyIOWktCk7DQogICAg6L+U5ZueICjmmK/lkKbmiJDlip8sIOWRveS4reaPj+i/")
parts.append("sCnjgIINCiAgICAiIiINCiAgICBmb3IgcCwgZGVzYyBpbiBwYXRoczoNCiAgICAgICAgc2QgPSBfbG9hZF9zdGF0ZV9kaWN0X2Zyb21fZmlsZShwKQ0KICAgICAgICBpZiBzZCBpcyBOb25lOg0KICAgICAgICAgICAgY29udGludWUNCiAgICAgICAgIyAxKSDpgILp")
parts.append("hY0gY29udjE66YCa6YGT5pWw5LiN5ZCM5YiZIHJlc2hhcGUrbWVhbijlpoIgMjQg6YCa6YGTIOKGkiAzIOmAmumBkykNCiAgICAgICAgaWYgImNvbnYxLndlaWdodCIgaW4gc2Q6DQogICAgICAgICAgICB3ID0gc2RbImNvbnYxLndlaWdodCJdDQogICAgICAgICAg")
parts.append("ICB0YXJnZXQgPSBuZXQuY29udjEud2VpZ2h0LnNoYXBlWzFdDQogICAgICAgICAgICBpZiB3LnNoYXBlWzFdICE9IHRhcmdldCBhbmQgdy5zaGFwZVsxXSAlIHRhcmdldCA9PSAwOg0KICAgICAgICAgICAgICAgIEIsIEMsIEgsIFcgPSB3LnNoYXBlDQogICAgICAg")
parts.append("ICAgICAgICAgdzIgPSB3LnJlc2hhcGUoQiwgdGFyZ2V0LCBDIC8vIHRhcmdldCwgSCwgVykubWVhbihkaW09MikNCiAgICAgICAgICAgICAgICBzZFsiY29udjEud2VpZ2h0Il0gPSB3Mg0KICAgICAgICAgICAgICAgIHByaW50KGYiW+adg+mHjV0ge2Rlc2N9IGNv")
parts.append("bnYxIOmAmumBkyB7Q33ihpJ7dGFyZ2V0fSDlubPlnYfpgILphY0iLCBmbHVzaD1UcnVlKQ0KICAgICAgICAgICAgZWxpZiB3LnNoYXBlWzFdICE9IHRhcmdldDoNCiAgICAgICAgICAgICAgICBzZC5wb3AoImNvbnYxLndlaWdodCIpICAjIOaXoOazlemAgumFjSzk")
parts.append("uKLlvIMo55So6ZqP5py65Yid5aeL5YyWKQ0KICAgICAgICAjIDIpIOWJpeemuyBmYy9oZWFkL2NsYXNzaWZpZXIo5qih5Z6L5Lya5pu/5o2i5oiQIDEyIOWktCkNCiAgICAgICAgZm9yIGsgaW4gbGlzdChzZC5rZXlzKCkpOg0KICAgICAgICAgICAgaWYgay5zdGFy")
parts.append("dHN3aXRoKCgiZmMuIiwgImhlYWQuIiwgImNsYXNzaWZpZXIuIikpOg0KICAgICAgICAgICAgICAgIHNkLnBvcChrKQ0KICAgICAgICB0cnk6DQogICAgICAgICAgICBtaXNzaW5nLCB1bmV4cGVjdGVkID0gbmV0LmxvYWRfc3RhdGVfZGljdChzZCwgc3RyaWN0PUZh")
parts.append("bHNlKQ0KICAgICAgICBleGNlcHQgRXhjZXB0aW9uIGFzIGU6DQogICAgICAgICAgICBwcmludChmIlvmnYPph41dIHtkZXNjfSDkuI4gUmVzTmV0MzQg57uT5p6E5LiN5Yy56YWNOiB7ZX0iLCBmbHVzaD1UcnVlKQ0KICAgICAgICAgICAgY29udGludWUNCiAgICAg")
parts.append("ICAgZXhlbXB0ID0ge2sgZm9yIGsgaW4gbWlzc2luZyBpZiBrLnN0YXJ0c3dpdGgoKCJjb252MS4iLCAiZmMuIikpfQ0KICAgICAgICBpZiBub3QgdW5leHBlY3RlZCBhbmQgbGVuKG1pc3NpbmcpID09IGxlbihleGVtcHQpOg0KICAgICAgICAgICAgcHJpbnQoZiJb")
parts.append("5p2D6YeNXSDnprvnur/pooTorq3nu4PliqDovb3miJDlip86IHtkZXNjfSAo6KaG55uWIHtsZW4oc2QpfSDkuKrlj4LmlbDlvKDph48pIiwgZmx1c2g9VHJ1ZSkNCiAgICAgICAgICAgIHJldHVybiBUcnVlLCBkZXNjDQogICAgICAgIHByaW50KGYiW+adg+mHjV0g")
parts.append("e2Rlc2N9IOe7k+aehOS4jeWMuemFjShtaXNzaW5nIHtsZW4obWlzc2luZykgLSBsZW4oZXhlbXB0KX0g5Liq6Z2e6LGB5YWNIGtleSwgIg0KICAgICAgICAgICAgICBmInVuZXhwZWN0ZWQge2xlbih1bmV4cGVjdGVkKX0pLOWwneivleS4i+S4gOS4qiIsIGZsdXNo")
parts.append("PVRydWUpDQogICAgcmV0dXJuIEZhbHNlLCAiIg0KDQoNCmRlZiBfc2Nhbl9wc2V1ZG8oYmFzZSwgZGVwdGg9MCwgbWF4X2RlcHRoPTMpOg0KICAgICIiIumAkuW9kuWcqOaMgui9veebruW9leS4i+aJvui9r+agh+etviBDU1Yo6ZmQ5Yi25rex5bqmLOi3s+i/hyBE")
parts.append("SUNPTSDnm67lvZUp44CCDQoNCiAgICDov5Tlm54gKERhdGFGcmFtZSDmiJYgTm9uZSwg5p2l5rqQ5o+P6L+wKeOAguivhuWIq+inhOWImTrlkKsgU3R1ZHlJbnN0YW5jZVVJRCDkuJToh7PlsJHmnInkuIDkuKoNCiAgICBwc2V1ZG9fIOWJjee8gOWIlyjlpoIgcHNl")
parts.append("dWRvX0FDTCnjgILmib7kuI3liLDov5Tlm54gKE5vbmUsICIiKeOAgg0KDQogICAg6K+G5Yir5pa55byP5Y+M6YCa6YGTOg0KICAgIDEuIOaWh+S7tuWQjeeZveWQjeWNlShfUFNFVURPX0ZJTEVTKeW/q+mAn+WRveS4rTsNCiAgICAyLiAqKuaMieWGheWuueivhuWI")
parts.append("qyoqOuS7u+aEjyAqLmNzdiDlj6ropoHooajlpLTlkKsgU3R1ZHlJbnN0YW5jZVVJRCArIHBzZXVkb18qIOWIl+WNs+WRveS4rQ0KICAgICAgICjkuI3kvp3otZbmlofku7blkI0s56S+5Yy65pWw5o2u6ZuG5pS55ZCNL+aNoue7k+aehOS5n+iDveaJq+WIsCnjgIIN")
parts.append("CiAgICAiIiINCiAgICBpZiBkZXB0aCA+IG1heF9kZXB0aCBvciBub3Qgb3MucGF0aC5pc2RpcihiYXNlKToNCiAgICAgICAgcmV0dXJuIE5vbmUsICIiDQogICAgdHJ5Og0KICAgICAgICBlbnRyaWVzID0gc29ydGVkKG9zLmxpc3RkaXIoYmFzZSkpDQogICAgZXhj")
parts.append("ZXB0IE9TRXJyb3I6DQogICAgICAgIHJldHVybiBOb25lLCAiIg0KICAgICMg6YCa6YGTIDE65paH5Lu25ZCN55m95ZCN5Y2VKOWQkeWQjuWFvOWuuSzor7vlhajph48pDQogICAgZm9yIGZuIGluIF9QU0VVRE9fRklMRVM6DQogICAgICAgIHAgPSBvcy5wYXRoLmpv")
parts.append("aW4oYmFzZSwgZm4pDQogICAgICAgIGlmIG9zLnBhdGguZXhpc3RzKHApIGFuZCBvcy5wYXRoLmlzZmlsZShwKToNCiAgICAgICAgICAgIHRyeToNCiAgICAgICAgICAgICAgICBjYW5kID0gcGQucmVhZF9jc3YocCkNCiAgICAgICAgICAgIGV4Y2VwdCBFeGNlcHRp")
parts.append("b246DQogICAgICAgICAgICAgICAgY29udGludWUNCiAgICAgICAgICAgIGlmICJTdHVkeUluc3RhbmNlVUlEIiBpbiBjYW5kLmNvbHVtbnMgYW5kIGFueSgNCiAgICAgICAgICAgICAgICAgICAgYy5zdGFydHN3aXRoKCJwc2V1ZG9fIikgZm9yIGMgaW4gY2FuZC5j")
parts.append("b2x1bW5zKToNCiAgICAgICAgICAgICAgICByZXR1cm4gY2FuZCwgb3MucGF0aC5yZWxwYXRoKHAsICIva2FnZ2xlL2lucHV0IikNCiAgICAjIOmAmumBkyAyOuaMieWGheWuueivhuWIqyjlj6ror7vooajlpLQgbnJvd3M9MyDliKTmlq0s5ZG95Lit5YaN5YWo6K+7")
parts.append("KQ0KICAgIGZvciBmbiBpbiBlbnRyaWVzOg0KICAgICAgICBpZiBub3QgZm4ubG93ZXIoKS5lbmRzd2l0aCgiLmNzdiIpOg0KICAgICAgICAgICAgY29udGludWUNCiAgICAgICAgcCA9IG9zLnBhdGguam9pbihiYXNlLCBmbikNCiAgICAgICAgaWYgbm90IG9zLnBh")
parts.append("dGguaXNmaWxlKHApOg0KICAgICAgICAgICAgY29udGludWUNCiAgICAgICAgdHJ5Og0KICAgICAgICAgICAgaGVhZCA9IHBkLnJlYWRfY3N2KHAsIG5yb3dzPTMpDQogICAgICAgIGV4Y2VwdCBFeGNlcHRpb246DQogICAgICAgICAgICBjb250aW51ZQ0KICAgICAg")
parts.append("ICBpZiAiU3R1ZHlJbnN0YW5jZVVJRCIgaW4gaGVhZC5jb2x1bW5zIGFuZCBhbnkoDQogICAgICAgICAgICAgICAgYy5zdGFydHN3aXRoKCJwc2V1ZG9fIikgZm9yIGMgaW4gaGVhZC5jb2x1bW5zKToNCiAgICAgICAgICAgIHRyeToNCiAgICAgICAgICAgICAgICBj")
parts.append("YW5kID0gcGQucmVhZF9jc3YocCkNCiAgICAgICAgICAgICAgICByZXR1cm4gY2FuZCwgb3MucGF0aC5yZWxwYXRoKHAsICIva2FnZ2xlL2lucHV0IikNCiAgICAgICAgICAgIGV4Y2VwdCBFeGNlcHRpb246DQogICAgICAgICAgICAgICAgY29udGludWUNCiAgICBm")
parts.append("b3Igc3ViIGluIGVudHJpZXM6DQogICAgICAgIGlmIHN1YiBpbiBfU0tJUF9ESVJTOg0KICAgICAgICAgICAgY29udGludWUNCiAgICAgICAgY2FuZCwgc3JjID0gX3NjYW5fcHNldWRvKG9zLnBhdGguam9pbihiYXNlLCBzdWIpLCBkZXB0aCArIDEsIG1heF9kZXB0")
parts.append("aCkNCiAgICAgICAgaWYgY2FuZCBpcyBub3QgTm9uZToNCiAgICAgICAgICAgIHJldHVybiBjYW5kLCBzcmMNCiAgICByZXR1cm4gTm9uZSwgIiINCg0KDQpkZWYgZmluZF9wc2V1ZG9fZGF0YXNldCgpOg0KICAgICIiIuWcqCAva2FnZ2xlL2lucHV0IOS4i+mAkuW9")
parts.append("kuaJq+aPj+ekvuWMuui9r+agh+etvuaVsOaNrumbhihMTE0g5LuO5oql5ZGK5o+Q5Y+W55qE5Lyq5qCH562+KeOAgg0KDQogICAg5YW85a655Y2V5bGCKC9rYWdnbGUvaW5wdXQvPHNsdWc+LynkuI7lpJrlsYIoL2thZ2dsZS9pbnB1dC9kYXRhc2V0cy88c2x1Zz4v")
parts.append("KeaMgui9veOAgg0KICAgIOi/lOWbniAoRGF0YUZyYW1lIOaIliBOb25lLCDmnaXmupDmj4/ov7ApO+aJvuS4jeWIsOi/lOWbniAoTm9uZSwgIiIp44CCDQogICAgIiIiDQogICAgaWYgbm90IG9zLnBhdGguaXNkaXIoIi9rYWdnbGUvaW5wdXQiKToNCiAgICAgICAg")
parts.append("cmV0dXJuIE5vbmUsICIiDQogICAgcmV0dXJuIF9zY2FuX3BzZXVkbygiL2thZ2dsZS9pbnB1dCIpDQoNCg0KZGVmIGxvYWRfbGFiZWxzKHJvb3QsIHBzZXVkb19jb25mPTAuMCwgdXNlX3JlcG9ydF9sYWJlbHM9VHJ1ZSk6DQogICAgIiIi6L+U5ZueIHRyYWluIHN0")
parts.append("dWR5IOe6p+agh+etviBEYXRhRnJhbWUo6KGMPVN0dWR5SW5zdGFuY2VVSUQsIOWIlz1MQUJFTFMsIOWIhuaVsC9OYU4p44CCDQoNCiAgICDmoIfnrb7mnaXmupDkvJjlhYjnuqcoMjAyNi0wOS0wNiDotbc75pen55qEIuekvuWMuui9r+agh+etvuaVsOaNrumbhiLm")
parts.append("lrnmoYjlt7Lor4HkvKrlvIPnlKjigJTigJQNCiAgICBiYXJ1bjIxMDQg55qEIHBzZXVkb18qIDk4LjclIOaYr+Wtl+mdoiAwLjUg5q275YC844CB6Zu25aKe6YeP5L+h5oGvLOaYryBsb3NzIOaBkiAwLjY5Mw0KICAgIOeahOecn+agueWboCk6DQogICAgMS4g5a6Y")
parts.append("5pa5IHRyYWluLmNzdiDmoIfms6goMC8xLOS7hSA1OC80NDA3IOS+iyk7DQogICAgMi4g5YaF572u5oql5ZGK6KeE5YiZ5oq95Y+W5ZmoIHJzbmFfcmVwb3J0X2xhYmVscyjnp7vmpI3oh6ogcnNuYS1rbmVlLWJhc2VsaW5lLXYxLA0KICAgICAgIOWkmuivreiogC/m")
parts.append("jInlj6XlkKblrprmnoHmgKcv5Lil6YeN5bqm5YiG57qnKTrlr7nmr4/ku73mlL7lsITmiqXlkYrkuqflh7ogMTIg55eF54G2IHNvZnQg5YiG5pWwDQogICAgICAgKOmYs+aApyAwLjcyfjAuOTUgLyDpmLTmgKcgMC4wNH4wLjIwIC8g5peg6K+B5o2uPU5hTik7DQog")
parts.append("ICAgICAg5a6Y5pa557y65aSx5qC855So5oq95Y+W5YiG5pWw5aGr5YWFLOaXoOivgeaNruagvOS/neaMgSBOYU4o5LiN6L+b5o2f5aSxLOeUsSB2YWxpZCDmjqnnoIHmjpLpmaQp44CCDQogICAgIiIiDQogICAgdHJhaW4gPSBwZC5yZWFkX2Nzdihvcy5wYXRoLmpv")
parts.append("aW4ocm9vdCwgInRyYWluLmNzdiIpKQ0KICAgIGRmID0gdHJhaW5bWyJTdHVkeUluc3RhbmNlVUlEIl0gKyBMQUJFTFNdLmNvcHkoKQ0KICAgIG9mZmljaWFsX2NudCA9IHtjOiBpbnQoZGZbY10ubm90bmEoKS5zdW0oKSkgZm9yIGMgaW4gTEFCRUxTfQ0KDQogICAg")
parts.append("IyBFWFAtMDEzOiBtdWx0aS1zb3VyY2UgY29uc2Vuc3VzIHNvZnQgbGFiZWxzDQogICAgaWYgb3MuZW52aXJvbi5nZXQoIkVYUDAxM19NVUxUSVNPVVJDRSIsICIiKToNCiAgICAgICAgZnJvbSBleHAwMTNfbXVsdGlzb3VyY2UgaW1wb3J0IGJ1aWxkX2V4cDAxM19s")
parts.append("YWJlbHMNCiAgICAgICAgb3V0ID0gYnVpbGRfZXhwMDEzX2xhYmVscyhvcy5wYXRoLmpvaW4ocm9vdCwgInRyYWluLmNzdiIpLCB2ZXJib3NlPVRydWUpDQogICAgICAgIHByaW50KCJb5qCH562+XSBFWFAtMDEzIG11bHRpLXNvdXJjZSB0YWJsZSBsb2FkZWQiKQ0K")
parts.append("ICAgICAgICByZXR1cm4gb3V0DQogICAgIyBFWFAtMDEyOiBMTE0tcHJpbWFyeSBzb2Z0IGxhYmVscyAoZ29sZCB3aW5zOyBzaWxlbmNlIDAuMjUvMC41IOKGkiBOYU4pDQogICAgZXhwMDEyID0gb3MuZW52aXJvbi5nZXQoIkVYUDAxMl9MTE1fUFJJTUFSWSIsICIi")
parts.append("KQ0KICAgIGlmIGV4cDAxMjoNCiAgICAgICAgZnJvbSBleHAwMTJfbGxtX3ByaW1hcnkgaW1wb3J0IGJ1aWxkX2V4cDAxMl9sYWJlbHMNCiAgICAgICAgb3V0ID0gYnVpbGRfZXhwMDEyX2xhYmVscyhvcy5wYXRoLmpvaW4ocm9vdCwgInRyYWluLmNzdiIpLCB2ZXJi")
parts.append("b3NlPVRydWUpDQogICAgICAgIHByaW50KCJb5qCH562+XSBFWFAtMDEyIExMTS1wcmltYXJ5IHRhYmxlIGxvYWRlZCIpDQogICAgICAgIHJldHVybiBvdXQNCg0KICAgIGlmIG5vdCB1c2VfcmVwb3J0X2xhYmVsczoNCiAgICAgICAgcHJpbnQoIlvmoIfnrb5dIC0t")
parts.append("bm8tcmVwb3J0LWxhYmVsczrku4XlrpjmlrnmoIfms6jlj6/nlKgiKQ0KICAgICAgICByZXR1cm4gZGYNCiAgICBpZiBub3QgSEFTX1JMOg0KICAgICAgICBwcmludCgiW+agh+etvl0g5pyq5om+5YiwIHJzbmFfcmVwb3J0X2xhYmVscy5weSjpnIDkuI4gcnNuYV92")
parts.append("My5weSDlkIznm67lvZUpLCINCiAgICAgICAgICAgICAgIuS7heWumOaWueagh+azqOWPr+eUqCIpDQogICAgICAgIHJldHVybiBkZg0KICAgIGlmICJSZXBvcnQiIG5vdCBpbiB0cmFpbi5jb2x1bW5zOg0KICAgICAgICBwcmludCgiW+agh+etvl0gdHJhaW4uY3N2")
parts.append("IOaXoCBSZXBvcnQg5YiXLOS7heWumOaWueagh+azqOWPr+eUqCIpDQogICAgICAgIHJldHVybiBkZg0KDQogICAgcHNldWRvID0gUkwuYnVpbGRfcHNldWRvX3RhYmxlKHRyYWluLCB2ZXJib3NlPVRydWUpDQogICAgcHNldWRvID0gcHNldWRvLnNldF9pbmRleCgi")
parts.append("U3R1ZHlJbnN0YW5jZVVJRCIpDQogICAgcHJpbnQoZiJb5qCH562+XSDloavlhYXnu5/orqEoe2xlbihkZil9IOS4qiBzdHVkeSk6IikNCiAgICBmb3IgYyBpbiBMQUJFTFM6DQogICAgICAgIG1hc2sgPSBkZltjXS5pc25hKCkudmFsdWVzDQogICAgICAgIGlmIG5v")
parts.append("dCBtYXNrLmFueSgpOg0KICAgICAgICAgICAgcHJpbnQoZiIgIHtjOjE4c30g5a6Y5pa55YWo6YePLOaXoOmcgOWhq+WFhSIpDQogICAgICAgICAgICBjb250aW51ZQ0KICAgICAgICB1aWRzID0gZGYubG9jW21hc2ssICJTdHVkeUluc3RhbmNlVUlEIl0NCiAgICAg")
parts.append("ICAgdmFscyA9IHBzZXVkb1sicHNldWRvXyIgKyBjXS5yZWluZGV4KHVpZHMpLnZhbHVlcw0KICAgICAgICBuX2ZpbGwgPSBpbnQocGQubm90bmEodmFscykuc3VtKCkpDQogICAgICAgIGRmLmxvY1ttYXNrLCBjXSA9IHZhbHMNCiAgICAgICAgcHJpbnQoZiIgIHtj")
parts.append("OjE4c30g5a6Y5pa5IHtvZmZpY2lhbF9jbnRbY106NGR9ICsg5oql5ZGK5oq95Y+W5aGr5YWFIHtuX2ZpbGw6NGR9Ig0KICAgICAgICAgICAgICBmIiA9IOacieaViCB7aW50KGRmW2NdLm5vdG5hKCkuc3VtKCkpOjVkfSIpDQogICAgcmV0dXJuIGRmDQoNCg0KX0RD")
parts.append("TV9NSVNTX1BSSU5URUQgPSBbRmFsc2VdICAjIOWPquaJk+WNsOS4gOasoSBESUNPTSDmnKrlkb3kuK3or4rmlq0s6YG/5YWNIDIg5LiHIHNlcmllcyDliLflsY8NCg0KDQpkZWYgcmVhZF9kaWNvbV9zbGljZXMoZGNtX2Rpciwgc2VyaWVzX3VpZCwgc3R1ZHlfdWlk")
parts.append("PU5vbmUsIHNhbXBsaW5nX21vZGU9ImxlZ2FjeSIpOg0KICAgICIiIuivu+WPluS4gOS4qiBzZXJpZXMg55qE5omA5pyJ5YiH54mH5bm25o6S5bqPLOi/lOWbniBbKHBhdGgsIHBvcyld44CCDQoNCiAgICBzYW1wbGluZ19tb2RlOg0KICAgICAgbGVnYWN5ICDigJQg")
parts.append("5oyJIEltYWdlUG9zaXRpb25QYXRpZW50WzJd77yI5Y6G5Y+y6KGM5Li6LCBFWFAtMDB4IGJhc2VsaW5l77yJDQogICAgICBwaHlzaWNhbCDigJQg5oyJIGRvdChJbWFnZVBvc2l0aW9uUGF0aWVudCwgc2xpY2Vfbm9ybWFsKe+8iEVYUC0wMTbvvIkNCiAgICAiIiIN")
parts.append("CiAgICBjYW5kcyA9IFtvcy5wYXRoLmpvaW4oZGNtX2Rpciwgc2VyaWVzX3VpZCldDQogICAgaWYgc3R1ZHlfdWlkOg0KICAgICAgICBjYW5kcy5hcHBlbmQob3MucGF0aC5qb2luKGRjbV9kaXIsIHN0dWR5X3VpZCwgc2VyaWVzX3VpZCkpDQogICAgc2RpciA9IG5l")
parts.append("eHQoKGMgZm9yIGMgaW4gY2FuZHMgaWYgb3MucGF0aC5pc2RpcihjKSksIE5vbmUpDQogICAgaWYgc2RpciBpcyBOb25lOg0KICAgICAgICBpZiBub3QgX0RDTV9NSVNTX1BSSU5URURbMF06DQogICAgICAgICAgICBfRENNX01JU1NfUFJJTlRFRFswXSA9IFRydWUN")
parts.append("CiAgICAgICAgICAgIHByaW50KGYiW+iviuaWrS1ESUNPTV0g5om+5LiN5YiwIHNlcmllcyDnm67lvZUs5bCd6K+V6L+HOiB7Y2FuZHN9IiwgZmx1c2g9VHJ1ZSkNCiAgICAgICAgICAgIHByaW50KGYiW+iviuaWrS1ESUNPTV0ge2RjbV9kaXJ9IOaYr+WQpuWtmOWc")
parts.append("qDoge29zLnBhdGguaXNkaXIoZGNtX2Rpcil9IiwgZmx1c2g9VHJ1ZSkNCiAgICAgICAgICAgIHRyeToNCiAgICAgICAgICAgICAgICBpbm5lciA9IHNvcnRlZChvcy5saXN0ZGlyKGRjbV9kaXIpKVs6OF0gaWYgb3MucGF0aC5pc2RpcihkY21fZGlyKSBlbHNlIFtd")
parts.append("DQogICAgICAgICAgICAgICAgcHJpbnQoZiJb6K+K5patLURJQ09NXSB7ZGNtX2Rpcn0g5a6e6ZmF5YaF5a655qC35pysOiB7aW5uZXJ9IiwgZmx1c2g9VHJ1ZSkNCiAgICAgICAgICAgIGV4Y2VwdCBPU0Vycm9yOg0KICAgICAgICAgICAgICAgIHBhc3MNCiAgICAg")
parts.append("ICAgcmV0dXJuIFtdDQogICAgZmlsZXMgPSBbb3MucGF0aC5qb2luKHNkaXIsIGYpIGZvciBmIGluIG9zLmxpc3RkaXIoc2RpcikgaWYgZi5sb3dlcigpLmVuZHN3aXRoKCIuZGNtIildDQogICAgaWYgbm90IGZpbGVzOg0KICAgICAgICBpZiBub3QgX0RDTV9NSVNT")
parts.append("X1BSSU5URURbMF06DQogICAgICAgICAgICBfRENNX01JU1NfUFJJTlRFRFswXSA9IFRydWUNCiAgICAgICAgICAgIHByaW50KGYiW+iviuaWrS1ESUNPTV0g55uu5b2V5a2Y5Zyo5L2G5pegIC5kY20g5paH5Lu2OiB7c2Rpcn0iLCBmbHVzaD1UcnVlKQ0KICAgICAg")
parts.append("ICAgICAgcHJpbnQoZiJb6K+K5patLURJQ09NXSDnm67lvZXlhoXlrrnmoLfmnKw6IHtzb3J0ZWQob3MubGlzdGRpcihzZGlyKSlbOjhdfSIsIGZsdXNoPVRydWUpDQogICAgICAgIHJldHVybiBbXQ0KICAgIGlmIHNhbXBsaW5nX21vZGUgPT0gInBoeXNpY2FsIiBh")
parts.append("bmQgSEFTX1NHOg0KICAgICAgICBkZWYgX2hkcihwYXRoKToNCiAgICAgICAgICAgIHJldHVybiBweWRpY29tLmRjbXJlYWQocGF0aCwgc3RvcF9iZWZvcmVfcGl4ZWxzPVRydWUpDQoNCiAgICAgICAgcm93czMsIG1ldGEgPSBTRy5zb3J0X3NsaWNlc19waHlzaWNh")
parts.append("bChmaWxlcywgX2hkcikNCiAgICAgICAgIyByZXR1cm4gKHBhdGgsIHBvcykgY29tcGF0aWJsZSB3aXRoIGxlZ2FjeSBjb25zdW1lcnMNCiAgICAgICAgcmV0dXJuIFsoZiwgcCkgZm9yIChmLCBwLCBfaykgaW4gcm93czNdDQogICAgcm93cyA9IFtdDQogICAgZm9y")
parts.append("IGYgaW4gZmlsZXM6DQogICAgICAgIHRyeToNCiAgICAgICAgICAgIGQgPSBweWRpY29tLmRjbXJlYWQoZiwgc3RvcF9iZWZvcmVfcGl4ZWxzPUZhbHNlKQ0KICAgICAgICAgICAgcG9zID0gZC5nZXQoIkltYWdlUG9zaXRpb25QYXRpZW50IikNCiAgICAgICAgICAg")
parts.append("IHogPSBmbG9hdChwb3NbMl0pIGlmIHBvcyBpcyBub3QgTm9uZSBhbmQgbGVuKHBvcykgPj0gMyBlbHNlIE5vbmUNCiAgICAgICAgICAgIHJvd3MuYXBwZW5kKChmLCB6KSkNCiAgICAgICAgZXhjZXB0IEV4Y2VwdGlvbjoNCiAgICAgICAgICAgIGNvbnRpbnVlDQog")
parts.append("ICAgcm93cy5zb3J0KGtleT1sYW1iZGEgcjogKHJbMV0gaXMgTm9uZSwgclsxXSBpZiByWzFdIGlzIG5vdCBOb25lIGVsc2UgMC4wKSkNCiAgICByZXR1cm4gcm93cw0KDQoNCiMgLS0tLS0tLS0tLS0tLS0tLSBESUNPTSAtPiAyLjVEIGNsaXAg6aKE5aSE55CGIC0t")
parts.append("LS0tLS0tLS0tLS0tLS0NCg0KZGVmIHdpbmRvd19hbmRfcmVzaXplKGRjbSwgaW1nX3NpemU9SU1HX1NJWkUpOg0KICAgICIiIuWNleW8oOWIh+eJhzpSZXNjYWxlIOWQjueZvuWIhuS9jeeql+WPo+WMliAtPiBbMCwxXSAtPiByZXNpemUgMjI044CCIiIiDQogICAg")
parts.append("YXJyID0gZGNtLnBpeGVsX2FycmF5LmFzdHlwZShucC5mbG9hdDMyKQ0KICAgIHNsb3BlID0gZmxvYXQoZGNtLmdldCgiUmVzY2FsZVNsb3BlIiwgMSkpDQogICAgaW50ZXIgPSBmbG9hdChkY20uZ2V0KCJSZXNjYWxlSW50ZXJjZXB0IiwgMCkpDQogICAgYXJyID0g")
parts.append("YXJyICogc2xvcGUgKyBpbnRlcg0KICAgIGlmIGRjbS5nZXQoIlBob3RvbWV0cmljSW50ZXJwcmV0YXRpb24iKSA9PSAiTU9OT0NIUk9NRTEiOg0KICAgICAgICBhcnIgPSBhcnIubWF4KCkgLSBhcnINCiAgICBsbywgaGkgPSBucC5wZXJjZW50aWxlKGFyciwgWzEs")
parts.append("IDk5XSkNCiAgICBhcnIgPSBucC5jbGlwKChhcnIgLSBsbykgLyBtYXgoaGkgLSBsbywgMWUtNiksIDAsIDEpDQogICAgZnJvbSBQSUwgaW1wb3J0IEltYWdlDQogICAgaW1nID0gSW1hZ2UuZnJvbWFycmF5KChhcnIgKiAyNTUpLmFzdHlwZShucC51aW50OCkpLnJl")
parts.append("c2l6ZSgoaW1nX3NpemUsIGltZ19zaXplKSwgSW1hZ2UuQklMSU5FQVIpDQogICAgcmV0dXJuIG5wLmFzYXJyYXkoaW1nLCBkdHlwZT1ucC5mbG9hdDMyKSAvIDI1NS4wDQoNCg0KZGVmIG1ha2VfY2xpcHMoc2VyaWVzX3Jvd3MsIG5fY2xpcHM9MiwgaW1nX3NpemU9")
parts.append("SU1HX1NJWkUsIHNhbXBsaW5nX21vZGU9ImxlZ2FjeSIpOg0KICAgICIiIuS7juaOkuW6j+WlveeahOWIh+eJh+WIl+ihqOeUn+aIkOiLpeW5siAzIOmAmumBkyBjbGlw44CCDQoNCiAgICBsZWdhY3kgIDog5Z2H5YyAIGluZGV4IOatpei/myzkuInlhYPnu4QgPSAo")
parts.append("Yy0xLGMsYysxKSDlnKjmjpLluo/mlbDnu4TkuIrnmoTpgrvlsYUNCiAgICBwaHlzaWNhbDog5oyJ54mp55CG6Leo5bqm6YCJIGNlbnRlcizkuInlhYPnu4Tlv4XpobvlnKjmjpLluo/lkI7nmoTniannkIbkvY3nva7kuIrnnJ/mraPnm7jpgrsNCiAgICAgICAgICAg")
parts.append("ICAgKGdhcCDnm7jlr7kgbWVkaWFuIHNwYWNpbmcg5Yik5a6aLOingSBzYW1wbGluZ19nZW9tZXRyeSkNCiAgICAiIiINCiAgICBpZiBzYW1wbGluZ19tb2RlID09ICJwaHlzaWNhbCIgYW5kIEhBU19TRzoNCiAgICAgICAgY2xpcHMsIF9tZXRhcyA9IFNHLm1ha2Vf")
parts.append("Y2xpcHNfcGh5c2ljYWwoDQogICAgICAgICAgICBzZXJpZXNfcm93cywNCiAgICAgICAgICAgIG5fY2xpcHM9bl9jbGlwcywNCiAgICAgICAgICAgIGltZ19zaXplPWltZ19zaXplLA0KICAgICAgICAgICAgd2luZG93X2ZuPXdpbmRvd19hbmRfcmVzaXplLA0KICAg")
parts.append("ICAgICAgICAgZGNtX3JlYWRfZm49bGFtYmRhIHA6IHB5ZGljb20uZGNtcmVhZChwKSwNCiAgICAgICAgKQ0KICAgICAgICByZXR1cm4gY2xpcHMNCiAgICBjbGlwcyA9IFtdDQogICAgcGF0aHMgPSBbclswXSBmb3IgciBpbiBzZXJpZXNfcm93c10NCiAgICBpZiBs")
parts.append("ZW4ocGF0aHMpIDwgQ0xJUDoNCiAgICAgICAgcmV0dXJuIGNsaXBzDQogICAgc3RlcCA9IG1heCgxLCAobGVuKHBhdGhzKSAtIDIpIC8vIG5fY2xpcHMpDQogICAgZm9yIGMgaW4gcmFuZ2UoMSwgbGVuKHBhdGhzKSAtIDEsIHN0ZXApOg0KICAgICAgICBpZiBsZW4o")
parts.append("Y2xpcHMpID49IG5fY2xpcHM6DQogICAgICAgICAgICBicmVhaw0KICAgICAgICBjaGFuID0gW10NCiAgICAgICAgb2sgPSBUcnVlDQogICAgICAgIGZvciBrIGluIChjIC0gMSwgYywgYyArIDEpOg0KICAgICAgICAgICAgdHJ5Og0KICAgICAgICAgICAgICAgIGRj")
parts.append("bSA9IHB5ZGljb20uZGNtcmVhZChwYXRoc1trXSkNCiAgICAgICAgICAgICAgICBjaGFuLmFwcGVuZCh3aW5kb3dfYW5kX3Jlc2l6ZShkY20sIGltZ19zaXplKSkNCiAgICAgICAgICAgIGV4Y2VwdCBFeGNlcHRpb246DQogICAgICAgICAgICAgICAgb2sgPSBGYWxz")
parts.append("ZQ0KICAgICAgICAgICAgICAgIGJyZWFrDQogICAgICAgIGlmIG9rOg0KICAgICAgICAgICAgY2xpcHMuYXBwZW5kKG5wLnN0YWNrKGNoYW4sIGF4aXM9MCkpICAjICgzLCBILCBXKQ0KICAgIHJldHVybiBjbGlwcw0KDQoNCmlmIEhBU19UT1JDSDoNCiAgICAjIC0t")
parts.append("LS0tLS0tLS0tLS0tLS0g5pWw5o2u6ZuGKHYyOuaMieW5s+mdoui/h+a7pCBzZXJpZXMpIC0tLS0tLS0tLS0tLS0tLS0NCg0KICAgIGNsYXNzIEtuZWVEYXRhc2V0KERhdGFzZXQpOg0KICAgICAgICAiIiJzdHVkeSDnuqfmlbDmja7pm4Ys5Y+q5Y+W5oyH5a6aIEFu")
parts.append("YXRvbWljYWxfUGxhbmUg55qEIHNlcmllcyzmi7zmiJAgKEMqMywgSCwgVynjgIIiIiINCg0KICAgICAgICBkZWYgX19pbml0X18oc2VsZiwgc3R1ZGllcywgbGFiZWxzX2RmLCBkY21fcm9vdCwgc2VyaWVzX2RmLCBwbGFuZSwNCiAgICAgICAgICAgICAgICAgICAg")
parts.append("IGhhc190YXJnZXRzPVRydWUsIG5fY2xpcHM9Miwgc2FtcGxpbmdfbW9kZT0ibGVnYWN5Iik6DQogICAgICAgICAgICBzZWxmLnN0dWRpZXMgPSBzdHVkaWVzDQogICAgICAgICAgICBzZWxmLmxhYmVsc19kZiA9IGxhYmVsc19kZg0KICAgICAgICAgICAgc2VsZi5k")
parts.append("Y21fcm9vdCA9IGRjbV9yb290DQogICAgICAgICAgICBzZWxmLnNlcmllc19kZiA9IHNlcmllc19kZg0KICAgICAgICAgICAgc2VsZi5wbGFuZSA9IHBsYW5lDQogICAgICAgICAgICBzZWxmLmhhc190YXJnZXRzID0gaGFzX3RhcmdldHMNCiAgICAgICAgICAgIHNl")
parts.append("bGYubl9jbGlwcyA9IG5fY2xpcHMNCiAgICAgICAgICAgIHNlbGYuc2FtcGxpbmdfbW9kZSA9IHNhbXBsaW5nX21vZGUNCiAgICAgICAgICAgIHNlbGYuY2FjaGUgPSB7fSAgIyDmr4/mipjni6znq4vnvJPlrZg6Zm9sZCDnu5PmnZ/pmo8gZGF0YXNldCDph4rmlL4s")
parts.append("6YG/5YWNIDUg5oqY57Sv56evIE9PTQ0KICAgICAgICAgICAgIyDor4rmlq3orqHmlbA66K+75YiwIGNsaXAg55qEIHN0dWR5IC8g56m6IHN0dWR5IC8g57Sv6K6hIGNsaXAg5pWwDQogICAgICAgICAgICBzZWxmLm5faGl0ID0gMA0KICAgICAgICAgICAgc2VsZi5u")
parts.append("X2VtcHR5ID0gMA0KICAgICAgICAgICAgc2VsZi5uX2NsaXBzX3RvdGFsID0gMA0KDQogICAgICAgIGRlZiBzdGF0cyhzZWxmKToNCiAgICAgICAgICAgICIiIui/lOWbniAo6K+75YiwIGNsaXAg55qEIHN0dWR5IOaVsCwg56m6IHN0dWR5IOaVsCwg5bmz5Z2HIGNs")
parts.append("aXAg5pWwKSzpmLLmraLpnZnpu5jnmb3orq3nu4PjgIIiIiINCiAgICAgICAgICAgIHRvdGFsID0gc2VsZi5uX2hpdCArIHNlbGYubl9lbXB0eQ0KICAgICAgICAgICAgYXZnID0gKHNlbGYubl9jbGlwc190b3RhbCAvIHNlbGYubl9oaXQpIGlmIHNlbGYubl9oaXQg")
parts.append("ZWxzZSAwLjANCiAgICAgICAgICAgIHJldHVybiBzZWxmLm5faGl0LCBzZWxmLm5fZW1wdHksIGF2Zw0KDQogICAgICAgIGRlZiBfbG9hZF9zdHVkeShzZWxmLCB1aWQpOg0KICAgICAgICAgICAga2V5ID0gKHVpZCwgc2VsZi5wbGFuZSkNCiAgICAgICAgICAgIGlm")
parts.append("IGtleSBpbiBzZWxmLmNhY2hlOg0KICAgICAgICAgICAgICAgIHJldHVybiBzZWxmLmNhY2hlW2tleV0NCiAgICAgICAgICAgIHNyb3dzID0gc2VsZi5zZXJpZXNfZGZbc2VsZi5zZXJpZXNfZGZbIlN0dWR5SW5zdGFuY2VVSUQiXSA9PSB1aWRdDQogICAgICAgICAg")
parts.append("ICBpZiAiQW5hdG9taWNhbF9QbGFuZSIgaW4gc3Jvd3MuY29sdW1uczoNCiAgICAgICAgICAgICAgICBzcm93cyA9IHNyb3dzW3Nyb3dzWyJBbmF0b21pY2FsX1BsYW5lIl0gPT0gc2VsZi5wbGFuZV0NCiAgICAgICAgICAgICAgICBzcm93cyA9IHNyb3dzLnNvcnRf")
parts.append("dmFsdWVzKCJGbHVpZF9TZW5zaXRpdmUiLCBraW5kPSJzdGFibGUiKS5kcm9wX2R1cGxpY2F0ZXMoDQogICAgICAgICAgICAgICAgICAgIHN1YnNldD1bIkZsdWlkX1NlbnNpdGl2ZSJdKS5oZWFkKE1BWF9TRVJJRVMpDQogICAgICAgICAgICBlbHNlOg0KICAgICAg")
parts.append("ICAgICAgICAgIHNyb3dzID0gc3Jvd3MuaGVhZChNQVhfU0VSSUVTKQ0KICAgICAgICAgICAgY2xpcHMgPSBbXQ0KICAgICAgICAgICAgZm9yIF8sIHNyIGluIHNyb3dzLml0ZXJyb3dzKCk6DQogICAgICAgICAgICAgICAgc2lkID0gc3JbIlNlcmllc0luc3RhbmNl")
parts.append("VUlEIl0NCiAgICAgICAgICAgICAgICByb3dzID0gcmVhZF9kaWNvbV9zbGljZXMoDQogICAgICAgICAgICAgICAgICAgIHNlbGYuZGNtX3Jvb3QsIHNpZCwgc3R1ZHlfdWlkPXVpZCwgc2FtcGxpbmdfbW9kZT1zZWxmLnNhbXBsaW5nX21vZGUNCiAgICAgICAgICAg")
parts.append("ICAgICApDQogICAgICAgICAgICAgICAgY2xpcHMuZXh0ZW5kKA0KICAgICAgICAgICAgICAgICAgICBtYWtlX2NsaXBzKHJvd3MsIG5fY2xpcHM9c2VsZi5uX2NsaXBzLCBzYW1wbGluZ19tb2RlPXNlbGYuc2FtcGxpbmdfbW9kZSkNCiAgICAgICAgICAgICAgICAp")
parts.append("DQogICAgICAgICAgICBzZWxmLmNhY2hlW3VpZF0gPSBjbGlwcw0KICAgICAgICAgICAgIyDku4XpppbmrKHliqDovb3orqHmlbAo57yT5a2Y5ZG95Lit5LiN5YaN57Sv5YqgLOmBv+WFjeWkmiBlcG9jaCDph43lpI0pDQogICAgICAgICAgICBpZiBjbGlwczoNCiAg")
parts.append("ICAgICAgICAgICAgICBzZWxmLm5faGl0ICs9IDENCiAgICAgICAgICAgICAgICBzZWxmLm5fY2xpcHNfdG90YWwgKz0gbGVuKGNsaXBzKQ0KICAgICAgICAgICAgZWxzZToNCiAgICAgICAgICAgICAgICBzZWxmLm5fZW1wdHkgKz0gMQ0KICAgICAgICAgICAgcmV0")
parts.append("dXJuIGNsaXBzDQoNCiAgICAgICAgZGVmIF9fbGVuX18oc2VsZik6DQogICAgICAgICAgICByZXR1cm4gbGVuKHNlbGYuc3R1ZGllcykNCg0KICAgICAgICBkZWYgX19nZXRpdGVtX18oc2VsZiwgaSk6DQogICAgICAgICAgICB1aWQgPSBzZWxmLnN0dWRpZXNbaV0N")
parts.append("CiAgICAgICAgICAgIGNsaXBzID0gc2VsZi5fbG9hZF9zdHVkeSh1aWQpICAjIOWRveS4rS/nqbrorqHmlbDlnKggX2xvYWRfc3R1ZHkg6aaW5qyh5Yqg6L295pe25a6M5oiQDQogICAgICAgICAgICBpbl9jaCA9IHNlbGYubl9jbGlwcyAqIENMSVAgICMg6L6T5YWl")
parts.append("6YCa6YGT5pWwOmNsaXAg5pWwIMOXIOavjyBjbGlwIDMg5YiH54mHDQogICAgICAgICAgICBpZiBub3QgY2xpcHM6DQogICAgICAgICAgICAgICAgeCA9IG5wLnplcm9zKChpbl9jaCwgSU1HX1NJWkUsIElNR19TSVpFKSwgZHR5cGU9bnAuZmxvYXQzMikNCiAgICAg")
parts.append("ICAgICAgIGVsc2U6DQogICAgICAgICAgICAgICAgeCA9IG5wLmNvbmNhdGVuYXRlKGNsaXBzWzogc2VsZi5uX2NsaXBzXSwgYXhpcz0wKSAgIyAoayozLCBILCBXKQ0KICAgICAgICAgICAgICAgIHBhZCA9IGluX2NoIC0geC5zaGFwZVswXQ0KICAgICAgICAgICAg")
parts.append("ICAgIGlmIHBhZCA+IDA6DQogICAgICAgICAgICAgICAgICAgIHggPSBucC5jb25jYXRlbmF0ZShbeCwgbnAuemVyb3MoKHBhZCwgSU1HX1NJWkUsIElNR19TSVpFKSwgZHR5cGU9bnAuZmxvYXQzMildKQ0KICAgICAgICAgICAgICAgIGVsaWYgcGFkIDwgMDoNCiAg")
parts.append("ICAgICAgICAgICAgICAgICAgeCA9IHhbOiBpbl9jaF0NCiAgICAgICAgICAgIHggPSB0b3JjaC50ZW5zb3IoeCwgZHR5cGU9dG9yY2guZmxvYXQzMikNCiAgICAgICAgICAgIHkgPSB0b3JjaC56ZXJvcyhsZW4oTEFCRUxTKSwgZHR5cGU9dG9yY2guZmxvYXQzMikN")
parts.append("CiAgICAgICAgICAgIHZhbGlkID0gdG9yY2guemVyb3MobGVuKExBQkVMUyksIGR0eXBlPXRvcmNoLmZsb2F0MzIpDQogICAgICAgICAgICBpZiBzZWxmLmhhc190YXJnZXRzOg0KICAgICAgICAgICAgICAgIHJvdyA9IHNlbGYubGFiZWxzX2RmW3NlbGYubGFiZWxz")
parts.append("X2RmWyJTdHVkeUluc3RhbmNlVUlEIl0gPT0gdWlkXQ0KICAgICAgICAgICAgICAgIGlmIGxlbihyb3cpOg0KICAgICAgICAgICAgICAgICAgICBmb3IgaiwgYyBpbiBlbnVtZXJhdGUoTEFCRUxTKToNCiAgICAgICAgICAgICAgICAgICAgICAgIHYgPSByb3cuaWxv")
parts.append("Y1swXVtjXQ0KICAgICAgICAgICAgICAgICAgICAgICAgaWYgcGQubm90bmEodik6DQogICAgICAgICAgICAgICAgICAgICAgICAgICAgeVtqXSA9IGZsb2F0KHYpDQogICAgICAgICAgICAgICAgICAgICAgICAgICAgdmFsaWRbal0gPSAxLjANCiAgICAgICAgICAg")
parts.append("IHJldHVybiB4LCB5LCB2YWxpZA0KDQoNCiAgICAjIC0tLS0tLS0tLS0tLS0tLS0g5qih5Z6LIC0tLS0tLS0tLS0tLS0tLS0NCg0KICAgIF9QUkVUUkFJTkVEX1RSSUVEID0gW0ZhbHNlXSAgIyDmqKHlnZfnuqfmoIflv5c65LiL6L295aSx6LSl6L+H5LiA5qyh5bCx")
parts.append("5LiN5YaN5Y+N5aSN6YeN6K+VDQoNCg0KICAgIGRlZiBfYnVpbGRfcmVzbmV0MzQodXNlX3ByZXRyYWluZWQ9RmFsc2UsIG9mZmxpbmVfcGF0aHM9Tm9uZSk6DQogICAgICAgICIiIuaehOmAoCBSZXNOZXQzNCDlubbliqDovb0gSW1hZ2VOZXQg6aKE6K6t57uD5p2D")
parts.append("6YeN44CCDQoNCiAgICAgICAg5p2D6YeN5p2l5rqQ5LyY5YWI57qnOg0KICAgICAgICAxLiBvZmZsaW5lX3BhdGhzOuaMgui9veeahOemu+e6v+adg+mHjeaWh+S7tijmiavmj48gL2thZ2dsZS9pbnB1dCDlvpfliLAs5peg6ZyA6IGU572RKTsNCiAgICAgICAgMi4g")
parts.append("dXNlX3ByZXRyYWluZWQ9VHJ1ZTrogZTnvZHkuIvovb0gdG9yY2h2aXNpb24g5p2D6YeNKOW4pumHjeivlSzlpLHotKXlm57pgIDpmo/mnLopOw0KICAgICAgICAzLiDlkKbliJnpmo/mnLrliJ3lp4vljJbjgIINCiAgICAgICAg5oiQ5Yqf5Yqg6L295ZCO5p2D6YeN")
parts.append("55SxIHRvcmNoIOe8k+WtmCzlkI7nu60gZm9sZCDnp5LliqDovb3jgIINCiAgICAgICAgIiIiDQogICAgICAgIG5ldCA9IHR2bS5yZXNuZXQzNCh3ZWlnaHRzPU5vbmUpDQogICAgICAgIGlmIG9mZmxpbmVfcGF0aHM6DQogICAgICAgICAgICBvaywgZGVzYyA9IF90")
parts.append("cnlfbG9hZF9vZmZsaW5lKG5ldCwgb2ZmbGluZV9wYXRocykNCiAgICAgICAgICAgIGlmIG9rOg0KICAgICAgICAgICAgICAgIHJldHVybiBuZXQNCiAgICAgICAgICAgIHByaW50KGYiW+adg+mHjV0g56a757q/5YCZ6YCJ5Z2H5LiN5Yy56YWNKHtkZXNjfSks5Zue")
parts.append("6YCA5Yiw6ZqP5py6L+iBlOe9kei3r+W+hCIsIGZsdXNoPVRydWUpDQogICAgICAgIGlmIG5vdCB1c2VfcHJldHJhaW5lZDoNCiAgICAgICAgICAgIHByaW50KCJb5p2D6YeNXSDmnKrlvIDlkK8gLS1wcmV0cmFpbmVkIOS4lOaXoOemu+e6v+adg+mHjSzkvb/nlKjp")
parts.append("mo/mnLrliJ3lp4vljJYiDQogICAgICAgICAgICAgICAgICAiKOS4jeiBlOe9kTvlpoLpnIDpooTorq3nu4Por7fliqAgLS1wcmV0cmFpbmVkIOW5tuW8gOWQryBJbnRlcm5ldCwiDQogICAgICAgICAgICAgICAgICAi5oiW5oyC6L2956a757q/5p2D6YeN5pWw5o2u")
parts.append("6ZuGKSIsIGZsdXNoPVRydWUpDQogICAgICAgICAgICByZXR1cm4gbmV0DQogICAgICAgIGlmIF9QUkVUUkFJTkVEX1RSSUVEWzBdOg0KICAgICAgICAgICAgcmV0dXJuIG5ldCAgIyDkuYvliY3lt7Lnoa7orqTkuIvovb3kuI3kuoYs55u05o6l6ZqP5py6DQogICAg")
parts.append("ICAgIGZvciBhdHRlbXB0IGluIHJhbmdlKDEsIDQpOg0KICAgICAgICAgICAgdHJ5Og0KICAgICAgICAgICAgICAgIG5ldCA9IHR2bS5yZXNuZXQzNCh3ZWlnaHRzPXR2bS5SZXNOZXQzNF9XZWlnaHRzLklNQUdFTkVUMUtfVjENCiAgICAgICAgICAgICAgICAgICAg")
parts.append("ICAgICAgICAgICAgICAgaWYgaGFzYXR0cih0dm0sICJSZXNOZXQzNF9XZWlnaHRzIikgZWxzZSBOb25lKQ0KICAgICAgICAgICAgICAgIHJldHVybiBuZXQNCiAgICAgICAgICAgIGV4Y2VwdCBFeGNlcHRpb24gYXMgZToNCiAgICAgICAgICAgICAgICBwcmludChm")
parts.append("IlvmnYPph41dIFJlc05ldDM0IOmihOiuree7g+S4i+i9veWksei0pSjnrKwge2F0dGVtcHR9LzMg5qyhKToge2V9IiwgZmx1c2g9VHJ1ZSkNCiAgICAgICAgICAgICAgICB0aW1lLnNsZWVwKDUpDQogICAgICAgIF9QUkVUUkFJTkVEX1RSSUVEWzBdID0gVHJ1ZQ0K")
parts.append("ICAgICAgICBwcmludCgiW+adg+mHjV0gMyDmrKHlnYflpLHotKUs5pS555So6ZqP5py65Yid5aeL5YyWKOeyvuW6puS8muaYjuaYvuS4i+mZjTvlu7rorq7ph43or5XmiJbmjILovb3pooTorq3nu4PmnYPph43mlbDmja7pm4YpIiwgZmx1c2g9VHJ1ZSkNCiAgICAg")
parts.append("ICAgcmV0dXJuIHR2bS5yZXNuZXQzNCh3ZWlnaHRzPU5vbmUpDQoNCg0KICAgIGNsYXNzIEtuZWVNb2RlbChubi5Nb2R1bGUpOg0KICAgICAgICAiIiJSZXNOZXQzNCDlhbHkuqsgYmFja2JvbmUgKyDlhajlsYDmsaDljJYgKyAxMiBzaWdtb2lkIOWktCjmr4/kuKrl")
parts.append("ubPpnaLkuIDkuKrlrp7kvosp44CCIiIiDQoNCiAgICAgICAgZGVmIF9faW5pdF9fKHNlbGYsIG51bV9sYWJlbHM9bGVuKExBQkVMUyksIGNoYW5uZWxzPUNMSVAsIHByZXRyYWluZWQ9RmFsc2UsIG9mZmxpbmVfcGF0aHM9Tm9uZSk6DQogICAgICAgICAgICBzdXBl")
parts.append("cigpLl9faW5pdF9fKCkNCiAgICAgICAgICAgIGlmIEhBU19UVjoNCiAgICAgICAgICAgICAgICBzZWxmLmJhY2tib25lID0gX2J1aWxkX3Jlc25ldDM0KHVzZV9wcmV0cmFpbmVkPXByZXRyYWluZWQsDQogICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAg")
parts.append("ICAgICAgICAgICAgICBvZmZsaW5lX3BhdGhzPW9mZmxpbmVfcGF0aHMpDQogICAgICAgICAgICBlbHNlOg0KICAgICAgICAgICAgICAgIHJhaXNlIFJ1bnRpbWVFcnJvcigi6ZyA6KaBIHRvcmNodmlzaW9uKOivt+WcqCBLYWdnbGUgTm90ZWJvb2sgR1BVIOeOr+Wi")
parts.append("g+i/kOihjCkiKQ0KICAgICAgICAgICAgY29udjFfdyA9IHNlbGYuYmFja2JvbmUuY29udjEud2VpZ2h0ICAjIOS/neeVmemihOiuree7gyBjb252MSDmnYPph40NCiAgICAgICAgICAgIHNlbGYuYmFja2JvbmUuY29udjEgPSBubi5Db252MmQoY2hhbm5lbHMsIDY0")
parts.append("LCBrZXJuZWxfc2l6ZT03LCBzdHJpZGU9MiwgcGFkZGluZz0zLCBiaWFzPUZhbHNlKQ0KICAgICAgICAgICAgaWYgY29udjFfdy5zaGFwZSA9PSBzZWxmLmJhY2tib25lLmNvbnYxLndlaWdodC5zaGFwZTogICMgY2hhbm5lbHM9PTMg5pe255u05o6l5rK/55SoDQog")
parts.append("ICAgICAgICAgICAgICAgc2VsZi5iYWNrYm9uZS5jb252MS53ZWlnaHQuZGF0YS5jb3B5Xyhjb252MV93LmRhdGEpDQogICAgICAgICAgICBlbGlmIGNoYW5uZWxzICUgMyA9PSAwIGFuZCBjb252MV93LnNoYXBlWzFdID09IDM6DQogICAgICAgICAgICAgICAgIyAy")
parts.append("LjVEIOmAmumBk+aJqeWxlTrmioogMyDpgJrpgZPpooTorq3nu4PmnYPph43lpI3liLYgbl9jbGlwcyDku70o5bi46KeB5YGa5rOVKQ0KICAgICAgICAgICAgICAgIHJlcCA9IGNoYW5uZWxzIC8vIDMNCiAgICAgICAgICAgICAgICBzZWxmLmJhY2tib25lLmNvbnYx")
parts.append("LndlaWdodC5kYXRhLmNvcHlfKGNvbnYxX3cucmVwZWF0KDEsIHJlcCwgMSwgMSkpDQogICAgICAgICAgICAgICAgcHJpbnQoZiJb5p2D6YeNXSBjb252MSDpooTorq3nu4PmnYPph43lpI3liLYge3JlcH0g5Lu95omp5bGV5YiwIHtjaGFubmVsc30g6YCa6YGTIiwg")
parts.append("Zmx1c2g9VHJ1ZSkNCiAgICAgICAgICAgIHNlbGYuYmFja2JvbmUuZmMgPSBubi5JZGVudGl0eSgpDQogICAgICAgICAgICBzZWxmLmhlYWQgPSBubi5MaW5lYXIoNTEyLCBudW1fbGFiZWxzKQ0KDQogICAgICAgIGRlZiBmb3J3YXJkKHNlbGYsIHgpOg0KICAgICAg")
parts.append("ICAgICAgZmVhdCA9IHNlbGYuYmFja2JvbmUoeCkgICMgKEIsIDUxMikNCiAgICAgICAgICAgIHJldHVybiBzZWxmLmhlYWQoZmVhdCkgICAjIGxvZ2l0cyAoQiwgMTIpDQoNCg0KDQojIC0tLS0tLS0tLS0tLS0tLS0g6K6t57uDIC8g6aqM6K+BIC0tLS0tLS0tLS0t")
parts.append("LS0tLS0NCg0KZGVmIHRyYWluX2Vwb2NoKG1vZGVsLCBsb2FkZXIsIG9wdCwgY3JpdCwgZGV2aWNlKToNCiAgICBtb2RlbC50cmFpbigpDQogICAgdG90LCBjbnQgPSAwLjAsIDANCiAgICBmb3IgeCwgeSwgdmFsaWQgaW4gbG9hZGVyOg0KICAgICAgICB4LCB5LCB2")
parts.append("YWxpZCA9IHgudG8oZGV2aWNlKSwgeS50byhkZXZpY2UpLCB2YWxpZC50byhkZXZpY2UpDQogICAgICAgIG9wdC56ZXJvX2dyYWQoKQ0KICAgICAgICBvdXQgPSBtb2RlbCh4KQ0KICAgICAgICBpZiB2YWxpZC5zdW0oKSA9PSAwOg0KICAgICAgICAgICAgY29udGlu")
parts.append("dWUNCiAgICAgICAgIyBFWFAtMDEzOiBjb25maWRlbmNlLXdlaWdodCBzb2Z0IEJDRSDigJQgdyA9IDIqfHktMC41fCAoVU5LL3NpbGVuY2UgYWxyZWFkeSBOYU4pDQogICAgICAgIHcgPSAyLjAgKiAoeSAtIDAuNSkuYWJzKCkNCiAgICAgICAgZGVub20gPSAodmFs")
parts.append("aWQgKiB3KS5zdW0oKQ0KICAgICAgICBsb3NzID0gKGNyaXQob3V0LCB5KSAqIHZhbGlkICogdykuc3VtKCkgLyBkZW5vbS5jbGFtcChtaW49MWUtNikNCiAgICAgICAgbG9zcy5iYWNrd2FyZCgpDQogICAgICAgIG9wdC5zdGVwKCkNCiAgICAgICAgdG90ICs9IGxv")
parts.append("c3MuaXRlbSgpICogbGVuKHgpDQogICAgICAgIGNudCArPSBsZW4oeCkNCiAgICByZXR1cm4gdG90IC8gbWF4KGNudCwgMSkNCg0KDQpkZWYgcHJlZGljdChtb2RlbCwgbG9hZGVyLCBkZXZpY2UpOg0KICAgIG1vZGVsLmV2YWwoKQ0KICAgIHJvd3MgPSBbXQ0KICAg")
parts.append("IHdpdGggdG9yY2gubm9fZ3JhZCgpOg0KICAgICAgICBmb3IgeCwgXywgXyBpbiBsb2FkZXI6DQogICAgICAgICAgICB4ID0geC50byhkZXZpY2UpDQogICAgICAgICAgICBvdXQgPSB0b3JjaC5zaWdtb2lkKG1vZGVsKHgpKQ0KICAgICAgICAgICAgIyAuY3B1KCku")
parts.append("dG9saXN0KCkg6ICM6Z2eIC5jcHUoKS5udW1weSgpOnRvcmNoIDIuMi4wK2N1MTE4IOe8luS6jiBudW1weTEueCwNCiAgICAgICAgICAgICMg5ZyoIEthZ2dsZSBudW1weTIuMC4yIOS4iyB0b3JjaOKGkm51bXB5IGludGVyb3Ag5bSpO3RvbGlzdCDnuq8gQysrIOi3")
parts.append("r+W+hOWuieWFqOOAgg0KICAgICAgICAgICAgcm93cy5leHRlbmQob3V0LmNwdSgpLnRvbGlzdCgpKQ0KICAgIHJldHVybiBucC5hc2FycmF5KHJvd3MsIGR0eXBlPW5wLmZsb2F0MzIpIGlmIHJvd3MgZWxzZSBucC56ZXJvcygoMCwgbGVuKExBQkVMUykpLCBkdHlw")
parts.append("ZT1ucC5mbG9hdDMyKQ0KDQoNCmRlZiBydW5fZm9sZChtb2RlbCwgdHJhaW5fZHMsIHZhbF9kcywgZGV2aWNlLCBhcmdzKToNCiAgICB0cmFpbl9sb2FkZXIgPSBEYXRhTG9hZGVyKHRyYWluX2RzLCBiYXRjaF9zaXplPWFyZ3MuYmF0Y2gsIHNodWZmbGU9VHJ1ZSwN")
parts.append("CiAgICAgICAgICAgICAgICAgICAgICAgICAgICAgIG51bV93b3JrZXJzPWFyZ3Mud29ya2VycywgZHJvcF9sYXN0PUZhbHNlKQ0KICAgIHZhbF9sb2FkZXIgPSBEYXRhTG9hZGVyKHZhbF9kcywgYmF0Y2hfc2l6ZT1hcmdzLmJhdGNoLCBzaHVmZmxlPUZhbHNlLA0K")
parts.append("ICAgICAgICAgICAgICAgICAgICAgICAgICAgIG51bV93b3JrZXJzPWFyZ3Mud29ya2VycywgZHJvcF9sYXN0PUZhbHNlKQ0KICAgIG9wdCA9IEFkYW1XKG1vZGVsLnBhcmFtZXRlcnMoKSwgbHI9YXJncy5sciwgd2VpZ2h0X2RlY2F5PTFlLTQpDQogICAgc2NoZWQg")
parts.append("PSBDb3NpbmVBbm5lYWxpbmdMUihvcHQsIFRfbWF4PWFyZ3MuZXBvY2hzKQ0KICAgIGNyaXQgPSBubi5CQ0VXaXRoTG9naXRzTG9zcyhyZWR1Y3Rpb249Im5vbmUiKQ0KICAgIGZvciBlcCBpbiByYW5nZShhcmdzLmVwb2Nocyk6DQogICAgICAgIHQwID0gdGltZS50")
parts.append("aW1lKCkNCiAgICAgICAgbG9zcyA9IHRyYWluX2Vwb2NoKG1vZGVsLCB0cmFpbl9sb2FkZXIsIG9wdCwgY3JpdCwgZGV2aWNlKQ0KICAgICAgICBzY2hlZC5zdGVwKCkNCiAgICAgICAgaWYgYXJncy52ZXJib3NlOg0KICAgICAgICAgICAgcHJpbnQoZiIgIGVwb2No")
parts.append("IHtlcCArIDF9L3thcmdzLmVwb2Noc30gbG9zcyB7bG9zczouNGZ9ICh7dGltZS50aW1lKCkgLSB0MDouMGZ9cykiLCBmbHVzaD1UcnVlKQ0KICAgIHZhbF9wcmVkID0gcHJlZGljdChtb2RlbCwgdmFsX2xvYWRlciwgZGV2aWNlKQ0KICAgICMgRElDT00g5ZG95Lit")
parts.append("6K+K5patOuiuree7gyvpqozor4Hlt7Lov63ku6Plrozmr5Us5q2k5pe26K6h5pWw5omN5YeG56GuDQogICAgdGgsIGVoLCBhdCA9IHRyYWluX2RzLnN0YXRzKCkNCiAgICB2aCwgZXYsIGF2ID0gdmFsX2RzLnN0YXRzKCkNCiAgICBwcmludChmIlvmlbDmja5dIHRy")
parts.append("YWluIOWRveS4rSB7dGh9L3tsZW4odHJhaW5fZHMpfSBzdHVkeSjnqboge2VofSwg5bmz5Z2HIHthdDouMWZ9IGNsaXApIHwgIg0KICAgICAgICAgIGYidmFsIOWRveS4rSB7dmh9L3tsZW4odmFsX2RzKX0gc3R1ZHko56m6IHtldn0pIiwgZmx1c2g9VHJ1ZSkNCiAg")
parts.append("ICBpZiB0aCA9PSAwIG9yIHZoID09IDA6DQogICAgICAgIHByaW50KCJb6K2m5ZGKXSDmnInmipggRElDT00g5YWo6YOo5rKh6K+75YiwKOWRveS4rSAwKeKAlOKAlOi+k+WFpeWwhuaYr+WFqOmbtizliIbmlbDlv4XnhLYgfjAuNSEiLCBmbHVzaD1UcnVlKQ0KICAg")
parts.append("IHZhbF91aWQgPSBucC5hcnJheSh2YWxfZHMuc3R1ZGllcykgICMgc2h1ZmZsZT1GYWxzZSzpobrluo/kuI7mlbDmja7pm4bkuIDoh7QNCiAgICByZXR1cm4gdmFsX3ByZWQsIHZhbF91aWQNCg0KDQojIC0tLS0tLS0tLS0tLS0tLS0g5o6o55CGIC8g6J6N5ZCIIC8g")
parts.append("5o+Q5LqkIC0tLS0tLS0tLS0tLS0tLS0NCg0KZGVmIHByZWRpY3RfdGVzdChtb2RlbCwgdGVzdF9kZiwgc2VyaWVzX2RmLCByb290LCBkZXZpY2UsIHBsYW5lLCBuX2NsaXBzPTIsIHNhbXBsaW5nX21vZGU9ImxlZ2FjeSIpOg0KICAgICIiIuWvuSB0ZXN0IOavj+S4")
parts.append("qiBzdHVkeSDmjInmjIflrprlubPpnaLmlLbpm4YgY2xpcHMs6aKE5rWLIDEyIOS4quamgueOh+OAgiIiIg0KICAgIGRzID0gS25lZURhdGFzZXQodGVzdF9kZlsiU3R1ZHlJbnN0YW5jZVVJRCJdLnRvbGlzdCgpLCBOb25lLCByb290LCBzZXJpZXNfZGYsDQogICAg")
parts.append("ICAgICAgICAgICAgICAgICBwbGFuZSwgaGFzX3RhcmdldHM9RmFsc2UsIG5fY2xpcHM9bl9jbGlwcywgc2FtcGxpbmdfbW9kZT1zYW1wbGluZ19tb2RlKQ0KICAgIGxvYWRlciA9IERhdGFMb2FkZXIoZHMsIGJhdGNoX3NpemU9MTYsIHNodWZmbGU9RmFsc2UsIG51")
parts.append("bV93b3JrZXJzPTApDQogICAgcmV0dXJuIHByZWRpY3QobW9kZWwsIGxvYWRlciwgZGV2aWNlKQ0KDQoNCmRlZiBwbGFuZV9vb2ZfYXVjKG9vZiwgbGFiZWxzX2RmLCB1aWRfbGlzdCk6DQogICAgIiIi6K6h566X5LiA5Liq5bmz6Z2i5qih5Z6L5Zyo5oyH5a6aIHN0")
parts.append("dWR5IOWtkOmbhuS4iueahOWuj+W5s+WdhyBPT0YgQVVDKOWPqueul+acieagh+azqOeahOWIlynjgIINCg0KICAgIGxhYmVsc19kZiDlv4XpobvmmK/jgJDlrpjmlrkgZ29sZCDljp/lp4vooajjgJEoMC8xL05hTizlpoIgdHJhaW5fY3N2IOWOn+WIlyks5LiN6IO9")
parts.append("5LygDQogICAgbG9hZF9sYWJlbHMg5aGr5YWF5ZCO55qEIHNvZnQg6KGo4oCU4oCUc29mdCDkvKrmoIfnrb7ml6LpnZ4gZ3JvdW5kIHRydXRoLOS5n+S8muiuqQ0KICAgIHJvY19hdWNfc2NvcmUg5bSpKGNvbnRpbnVvdXMp44CCNTgg5L6L5bCP5qC35pys5LiL5p+Q")
parts.append("5YiX5Y+v6IO95Y+q5pyJ5Y2V5LiA57G75YirLOi3s+i/h+ivpeWIl+OAgg0KICAgICIiIg0KICAgIGF1Y3MgPSBbXQ0KICAgIGZvciBqLCBjIGluIGVudW1lcmF0ZShMQUJFTFMpOg0KICAgICAgICB5X3RydWUgPSBsYWJlbHNfZGYuc2V0X2luZGV4KCJTdHVkeUlu")
parts.append("c3RhbmNlVUlEIilbY10ucmVpbmRleCh1aWRfbGlzdCkudmFsdWVzLmFzdHlwZShmbG9hdCkNCiAgICAgICAgdmFsaWQgPSB+bnAuaXNuYW4oeV90cnVlKQ0KICAgICAgICBpZiB2YWxpZC5zdW0oKSA+PSAyMDoNCiAgICAgICAgICAgIHl2ID0geV90cnVlW3ZhbGlk")
parts.append("XQ0KICAgICAgICAgICAgaWYgbGVuKG5wLnVuaXF1ZSh5dikpIDwgMjoNCiAgICAgICAgICAgICAgICBjb250aW51ZQ0KICAgICAgICAgICAgYXVjcy5hcHBlbmQocm9jX2F1Y19zY29yZSh5diwgb29mW3ZhbGlkLCBqXSkpDQogICAgcmV0dXJuIGZsb2F0KG5wLm1l")
parts.append("YW4oYXVjcykpIGlmIGF1Y3MgZWxzZSAwLjANCg0KDQojIC0tLS0tLS0tLS0tLS0tLS0gYXJ0aWZhY3QgLyByZXN1bWUgLyBlbnNlbWJsZSBoZWxwZXJzIC0tLS0tLS0tLS0tLS0tLS0NCg0KZGVmIF9hcnRpZmFjdF9yb290KGFyZ3MpOg0KICAgIGJhc2UgPSBQYXRo")
parts.append("KGdldGF0dHIoYXJncywgImFydGlmYWN0X2RpciIsICIiKSBvciAiYXJ0aWZhY3RzIikNCiAgICBleHAgPSBnZXRhdHRyKGFyZ3MsICJleHBlcmltZW50X2lkIiwgImV4cF8wMDFfNWZvbGRfc2FnaXR0YWwiKQ0KICAgIHJldHVybiBiYXNlIC8gZXhwDQoNCg0KZGVm")
parts.append("IF9mb2xkX21ldGFfcGF0aChhcnRpZmFjdF9yb290LCBmb2xkKToNCiAgICByZXR1cm4gYXJ0aWZhY3Rfcm9vdCAvIGYiZm9sZHtmb2xkfSIgLyAibWV0YS5qc29uIg0KDQoNCmRlZiBfaXNfZm9sZF9kb25lKGFydGlmYWN0X3Jvb3QsIGZvbGQpOg0KICAgIHAgPSBf")
parts.append("Zm9sZF9tZXRhX3BhdGgoYXJ0aWZhY3Rfcm9vdCwgZm9sZCkNCiAgICBpZiBub3QgcC5leGlzdHMoKToNCiAgICAgICAgcmV0dXJuIEZhbHNlDQogICAgdHJ5Og0KICAgICAgICBtZXRhID0ganNvbi5sb2FkcyhwLnJlYWRfdGV4dChlbmNvZGluZz0idXRmLTgiKSkN")
parts.append("CiAgICAgICAgcmV0dXJuIG1ldGEuZ2V0KCJjb21wbGV0aW9uX3N0YXR1cyIpID09ICJkb25lIg0KICAgIGV4Y2VwdCBFeGNlcHRpb246DQogICAgICAgIHJldHVybiBGYWxzZQ0KDQoNCmRlZiBfc2F2ZV9mb2xkX2FydGlmYWN0KGFydGlmYWN0X3Jvb3QsIGZvbGQs")
parts.append("IHRlc3RfcHJlZCwgdGVzdF91aWRzLCBtZXRhKToNCiAgICBmb2xkX2RpciA9IGFydGlmYWN0X3Jvb3QgLyBmImZvbGR7Zm9sZH0iDQogICAgZm9sZF9kaXIubWtkaXIocGFyZW50cz1UcnVlLCBleGlzdF9vaz1UcnVlKQ0KICAgIG5wLnNhdmUoc3RyKGZvbGRfZGly")
parts.append("IC8gInRlc3RfcHJlZC5ucHkiKSwgdGVzdF9wcmVkLmFzdHlwZShucC5mbG9hdDMyKSkNCiAgICBwZC5EYXRhRnJhbWUoeyJTdHVkeUluc3RhbmNlVUlEIjogdGVzdF91aWRzfSkudG9fY3N2KA0KICAgICAgICBmb2xkX2RpciAvICJ0ZXN0X3VpZC5jc3YiLCBpbmRl")
parts.append("eD1GYWxzZQ0KICAgICkNCiAgICBtZXRhWyJjb21wbGV0aW9uX3N0YXR1cyJdID0gImRvbmUiDQogICAgbWV0YVsicHJlZGljdGlvbl9zaGFwZSJdID0gbGlzdCh0ZXN0X3ByZWQuc2hhcGUpDQogICAgKGZvbGRfZGlyIC8gIm1ldGEuanNvbiIpLndyaXRlX3RleHQo")
parts.append("DQogICAgICAgIGpzb24uZHVtcHMobWV0YSwgZW5zdXJlX2FzY2lpPUZhbHNlLCBpbmRlbnQ9MiksIGVuY29kaW5nPSJ1dGYtOCINCiAgICApDQoNCg0KZGVmIF9sb2FkX2ZvbGRfcHJlZGljdGlvbnMoYXJ0aWZhY3Rfcm9vdCwgZm9sZCwgY2Fub25pY2FsX3VpZHMp")
parts.append("Og0KICAgIGZvbGRfZGlyID0gYXJ0aWZhY3Rfcm9vdCAvIGYiZm9sZHtmb2xkfSINCiAgICBwcmVkID0gbnAubG9hZChmb2xkX2RpciAvICJ0ZXN0X3ByZWQubnB5IikNCiAgICB1aWRzID0gcGQucmVhZF9jc3YoZm9sZF9kaXIgLyAidGVzdF91aWQuY3N2IilbIlN0")
parts.append("dWR5SW5zdGFuY2VVSUQiXS50b2xpc3QoKQ0KICAgIGlmIHByZWQuc2hhcGUgIT0gKGxlbihjYW5vbmljYWxfdWlkcyksIGxlbihMQUJFTFMpKToNCiAgICAgICAgcmFpc2UgVmFsdWVFcnJvcihmImZvbGR7Zm9sZH0gc2hhcGUgbWlzbWF0Y2g6IHtwcmVkLnNoYXBl")
parts.append("fSB2cyBleHBlY3RlZCB7KGxlbihjYW5vbmljYWxfdWlkcyksIGxlbihMQUJFTFMpKX0iKQ0KICAgIGlmIHVpZHMgIT0gY2Fub25pY2FsX3VpZHM6DQogICAgICAgIHJhaXNlIFZhbHVlRXJyb3IoZiJmb2xke2ZvbGR9IFVJRCBvcmRlciBtaXNtYXRjaCIpDQogICAg")
parts.append("aWYgbm90IG5wLmlzZmluaXRlKHByZWQpLmFsbCgpOg0KICAgICAgICByYWlzZSBWYWx1ZUVycm9yKGYiZm9sZHtmb2xkfSBjb250YWlucyBOYU4vSW5mIikNCiAgICBpZiBwcmVkLm1pbigpIDwgMCBvciBwcmVkLm1heCgpID4gMToNCiAgICAgICAgcmFpc2UgVmFs")
parts.append("dWVFcnJvcihmImZvbGR7Zm9sZH0gcHJlZGljdGlvbnMgb3V0IG9mIFswLDFdIikNCiAgICByZXR1cm4gcHJlZA0KDQoNCmRlZiBfc2F2ZV9leHBlcmltZW50X21ldGEoYXJ0aWZhY3Rfcm9vdCwgdHJhaW5fZm9sZF91aWRzLCB0ZXN0X3VpZHMsIGFyZ3MpOg0KICAg")
parts.append("IG1ldGEgPSB7DQogICAgICAgICJleHBlcmltZW50X2lkIjogZ2V0YXR0cihhcmdzLCAiZXhwZXJpbWVudF9pZCIsICJleHBfMDAxXzVmb2xkX3NhZ2l0dGFsIiksDQogICAgICAgICJnaXRfY29tbWl0IjogX19naXRfY29tbWl0KCksDQogICAgICAgICJhcmdzIjog")
parts.append("dmFycyhhcmdzKSwNCiAgICAgICAgImxhYmVscyI6IExBQkVMUywNCiAgICAgICAgInRyYWluX2ZvbGQwX3VpZHMiOiB0cmFpbl9mb2xkX3VpZHMsDQogICAgICAgICJ0ZXN0X3VpZHMiOiB0ZXN0X3VpZHMsDQogICAgfQ0KICAgIChhcnRpZmFjdF9yb290IC8gImNv")
parts.append("bmZpZy5qc29uIikud3JpdGVfdGV4dCgNCiAgICAgICAganNvbi5kdW1wcyhtZXRhLCBlbnN1cmVfYXNjaWk9RmFsc2UsIGluZGVudD0yKSwgZW5jb2Rpbmc9InV0Zi04Ig0KICAgICkNCiAgICBwZC5EYXRhRnJhbWUoeyJTdHVkeUluc3RhbmNlVUlEIjogdGVzdF91")
parts.append("aWRzfSkudG9fY3N2KA0KICAgICAgICBhcnRpZmFjdF9yb290IC8gInRlc3RfdWlkLmNzdiIsIGluZGV4PUZhbHNlDQogICAgKQ0KICAgIGZvbGRfcm93cyA9IFtdDQogICAgZm9yIGsgaW4gcmFuZ2UoYXJncy5mb2xkcyk6DQogICAgICAgIGZvbGRfcm93cy5hcHBl")
parts.append("bmQoeyJmb2xkIjogaywgInRyYWluX3VpZHMiOiB0cmFpbl9mb2xkX3VpZHNba119KQ0KICAgIHBkLkRhdGFGcmFtZShmb2xkX3Jvd3MpLnRvX2NzdihhcnRpZmFjdF9yb290IC8gImZvbGRfYXNzaWdubWVudC5jc3YiLCBpbmRleD1GYWxzZSkNCg0KDQpkZWYgX21h")
parts.append("eWJlX2thZ2dsZV9kYXRhc2V0X3ZlcnNpb24oYXJ0aWZhY3Rfcm9vdCwgbWVzc2FnZSk6DQogICAgaWYgb3MuZW52aXJvbi5nZXQoIktBR0dMRV9LRVJORUxfUlVOX1RZUEUiKSAhPSAiSW50ZXJhY3RpdmUiOg0KICAgICAgICByZXR1cm4NCiAgICBzbHVnID0gb3Mu")
parts.append("ZW52aXJvbi5nZXQoIktBR0dMRV9LRVJORUxfU0xVRyIsICJyc25hLWFydGlmYWN0cy10bXAiKQ0KICAgIG93bmVyID0gb3MuZW52aXJvbi5nZXQoIktBR0dMRV9VU0VSTkFNRSIsICJ6aXlpbWluZyIpDQogICAgZGF0YXNldF9pZCA9IGYie293bmVyfS97c2x1Z30i")
parts.append("DQogICAgdHJ5Og0KICAgICAgICBzdWJwcm9jZXNzLnJ1bigNCiAgICAgICAgICAgIFsNCiAgICAgICAgICAgICAgICAia2FnZ2xlIiwNCiAgICAgICAgICAgICAgICAiZGF0YXNldHMiLA0KICAgICAgICAgICAgICAgICJ2ZXJzaW9uIiwNCiAgICAgICAgICAgICAg")
parts.append("ICAiLXAiLA0KICAgICAgICAgICAgICAgIHN0cihhcnRpZmFjdF9yb290KSwNCiAgICAgICAgICAgICAgICAiLW0iLA0KICAgICAgICAgICAgICAgIG1lc3NhZ2UsDQogICAgICAgICAgICAgICAgIi0tZGlyLW1vZGUiLA0KICAgICAgICAgICAgICAgICJ0YXIiLA0K")
parts.append("ICAgICAgICAgICAgXSwNCiAgICAgICAgICAgIGNoZWNrPVRydWUsDQogICAgICAgICAgICBjYXB0dXJlX291dHB1dD1UcnVlLA0KICAgICAgICApDQogICAgICAgIHByaW50KGYiW2FydGlmYWN0XSB1cGxvYWRlZCBkYXRhc2V0IHZlcnNpb246IHtkYXRhc2V0X2lk")
parts.append("fSIsIGZsdXNoPVRydWUpDQogICAgZXhjZXB0IEV4Y2VwdGlvbiBhcyBlOg0KICAgICAgICBwcmludChmIlthcnRpZmFjdF0gZGF0YXNldCB1cGxvYWQgc2tpcHBlZDoge2V9IiwgZmx1c2g9VHJ1ZSkNCg0KDQpkZWYgX19naXRfY29tbWl0KCk6DQogICAgdHJ5Og0K")
parts.append("ICAgICAgICBpbXBvcnQgc3VicHJvY2Vzcw0KICAgICAgICByZXR1cm4gc3VicHJvY2Vzcy5jaGVja19vdXRwdXQoDQogICAgICAgICAgICBbImdpdCIsICJyZXYtcGFyc2UiLCAiSEVBRCJdLCB0ZXh0PVRydWUNCiAgICAgICAgKS5zdHJpcCgpDQogICAgZXhjZXB0")
parts.append("IEV4Y2VwdGlvbjoNCiAgICAgICAgcmV0dXJuICJ1bmtub3duIg0KDQoNCmRlZiBtYWluKCk6DQogICAgYXAgPSBhcmdwYXJzZS5Bcmd1bWVudFBhcnNlcigpDQogICAgYXAuYWRkX2FyZ3VtZW50KCItLWVwb2NocyIsIHR5cGU9aW50LCBkZWZhdWx0PTIpDQogICAg")
parts.append("YXAuYWRkX2FyZ3VtZW50KCItLWJhdGNoIiwgdHlwZT1pbnQsIGRlZmF1bHQ9OCkNCiAgICBhcC5hZGRfYXJndW1lbnQoIi0tbHIiLCB0eXBlPWZsb2F0LCBkZWZhdWx0PTNlLTQpDQogICAgYXAuYWRkX2FyZ3VtZW50KCItLXdvcmtlcnMiLCB0eXBlPWludCwgZGVm")
parts.append("YXVsdD0wKQ0KICAgIGFwLmFkZF9hcmd1bWVudCgiLS1uLWNsaXBzIiwgdHlwZT1pbnQsIGRlZmF1bHQ9MikNCiAgICBhcC5hZGRfYXJndW1lbnQoIi0tc2FtcGxpbmctbW9kZSIsIHR5cGU9c3RyLCBkZWZhdWx0PSJsZWdhY3kiLA0KICAgICAgICAgICAgICAgICAg")
parts.append("ICBjaG9pY2VzPVsibGVnYWN5IiwgInBoeXNpY2FsIl0sDQogICAgICAgICAgICAgICAgICAgIGhlbHA9IkVYUC0wMTY6IGxlZ2FjeT16LXNvcnQraW5kZXggY2xpcHM7IHBoeXNpY2FsPUlPUCBub3JtYWwgc29ydCArIHBoeXNpY2FsIDIuNUQiKQ0KICAgIGFwLmFk")
parts.append("ZF9hcmd1bWVudCgiLS1zZWVkIiwgdHlwZT1pbnQsIGRlZmF1bHQ9U0VFRCkNCiAgICBhcC5hZGRfYXJndW1lbnQoIi0tZm9sZHMiLCB0eXBlPWludCwgZGVmYXVsdD1GT0xEUykNCiAgICBhcC5hZGRfYXJndW1lbnQoIi0tb25seS1mb2xkIiwgdHlwZT1pbnQsIGRl")
parts.append("ZmF1bHQ9LTEsDQogICAgICAgICAgICAgICAgICAgIGhlbHA9IuWPqui3keaMh+WumuaKmCjlhbbkvZnot7Pov4c76buY6K6kIC0xPeWFqOmDqCnjgILmjqLot6/nlKg6LS1mb2xkcyA1IC0tb25seS1mb2xkIDAg6K6t56ysIDAg5oqYIg0KICAgICAgICAgICAgICAg")
parts.append("ICAgICAgICAgICIodHJhaW4gNC81IOS4juato+W8j+WQjOWIhuW4gyksdGVzdCDmjqjnkIbnlKjor6XmipjmqKHlnovlh7rmj5DkuqQiKQ0KICAgIGFwLmFkZF9hcmd1bWVudCgiLS1wbGFuZXMiLCB0eXBlPXN0ciwgZGVmYXVsdD0iLCIuam9pbihQTEFORVMpLA0K")
parts.append("ICAgICAgICAgICAgICAgICAgICBoZWxwPSLpgJflj7fliIbpmpTnmoTlubPpnaLlrZDpm4Ys5aaCIGF4aWFsLHNhZ2l0dGFsKOWwj+WGmSzkuI3ljLrliIblpKflsI/lhpkpIikNCiAgICBhcC5hZGRfYXJndW1lbnQoIi0tZnVzZSIsIHR5cGU9c3RyLCBkZWZhdWx0")
parts.append("PSJ3ZWlnaHQiLA0KICAgICAgICAgICAgICAgICAgICBjaG9pY2VzPVsid2VpZ2h0IiwgIm1lYW4iXSwNCiAgICAgICAgICAgICAgICAgICAgaGVscD0i5bmz6Z2i6J6N5ZCI5pa55byPOndlaWdodD3mjIkgT09GIEFVQyDliqDmnYM7bWVhbj3nroDljZXlubPlnYci")
parts.append("KQ0KICAgIGFwLmFkZF9hcmd1bWVudCgiLS1wc2V1ZG8tY29uZiIsIHR5cGU9ZmxvYXQsIGRlZmF1bHQ9MC4wLA0KICAgICAgICAgICAgICAgICAgICBoZWxwPSLkvKrmoIfnrb7nva7kv6Hluqbov4fmu6TpmIjlgLwoMD3kuI3ov4fmu6Q7MC41PeS4ouW8g+e9ruS/")
parts.append("oeW6pjwwLjXnmoTkvKrmoIfnrb4pIikNCiAgICBhcC5hZGRfYXJndW1lbnQoIi0tbm8tcmVwb3J0LWxhYmVscyIsIGFjdGlvbj0ic3RvcmVfdHJ1ZSIsDQogICAgICAgICAgICAgICAgICAgIGhlbHA9IuemgeeUqOWGhee9ruaKpeWRiuWFs+mUruivjei9r+agh+et")
parts.append("vuWFnOW6lSjpu5jorqTlvIDlkK865pyq5oyC56S+5Yy65pWw5o2u6ZuG5pe2Ig0KICAgICAgICAgICAgICAgICAgICAgICAgICLoh6rliqjku44gNDQwNyDku73mlL7lsITmiqXlkYrnlJ/miJDova/moIfnrb4s6Zu25oyC6L29KSIpDQogICAgYXAuYWRkX2FyZ3Vt")
parts.append("ZW50KCItLXZlcmJvc2UiLCBhY3Rpb249InN0b3JlX3RydWUiKQ0KICAgIGFwLmFkZF9hcmd1bWVudCgiLS1wcmV0cmFpbmVkIiwgYWN0aW9uPSJzdG9yZV90cnVlIiwNCiAgICAgICAgICAgICAgICAgICAgaGVscD0i5bCd6K+V5LiL6L29IEltYWdlTmV0IOmihOiu")
parts.append("ree7g+adg+mHjSjpnIDlvIDlkK8gSW50ZXJuZXQpO+m7mOiupOmaj+acuuWIneWni+WMluS4jeiBlOe9kSIpDQogICAgYXAuYWRkX2FyZ3VtZW50KCItLW9mZmxpbmUtd2VpZ2h0cyIsIHR5cGU9c3RyLCBkZWZhdWx0PSIiLA0KICAgICAgICAgICAgICAgICAgICBo")
parts.append("ZWxwPSLpgJflj7fliIbpmpTnmoTnprvnur/mnYPph40gLnB0aCDot6/lvoQo5LyY5YWI57qn5pyA6auYKTvnlZnnqbrliJnoh6rliqjmiavmj4/mjILovb3nm67lvZUiKQ0KICAgIGFwLmFkZF9hcmd1bWVudCgiLS1uby1vZmZsaW5lLXNjYW4iLCBhY3Rpb249InN0")
parts.append("b3JlX3RydWUiLA0KICAgICAgICAgICAgICAgICAgICBoZWxwPSLnpoHnlKjoh6rliqjmiavmj48gL2thZ2dsZS9pbnB1dCDkuIvnmoTnprvnur/mnYPph43mlofku7YiKQ0KICAgIGFwLmFkZF9hcmd1bWVudCgiLS1kYXRhIiwgdHlwZT1zdHIsIGRlZmF1bHQ9IiIs")
parts.append("DQogICAgICAgICAgICAgICAgICAgIGhlbHA9IuaYvuW8j+aMh+WumuaVsOaNruagueebruW9lSjlpoIga2FnZ2xlaHViLmNvbXBldGl0aW9uX2Rvd25sb2FkIOi/lOWbnueahCBwYXRoKSwiDQogICAgICAgICAgICAgICAgICAgICAgICAgIuaMh+WumuWQjui3s+i/")
parts.append("h+iHquWKqOafpeaJviIpDQogICAgYXAuYWRkX2FyZ3VtZW50KCItLWFydGlmYWN0LWRpciIsIHR5cGU9c3RyLCBkZWZhdWx0PSJhcnRpZmFjdHMiLA0KICAgICAgICAgICAgICAgICAgICBoZWxwPSJhcnRpZmFjdCDmoLnnm67lvZUo6buY6K6kIGFydGlmYWN0cyki")
parts.append("KQ0KICAgIGFwLmFkZF9hcmd1bWVudCgiLS1leHBlcmltZW50LWlkIiwgdHlwZT1zdHIsIGRlZmF1bHQ9ImV4cF8wMDFfNWZvbGRfc2FnaXR0YWwiLA0KICAgICAgICAgICAgICAgICAgICBoZWxwPSJleHBlcmltZW50IElEKOeUqOS6jiBhcnRpZmFjdCDlrZDnm67l")
parts.append("vZUpIikNCiAgICBhcC5hZGRfYXJndW1lbnQoIi0tcmVzdW1lLWRpciIsIHR5cGU9c3RyLCBkZWZhdWx0PSIiLA0KICAgICAgICAgICAgICAgICAgICBoZWxwPSLlt7LmnIkgYXJ0aWZhY3Qg55uu5b2VLOWQr+WKqOaXtuaJq+aPjyBtZXRhLmpzb24g6Lez6L+H5bey")
parts.append("5a6M5oiQIGZvbGQiKQ0KICAgIGFyZ3MsIF8gPSBhcC5wYXJzZV9rbm93bl9hcmdzKCkNCg0KICAgIHJlc3VtZV9yb290ID0gUGF0aChhcmdzLnJlc3VtZV9kaXIpIGlmIGFyZ3MucmVzdW1lX2RpciBlbHNlIE5vbmUNCiAgICBpZiByZXN1bWVfcm9vdCBhbmQgYXJn")
parts.append("cy5hcnRpZmFjdF9kaXIgYW5kIG5vdCBhcmdzLmV4cGVyaW1lbnRfaWQ6DQogICAgICAgIGFyZ3MuZXhwZXJpbWVudF9pZCA9IHJlc3VtZV9yb290Lm5hbWUNCg0KICAgIGlmIG5vdCBIQVNfVE9SQ0ggb3Igbm90IEhBU19UViBvciBub3QgSEFTX1BZRElDT006DQog")
parts.append("ICAgICAgIHN5cy5leGl0KCLkvp3otZbnvLrlpLE66K+35ZyoIEthZ2dsZSBOb3RlYm9vayhHUFUp546v5aKD6L+Q6KGMKHRvcmNoL3RvcmNodmlzaW9uL3B5ZGljb20pO+acrOWcsOS7heWPr+WBmuaVsOaNruWxgumqjOivgeOAgiIpDQoNCiAgICBucC5yYW5kb20u")
parts.append("c2VlZChhcmdzLnNlZWQpDQogICAgdG9yY2gubWFudWFsX3NlZWQoYXJncy5zZWVkKQ0KDQogICAgaWYgYXJncy5kYXRhOg0KICAgICAgICBpZiBvcy5wYXRoLmV4aXN0cyhvcy5wYXRoLmpvaW4oYXJncy5kYXRhLCAidHJhaW4uY3N2IikpOg0KICAgICAgICAgICAg")
parts.append("cm9vdCwgb3V0ID0gYXJncy5kYXRhLCBhcmdzLmRhdGENCiAgICAgICAgICAgIHByaW50KGYiW+aVsOaNrl0g5L2/55SoIC0tZGF0YSDmjIflrprnm67lvZU6IHtyb290fSIpDQogICAgICAgIGVsc2U6DQogICAgICAgICAgICBzeXMuZXhpdChmIi0tZGF0YSDmjIfl")
parts.append("rprnm67lvZUge2FyZ3MuZGF0YX0g5LiL5rKh5pyJIHRyYWluLmNzdizor7fmo4Dmn6Xot6/lvoQiKQ0KICAgIGVsc2U6DQogICAgICAgIHJvb3QsIG91dCA9IHJlc29sdmVfcGF0aHMoKQ0KICAgIHByaW50KGYiW+aVsOaNrl0gcm9vdD17cm9vdH0gIG91dD17b3V0")
parts.append("fSIpDQoNCiAgICB0cmFpbl9jc3YgPSBwZC5yZWFkX2Nzdihvcy5wYXRoLmpvaW4ocm9vdCwgInRyYWluLmNzdiIpKQ0KICAgIGxhYmVsc19kZiA9IGxvYWRfbGFiZWxzKHJvb3QsIHBzZXVkb19jb25mPWFyZ3MucHNldWRvX2NvbmYsDQogICAgICAgICAgICAgICAg")
parts.append("ICAgICAgICAgICAgdXNlX3JlcG9ydF9sYWJlbHM9bm90IGFyZ3Mubm9fcmVwb3J0X2xhYmVscykNCiAgICBzZXJpZXNfZGYgPSBwZC5yZWFkX2Nzdihvcy5wYXRoLmpvaW4ocm9vdCwgInRyYWluX3Nlcmllcy5jc3YiKSkNCiAgICB0ZXN0X2RmID0gcGQucmVhZF9j")
parts.append("c3Yob3MucGF0aC5qb2luKHJvb3QsICJ0ZXN0LmNzdiIpKQ0KICAgIHRlc3Rfc2VyaWVzID0gcGQucmVhZF9jc3Yob3MucGF0aC5qb2luKHJvb3QsICJ0ZXN0X3Nlcmllcy5jc3YiKSkNCiAgICBkY21fdHJhaW4gPSBvcy5wYXRoLmpvaW4ocm9vdCwgInRyYWluX3Nl")
parts.append("cmllcyIpDQogICAgZGNtX3Rlc3QgPSBvcy5wYXRoLmpvaW4ocm9vdCwgInRlc3Rfc2VyaWVzIikNCiAgICBwcmludChmIlvmlbDmja5dIHRyYWluIHtsZW4odHJhaW5fY3N2KX0gc3R1ZHkgLyB0ZXN0IHtsZW4odGVzdF9kZil9IHN0dWR5IC8gc2VyaWVzIHtsZW4o")
parts.append("c2VyaWVzX2RmKX0iKQ0KICAgIF9wcmludF9kY21fdHJlZShyb290LCAiRElDT00iKSAgIyDmiZPljbAgdHJhaW5fc2VyaWVzIOecn+WunuebruW9lee7k+aehCznoa7orqTluIPlsYANCg0KICAgIHByaW50KGYiW+mHh+agt10gc2FtcGxpbmdfbW9kZT17YXJncy5z")
parts.append("YW1wbGluZ19tb2RlfSBuX2NsaXBzPXthcmdzLm5fY2xpcHN9ICINCiAgICAgICAgICBmIkhBU19TRz17SEFTX1NHfSIpDQogICAgcGxhbmVzID0gW3Auc3RyaXAoKS50aXRsZSgpIGZvciBwIGluIGFyZ3MucGxhbmVzLnNwbGl0KCIsIikgaWYgcC5zdHJpcCgpXQ0K")
parts.append("ICAgIHBsYW5lcyA9IFtwIGZvciBwIGluIHBsYW5lcyBpZiBwIGluIFBMQU5FU10NCiAgICBpZiBub3QgcGxhbmVzOg0KICAgICAgICBzeXMuZXhpdChmIi0tcGxhbmVzIOaXoOaViCzlj6/pgIk6IHtQTEFORVN9IikNCiAgICBwcmludChmIlvlubPpnaJdIOS9v+eU")
parts.append("qCB7cGxhbmVzfSIpDQoNCiAgICBkZXZpY2UgPSB0b3JjaC5kZXZpY2UoImN1ZGEiIGlmIHRvcmNoLmN1ZGEuaXNfYXZhaWxhYmxlKCkgZWxzZSAiY3B1IikNCiAgICBwcmludChmIlvorr7lpIddIHtkZXZpY2V9IikNCg0KICAgICMg56a757q/6aKE6K6t57uD5p2D")
parts.append("6YeNOuaYvuW8j+aMh+WumuS8mOWFiCzlkKbliJnoh6rliqjmiavmj48gL2thZ2dsZS9pbnB1dA0KICAgIG9mZmxpbmVfcGF0aHMgPSBbXQ0KICAgIGlmIGFyZ3Mub2ZmbGluZV93ZWlnaHRzLnN0cmlwKCk6DQogICAgICAgIGZvciBwIGluIFt4LnN0cmlwKCkgZm9y")
parts.append("IHggaW4gYXJncy5vZmZsaW5lX3dlaWdodHMuc3BsaXQoIiwiKSBpZiB4LnN0cmlwKCldOg0KICAgICAgICAgICAgaWYgb3MucGF0aC5pc2ZpbGUocCk6DQogICAgICAgICAgICAgICAgb2ZmbGluZV9wYXRocy5hcHBlbmQoKHAsIHApKQ0KICAgICAgICAgICAgZWxz")
parts.append("ZToNCiAgICAgICAgICAgICAgICBwcmludChmIlvmnYPph41dIC0tb2ZmbGluZS13ZWlnaHRzIOaMh+WumueahOaWh+S7tuS4jeWtmOWcqCzot7Pov4c6IHtwfSIsIGZsdXNoPVRydWUpDQogICAgZWxpZiBub3QgYXJncy5ub19vZmZsaW5lX3NjYW46DQogICAgICAg")
parts.append("IG9mZmxpbmVfcGF0aHMgPSBfc2Nhbl9vZmZsaW5lX3dlaWdodHMoKQ0KICAgICAgICBpZiBvZmZsaW5lX3BhdGhzOg0KICAgICAgICAgICAgcHJpbnQoZiJb5p2D6YeNXSDoh6rliqjmiavmj4/liLAge2xlbihvZmZsaW5lX3BhdGhzKX0g5Liq56a757q/5p2D6YeN")
parts.append("5YCZ6YCJOiIsIGZsdXNoPVRydWUpDQogICAgICAgICAgICBmb3IgcCwgZGVzYyBpbiBvZmZsaW5lX3BhdGhzWzo2XToNCiAgICAgICAgICAgICAgICBwcmludChmIiAge2Rlc2N9IiwgZmx1c2g9VHJ1ZSkNCiAgICAgICAgICAgIGlmIGxlbihvZmZsaW5lX3BhdGhz")
parts.append("KSA+IDY6DQogICAgICAgICAgICAgICAgcHJpbnQoZiIgIC4uLiDlhbEge2xlbihvZmZsaW5lX3BhdGhzKX0g5LiqIiwgZmx1c2g9VHJ1ZSkNCiAgICAgICAgZWxzZToNCiAgICAgICAgICAgIHByaW50KCJb5p2D6YeNXSDmnKrlnKjmjILovb3nm67lvZXlj5HnjrDn")
parts.append("prvnur/mnYPph40o5Y+v5oyC6L2956S+5Yy65omT5YyF55qEIHJlc25ldDM0IOadg+mHjeaVsOaNrumbhikiLCBmbHVzaD1UcnVlKQ0KICAgICAgICAgICAgX3ByaW50X21vdW50X3RyZWUoIi9rYWdnbGUvaW5wdXQiLCAi5p2D6YeNIikNCg0KICAgIGdrZiA9IEdy")
parts.append("b3VwS0ZvbGQobl9zcGxpdHM9YXJncy5mb2xkcykNCiAgICBncm91cHMgPSB0cmFpbl9jc3ZbIlN0dWR5SW5zdGFuY2VVSUQiXS52YWx1ZXMNCg0KICAgICMg5q+P5Liq5bmz6Z2i54us56uL6LeR5oqYLOiusOW9lSBPT0Yg5LiOIHRlc3Qg6aKE5rWLDQogICAgcGxh")
parts.append("bmVfb29mLCBwbGFuZV90ZXN0LCBwbGFuZV9hdWMsIHBsYW5lX3VpZHMgPSB7fSwge30sIHt9LCB7fQ0KICAgIGZvciBwbGFuZSBpbiBwbGFuZXM6DQogICAgICAgIHByaW50KGYiXG4jIyMjIyMjIyMjIOW5s+mdojoge3BsYW5lfSAjIyMjIyMjIyMjIikNCiAgICAg")
parts.append("ICAgb29mID0gbnAuemVyb3MoKGxlbih0cmFpbl9jc3YpLCBsZW4oTEFCRUxTKSkpDQogICAgICAgIGFydGlmYWN0X3Jvb3QgPSBfYXJ0aWZhY3Rfcm9vdChhcmdzKQ0KICAgICAgICBjYW5vbmljYWxfdGVzdF91aWRzID0gdGVzdF9kZlsiU3R1ZHlJbnN0YW5jZVVJ")
parts.append("RCJdLnRvbGlzdCgpDQogICAgICAgIHRyYWluX2ZvbGRfdWlkcyA9IFtdDQogICAgICAgIGZvciBrLCAodHJfaWR4LCB2YV9pZHgpIGluIGVudW1lcmF0ZShna2Yuc3BsaXQodHJhaW5fY3N2LCBncm91cHM9Z3JvdXBzKSk6DQogICAgICAgICAgICB0cl9zdHVkaWVz")
parts.append("ID0gdHJhaW5fY3N2Lmlsb2NbdHJfaWR4XVsiU3R1ZHlJbnN0YW5jZVVJRCJdLnRvbGlzdCgpDQogICAgICAgICAgICB2YV9zdHVkaWVzID0gdHJhaW5fY3N2Lmlsb2NbdmFfaWR4XVsiU3R1ZHlJbnN0YW5jZVVJRCJdLnRvbGlzdCgpDQogICAgICAgICAgICB0cmFp")
parts.append("bl9mb2xkX3VpZHMuYXBwZW5kKHRyX3N0dWRpZXMpDQogICAgICAgICAgICBpZiBhcmdzLm9ubHlfZm9sZCA+PSAwIGFuZCBrICE9IGFyZ3Mub25seV9mb2xkOg0KICAgICAgICAgICAgICAgIGNvbnRpbnVlDQogICAgICAgICAgICBpZiByZXN1bWVfcm9vdCBhbmQg")
parts.append("X2lzX2ZvbGRfZG9uZShhcnRpZmFjdF9yb290LCBrKToNCiAgICAgICAgICAgICAgICBwcmludChmIlthcnRpZmFjdF0gZm9sZCB7a30gZG9uZSwgc2tpcCIsIGZsdXNoPVRydWUpDQogICAgICAgICAgICAgICAgY29udGludWUNCiAgICAgICAgICAgIHRyX2RzID0g")
parts.append("S25lZURhdGFzZXQodHJfc3R1ZGllcywgbGFiZWxzX2RmLCBkY21fdHJhaW4sIHNlcmllc19kZiwgcGxhbmUsDQogICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgIG5fY2xpcHM9YXJncy5uX2NsaXBzLCBzYW1wbGluZ19tb2RlPWFyZ3Muc2FtcGxpbmdfbW9k")
parts.append("ZSkNCiAgICAgICAgICAgIHZhX2RzID0gS25lZURhdGFzZXQodmFfc3R1ZGllcywgbGFiZWxzX2RmLCBkY21fdHJhaW4sIHNlcmllc19kZiwgcGxhbmUsDQogICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgIG5fY2xpcHM9YXJncy5uX2NsaXBzLCBzYW1wbGlu")
parts.append("Z19tb2RlPWFyZ3Muc2FtcGxpbmdfbW9kZSkNCiAgICAgICAgICAgIG1vZGVsID0gS25lZU1vZGVsKHByZXRyYWluZWQ9YXJncy5wcmV0cmFpbmVkLA0KICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgb2ZmbGluZV9wYXRocz1vZmZsaW5lX3BhdGhzLA0KICAg")
parts.append("ICAgICAgICAgICAgICAgICAgICAgICAgICAgY2hhbm5lbHM9YXJncy5uX2NsaXBzICogQ0xJUCkudG8oZGV2aWNlKQ0KICAgICAgICAgICAgcHJpbnQoZiJb5oqYIHtrICsgMX0ve2FyZ3MuZm9sZHN9XSB7cGxhbmV9IHRyYWluIHtsZW4odHJfZHMpfSAvIHZhbCB7")
parts.append("bGVuKHZhX2RzKX0iLCBmbHVzaD1UcnVlKQ0KICAgICAgICAgICAgdDAgPSB0aW1lLnRpbWUoKQ0KICAgICAgICAgICAgdmFsX3ByZWQsIHZhbF91aWQgPSBydW5fZm9sZChtb2RlbCwgdHJfZHMsIHZhX2RzLCBkZXZpY2UsIGFyZ3MpDQogICAgICAgICAgICBmb2xk")
parts.append("X3RyYWluX3RpbWUgPSB0aW1lLnRpbWUoKSAtIHQwDQogICAgICAgICAgICBmb3IgaiwgdWlkIGluIGVudW1lcmF0ZSh2YWxfdWlkKToNCiAgICAgICAgICAgICAgICByb3cgPSBucC53aGVyZSh0cmFpbl9jc3ZbIlN0dWR5SW5zdGFuY2VVSUQiXS52YWx1ZXMgPT0g")
parts.append("dWlkKVswXQ0KICAgICAgICAgICAgICAgIGlmIGxlbihyb3cpOg0KICAgICAgICAgICAgICAgICAgICBvb2Zbcm93WzBdXSA9IHZhbF9wcmVkW2pdDQogICAgICAgICAgICB0MCA9IHRpbWUudGltZSgpDQogICAgICAgICAgICB0ZXN0X3ByZWQgPSBwcmVkaWN0X3Rl")
parts.append("c3QobW9kZWwsIHRlc3RfZGYsIHRlc3Rfc2VyaWVzLCBkY21fdGVzdCwgZGV2aWNlLCBwbGFuZSwNCiAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICBuX2NsaXBzPWFyZ3Mubl9jbGlwcywgc2FtcGxpbmdfbW9kZT1hcmdzLnNhbXBsaW5nX21vZGUp")
parts.append("DQogICAgICAgICAgICBmb2xkX2luZmVyZW5jZV90aW1lID0gdGltZS50aW1lKCkgLSB0MA0KICAgICAgICAgICAgbWV0YSA9IHsNCiAgICAgICAgICAgICAgICAiZm9sZCI6IGssDQogICAgICAgICAgICAgICAgInBsYW5lIjogcGxhbmUsDQogICAgICAgICAgICAg")
parts.append("ICAgIm5fY2xpcHMiOiBhcmdzLm5fY2xpcHMsDQogICAgICAgICAgICAgICAgInNhbXBsaW5nX21vZGUiOiBnZXRhdHRyKGFyZ3MsICJzYW1wbGluZ19tb2RlIiwgImxlZ2FjeSIpLA0KICAgICAgICAgICAgICAgICJpbWFnZV9zaXplIjogSU1HX1NJWkUsDQogICAg")
parts.append("ICAgICAgICAgICAgImVwb2NocyI6IGFyZ3MuZXBvY2hzLA0KICAgICAgICAgICAgICAgICJiYXRjaF9zaXplIjogYXJncy5iYXRjaCwNCiAgICAgICAgICAgICAgICAic2VlZCI6IGFyZ3Muc2VlZCwNCiAgICAgICAgICAgICAgICAiZ2l0X2NvbW1pdCI6IF9fZ2l0")
parts.append("X2NvbW1pdCgpLA0KICAgICAgICAgICAgICAgICJmb2xkX3RyYWluX3RpbWVfc2VjIjogZm9sZF90cmFpbl90aW1lLA0KICAgICAgICAgICAgICAgICJmb2xkX2luZmVyZW5jZV90aW1lX3NlYyI6IGZvbGRfaW5mZXJlbmNlX3RpbWUsDQogICAgICAgICAgICB9DQog")
parts.append("ICAgICAgICAgICBfc2F2ZV9mb2xkX2FydGlmYWN0KGFydGlmYWN0X3Jvb3QsIGssIHRlc3RfcHJlZCwgY2Fub25pY2FsX3Rlc3RfdWlkcywgbWV0YSkNCiAgICAgICAgICAgIHRyeToNCiAgICAgICAgICAgICAgICB3ZGlyID0gYXJ0aWZhY3Rfcm9vdCAvIGYiZm9s")
parts.append("ZHtrfSIgLyBwbGFuZQ0KICAgICAgICAgICAgICAgIHdkaXIubWtkaXIocGFyZW50cz1UcnVlLCBleGlzdF9vaz1UcnVlKQ0KICAgICAgICAgICAgICAgIHdwYXRoID0gd2RpciAvICJtb2RlbC5wdCINCiAgICAgICAgICAgICAgICB0b3JjaC5zYXZlKG1vZGVsLnN0")
parts.append("YXRlX2RpY3QoKSwgc3RyKHdwYXRoKSkNCiAgICAgICAgICAgICAgICBtZXRhMiA9IGRpY3QobWV0YSkNCiAgICAgICAgICAgICAgICBtZXRhMlsiY2hhbm5lbHMiXSA9IGludChhcmdzLm5fY2xpcHMgKiBDTElQKQ0KICAgICAgICAgICAgICAgIG1ldGEyWyJudW1f")
parts.append("bGFiZWxzIl0gPSBsZW4oTEFCRUxTKQ0KICAgICAgICAgICAgICAgICh3ZGlyIC8gIm1vZGVsX21ldGEuanNvbiIpLndyaXRlX3RleHQoDQogICAgICAgICAgICAgICAgICAgIGpzb24uZHVtcHMobWV0YTIsIGVuc3VyZV9hc2NpaT1GYWxzZSwgaW5kZW50PTIpLCBl")
parts.append("bmNvZGluZz0idXRmLTgiKQ0KICAgICAgICAgICAgICAgIHByaW50KGYiW3dlaWdodHNdIHNhdmVkIHt3cGF0aH0iLCBmbHVzaD1UcnVlKQ0KICAgICAgICAgICAgZXhjZXB0IEV4Y2VwdGlvbiBhcyBlOg0KICAgICAgICAgICAgICAgIHByaW50KGYiW3dlaWdodHNd")
parts.append("IHNhdmUgZmFpbGVkOiB7ZX0iLCBmbHVzaD1UcnVlKQ0KICAgICAgICAgICAgcHJpbnQoZiJbYXJ0aWZhY3RdIHNhdmVkIGZvbGR7a30gLT4ge2FydGlmYWN0X3Jvb3QgLyBmJ2ZvbGR7a30nfSIsIGZsdXNoPVRydWUpDQogICAgICAgICAgICBpZiBIQVNfU0cgYW5k")
parts.append("IGdldGF0dHIoYXJncywgInNhbXBsaW5nX21vZGUiLCAibGVnYWN5IikgPT0gInBoeXNpY2FsIjoNCiAgICAgICAgICAgICAgICBpbXBvcnQganNvbiBhcyBfanNvbg0KICAgICAgICAgICAgICAgIHN0ID0gU0cuZHVtcF9zdGF0cygpDQogICAgICAgICAgICAgICAg")
parts.append("c3AgPSBhcnRpZmFjdF9yb290IC8gInNhbXBsaW5nX3N0YXRzLmpzb24iDQogICAgICAgICAgICAgICAgc3Aud3JpdGVfdGV4dChfanNvbi5kdW1wcyhzdCwgaW5kZW50PTIpLCBlbmNvZGluZz0idXRmLTgiKQ0KICAgICAgICAgICAgICAgIHByaW50KGYiW3NhbXBs")
parts.append("aW5nLXN0YXRzXSB7c3R9IiwgZmx1c2g9VHJ1ZSkNCiAgICAgICAgIyBleHBlcmltZW50LWxldmVsIG1ldGEgb25jZSBwZXIgcGxhbmUNCiAgICAgICAgX3NhdmVfZXhwZXJpbWVudF9tZXRhKGFydGlmYWN0X3Jvb3QsIHRyYWluX2ZvbGRfdWlkcywgY2Fub25pY2Fs")
parts.append("X3Rlc3RfdWlkcywgYXJncykNCiAgICAgICAgcHJpbnQoZiJbYXJ0aWZhY3RdIHNhdmVkIGV4cGVyaW1lbnQgbWV0YSAtPiB7YXJ0aWZhY3Rfcm9vdH0iLCBmbHVzaD1UcnVlKQ0KICAgICAgICAjIOivpeW5s+mdoiBPT0YgQVVDKOWuj+W5s+WdhykNCiAgICAgICAg")
parts.append("dWlkX2xpc3QgPSB0cmFpbl9jc3ZbIlN0dWR5SW5zdGFuY2VVSUQiXS50b2xpc3QoKQ0KICAgICAgICBpZiBhcmdzLm9ubHlfZm9sZCA+PSAwOg0KICAgICAgICAgICAgcHJpbnQoZiJbe3BsYW5lfV0g5o6i6Lev5qih5byPKC0tb25seS1mb2xkIHthcmdzLm9ubHlf")
parts.append("Zm9sZH0pOk9PRiDljZXmipjkuI3lrozmlbQsIg0KICAgICAgICAgICAgICAgICAgZiLot7Pov4cgT09GIEFVQyjmnYPph43orrAgMD3lnYfnrYkpO3Rlc3Qg5o6o55CG55So6K+l5oqY5qih5Z6LIikNCiAgICAgICAgICAgIGF1YyA9IDAuMA0KICAgICAgICBlbHNl")
parts.append("Og0KICAgICAgICAgICAgYXVjID0gcGxhbmVfb29mX2F1Yyhvb2YsIHRyYWluX2NzdltbIlN0dWR5SW5zdGFuY2VVSUQiXSArIExBQkVMU10sIHVpZF9saXN0KQ0KICAgICAgICAgICAgcHJpbnQoZiJbe3BsYW5lfV0gT09GIOWuj+W5s+WdhyBBVUMge2F1YzouNGZ9")
parts.append("IikNCiAgICAgICAgcGxhbmVfb29mW3BsYW5lXSwgcGxhbmVfYXVjW3BsYW5lXSwgcGxhbmVfdWlkc1twbGFuZV0gPSBvb2YsIGF1YywgdWlkX2xpc3QNCiAgICAgICAgIyBsb2FkIHNhdmVkIGZvbGQgcHJlZGljdGlvbnMgZm9yIHRoaXMgcGxhbmUNCiAgICAgICAg")
parts.append("cGVyX2ZvbGQgPSBbXQ0KICAgICAgICBtaXNzaW5nID0gW10NCiAgICAgICAgZm9yIGsgaW4gcmFuZ2UoYXJncy5mb2xkcyk6DQogICAgICAgICAgICBwID0gYXJ0aWZhY3Rfcm9vdCAvIGYiZm9sZHtrfSIgLyAidGVzdF9wcmVkLm5weSINCiAgICAgICAgICAgIGlm")
parts.append("IG5vdCBwLmV4aXN0cygpOg0KICAgICAgICAgICAgICAgIG1pc3NpbmcuYXBwZW5kKGspDQogICAgICAgICAgICAgICAgY29udGludWUNCiAgICAgICAgICAgIHRyeToNCiAgICAgICAgICAgICAgICBwZXJfZm9sZC5hcHBlbmQoX2xvYWRfZm9sZF9wcmVkaWN0aW9u")
parts.append("cyhhcnRpZmFjdF9yb290LCBrLCBjYW5vbmljYWxfdGVzdF91aWRzKSkNCiAgICAgICAgICAgIGV4Y2VwdCBFeGNlcHRpb24gYXMgZToNCiAgICAgICAgICAgICAgICBtaXNzaW5nLmFwcGVuZChrKQ0KICAgICAgICAgICAgICAgIHByaW50KGYiW2Vuc2VtYmxlXSBm")
parts.append("b2xke2t9IGludmFsaWQ6IHtlfSIsIGZsdXNoPVRydWUpDQogICAgICAgIGlmIGxlbihtaXNzaW5nKSA9PSBhcmdzLmZvbGRzOg0KICAgICAgICAgICAgc3lzLmV4aXQoZiJbZW5zZW1ibGVdIG5vIHZhbGlkIGZvbGQgcHJlZGljdGlvbnMgYXZhaWxhYmxlIHVuZGVy")
parts.append("IHthcnRpZmFjdF9yb290fSIpDQogICAgICAgIGlmIG1pc3Npbmc6DQogICAgICAgICAgICBwcmludChmIltlbnNlbWJsZV0gbWlzc2luZyBmb2xkczoge21pc3Npbmd9OyBlbnNlbWJsZSB3aXRoIGF2YWlsYWJsZSBmb2xkcyBvbmx5IiwgZmx1c2g9VHJ1ZSkNCiAg")
parts.append("ICAgICAgcGxhbmVfdGVzdFtwbGFuZV0gPSBucC5tZWFuKHBlcl9mb2xkLCBheGlzPTApIGlmIHBlcl9mb2xkIGVsc2UgTm9uZQ0KICAgICAgICBpZiBwbGFuZV90ZXN0W3BsYW5lXSBpcyBub3QgTm9uZToNCiAgICAgICAgICAgIHByaW50KGYiW2Vuc2VtYmxlXSB7")
parts.append("cGxhbmV9IGVuc2VtYmxlZCBmcm9tIHtsZW4ocGVyX2ZvbGQpfSBmb2xkcywgc2hhcGU9e3BsYW5lX3Rlc3RbcGxhbmVdLnNoYXBlfSIpDQoNCiAgICAjIOiejeWQiA0KICAgIGlmIGFyZ3MuZnVzZSA9PSAid2VpZ2h0IjoNCiAgICAgICAgd3N1bSA9IHN1bShtYXgo")
parts.append("YSwgMWUtNikgZm9yIGEgaW4gcGxhbmVfYXVjLnZhbHVlcygpKQ0KICAgICAgICB3ZWlnaHRzID0ge3A6IG1heChhLCAxZS02KSAvIHdzdW0gZm9yIHAsIGEgaW4gcGxhbmVfYXVjLml0ZW1zKCl9DQogICAgICAgIHByaW50KGYiW+iejeWQiF0gT09GIEFVQyDliqDm")
parts.append("nYM6IHsge3A6IHJvdW5kKHcsIDQpIGZvciBwLCB3IGluIHdlaWdodHMuaXRlbXMoKX0gfSIpDQogICAgZWxzZToNCiAgICAgICAgd2VpZ2h0cyA9IHtwOiAxLjAgLyBsZW4ocGxhbmVzKSBmb3IgcCBpbiBwbGFuZXN9DQogICAgICAgIHByaW50KCJb6J6N5ZCIXSDn")
parts.append("roDljZXlubPlnYciKQ0KICAgIGZ1c2VkID0gbnAuemVyb3NfbGlrZShwbGFuZV90ZXN0W3BsYW5lc1swXV0pDQogICAgZm9yIHAsIHcgaW4gd2VpZ2h0cy5pdGVtcygpOg0KICAgICAgICBmdXNlZCArPSB3ICogcGxhbmVfdGVzdFtwXQ0KICAgIHN1YiA9IHBkLkRh")
parts.append("dGFGcmFtZSh7IlN0dWR5SW5zdGFuY2VVSUQiOiB0ZXN0X2RmWyJTdHVkeUluc3RhbmNlVUlEIl0sICoqew0KICAgICAgICBMQUJFTFNbal06IGZ1c2VkWzosIGpdIGZvciBqIGluIHJhbmdlKGxlbihMQUJFTFMpKX19KQ0KICAgIHN1Yl9wYXRoID0gb3MucGF0aC5q")
parts.append("b2luKG91dCwgInN1Ym1pc3Npb24uY3N2IikNCiAgICBzdWIudG9fY3N2KHN1Yl9wYXRoLCBpbmRleD1GYWxzZSkNCiAgICBwcmludChmIlxu5o+Q5Lqk5paH5Lu25bey55Sf5oiQOiB7c3ViX3BhdGh9ICBzaGFwZT17c3ViLnNoYXBlfSIpDQoNCg0KaWYgX19uYW1l")
parts.append("X18gPT0gIl9fbWFpbl9fIjoNCiAgICBtYWluKCkNCg==")
open('/kaggle/working/rsna_v3.py','wb').write(base64.b64decode(''.join(parts)))
print('wrote rsna_v3.py', os.path.getsize('/kaggle/working/rsna_v3.py'))

import base64, os
# embed exp013_multisource.py
parts=[]
parts.append("IiIiRVhQLTAxMyBtdWx0aS1zb3VyY2Ugc29mdCBsYWJlbHMuCgpTb3VyY2VzIChwcm9iYWJpbGl0eS1zcGFjZSBibGVuZCwgc2lsZW5jZSDihpIgTmFOKToKICAtIHN0ZXZlbmxlZWhhbnMgbGxtX2xhYmVsc192NF9ibGVuZC5jc3YgIChnb2xkLWtleSAwLjg5MykK")
parts.append("ICAtIGxpeGluNzMgbGFiZWxzX2xsbV9ncHQ1NnNvbC5jc3YgICAgICAgICgwLjgzNSwgZGVjb3JyZWxhdGVkKQogIC0gcGlsa3dhbmcgcmVwb3J0X2xhYmVsc192Mi5jc3YgICAgICAgICAgKDAuODcwLCBjb25mICsgVU5LIHZlcmRpY3QpCgpXZWlnaHRzOiBiYXNl")
parts.append("IDAuNSAvIDAuMiAvIDAuMywgdGhlbiBvcHRpb25hbCBwZXItY2xhc3MgcmVsaWFiaWxpdHkgb24gZ29sZDU4LgpTaWxlbmNlOiAwLjI1LzAuNSByb3VuZGVkIGJ1Y2tldHMgYW5kIHBpbGt3YW5nIFVOSyDihpIgTmFOIChuZXZlciB0cmFpbikuClRhcmdldHM6IHNo")
parts.append("cmluayAwLjUrKHAtMC41KSowLjc1LCBjbGlwIFswLjA4LCAwLjkyXS4KIiIiCmZyb20gX19mdXR1cmVfXyBpbXBvcnQgYW5ub3RhdGlvbnMKCmltcG9ydCBvcwpmcm9tIHBhdGhsaWIgaW1wb3J0IFBhdGgKCmltcG9ydCBudW1weSBhcyBucAppbXBvcnQgcGFuZGFz")
parts.append("IGFzIHBkCgpMQUJFTFMgPSBbCiAgICAiQUNMIiwgIk1DTCIsICJNZWRpYWwgTWVuaXNjdXMiLCAiTGF0ZXJhbCBNZW5pc2N1cyIsCiAgICAiTWVkaWFsIE9BIiwgIkxhdGVyYWwgT0EiLCAiUEYgT0EiLCAiRWZmdXNpb24iLAogICAgIlN5bm92aXRpcyIsICJCYWtl")
parts.append("cidzIiwgIkNvbnR1c2lvbiIsICJGcmFjdHVyZSIsCl0KClNJTEVOQ0VfUk9VTkRFRCA9IHswLjI1LCAwLjV9CkNMSVBfTE8sIENMSVBfSEkgPSAwLjA4LCAwLjkyCkJBU0VfVyA9IHsic3RldmVuIjogMC41MCwgImxpeGluIjogMC4yMCwgInBpbGsiOiAwLjMwfQoK")
parts.append("CmRlZiBfc3RhZ2Vfcm9vdCgpIC0+IFBhdGg6CiAgICByZXR1cm4gUGF0aChvcy5lbnZpcm9uLmdldCgKICAgICAgICAiRVhQMDEzX0xBQkVMX1JPT1QiLAogICAgICAgIHIiRDov5qGM6Z2iL0thZ2dsZS9yc25hX2tuZWUvX2RhdGFzZXRfc3RhZ2UiLAogICAgKSkK")
parts.append("CgpkZWYgX2ZpbmQobmFtZTogc3RyKSAtPiBQYXRoOgogICAgZW52ID0gb3MuZW52aXJvbi5nZXQoZiJFWFAwMTNfe25hbWUudXBwZXIoKX0iKQogICAgaWYgZW52IGFuZCBQYXRoKGVudikuaXNfZmlsZSgpOgogICAgICAgIHJldHVybiBQYXRoKGVudikKICAgIGZv")
parts.append("ciByb290IGluICgKICAgICAgICBQYXRoKCIva2FnZ2xlL2lucHV0IiksCiAgICAgICAgX3N0YWdlX3Jvb3QoKSwKICAgICk6CiAgICAgICAgaWYgbm90IHJvb3QuaXNfZGlyKCk6CiAgICAgICAgICAgIGNvbnRpbnVlCiAgICAgICAgZm9yIHAgaW4gcm9vdC5yZ2xv")
parts.append("YihuYW1lKToKICAgICAgICAgICAgcmV0dXJuIHAKICAgIHJhaXNlIEZpbGVOb3RGb3VuZEVycm9yKG5hbWUpCgoKZGVmIF9jbGVhbihhcnI6IG5wLm5kYXJyYXkpIC0+IG5wLm5kYXJyYXk6CiAgICBvdXQgPSBhcnIuYXN0eXBlKG5wLmZsb2F0NjQpLmNvcHkoKQog")
parts.append("ICAgb3V0ID0gbnAud2hlcmUobnAuaXNpbihucC5yb3VuZChvdXQsIDIpLCBsaXN0KFNJTEVOQ0VfUk9VTkRFRCkpLCBucC5uYW4sIG91dCkKICAgIG91dCA9IG5wLndoZXJlKG5wLmlzZmluaXRlKG91dCksIDAuNSArIChvdXQgLSAwLjUpICogMC43NSwgb3V0KQog")
parts.append("ICAgb3V0ID0gbnAuY2xpcChvdXQsIENMSVBfTE8sIENMSVBfSEkpCiAgICByZXR1cm4gb3V0CgoKZGVmIF9sb2FkX3NvdXJjZXMoKToKICAgIHN0ZXZlbiA9IHBkLnJlYWRfY3N2KF9maW5kKCJsbG1fbGFiZWxzX3Y0X2JsZW5kLmNzdiIpKS5zZXRfaW5kZXgoIlN0")
parts.append("dWR5SW5zdGFuY2VVSUQiKQogICAgbGl4aW4gPSBwZC5yZWFkX2NzdihfZmluZCgibGFiZWxzX2xsbV9ncHQ1NnNvbC5jc3YiKSkuc2V0X2luZGV4KCJTdHVkeUluc3RhbmNlVUlEIikKICAgIHBpbGsgPSBwZC5yZWFkX2NzdihfZmluZCgicmVwb3J0X2xhYmVsc192")
parts.append("Mi5jc3YiKSkuc2V0X2luZGV4KCJTdHVkeUluc3RhbmNlVUlEIikKICAgIHJldHVybiBzdGV2ZW4sIGxpeGluLCBwaWxrCgoKZGVmIF9waWxrX3NvZnQocGlsazogcGQuRGF0YUZyYW1lLCBjOiBzdHIpIC0+IHBkLlNlcmllczoKICAgICIiIk1hcCBwaWxrd2FuZyB2")
parts.append("ZXJkaWN0K2NvbmYgdG8gc29mdCBwcm9iczsgVU5LIOKGkiBOYU4uIiIiCiAgICBpZiBmIntjfV9fdmVyZGljdCIgbm90IGluIHBpbGsuY29sdW1ucyBvciBjIG5vdCBpbiBwaWxrLmNvbHVtbnM6CiAgICAgICAgcmV0dXJuIHBkLlNlcmllcyhucC5uYW4sIGluZGV4")
parts.append("PXBpbGsuaW5kZXgpCiAgICB2ID0gcGlsa1tmIntjfV9fdmVyZGljdCJdLmFzdHlwZShzdHIpLnN0ci51cHBlcigpCiAgICByYXcgPSBwaWxrW2NdLmFzdHlwZShmbG9hdCkKICAgIG91dCA9IHJhdy5jb3B5KCkKICAgIG91dFt2ID09ICJVTksiXSA9IG5wLm5hbgog")
parts.append("ICAgcmV0dXJuIG91dAoKCmRlZiBidWlsZF9leHAwMTNfbGFiZWxzKHRyYWluX2Nzdjogc3RyIHwgUGF0aCwgdmVyYm9zZTogYm9vbCA9IFRydWUpIC0+IHBkLkRhdGFGcmFtZToKICAgIHRyYWluID0gcGQucmVhZF9jc3YodHJhaW5fY3N2KQogICAgZGYgPSB0cmFp")
parts.append("bltbIlN0dWR5SW5zdGFuY2VVSUQiXSArIExBQkVMU10uY29weSgpCiAgICBzdGV2ZW4sIGxpeGluLCBwaWxrID0gX2xvYWRfc291cmNlcygpCiAgICBpZiB2ZXJib3NlOgogICAgICAgIHByaW50KCJbRVhQLTAxM10gc291cmNlczoiLAogICAgICAgICAgICAgIGYi")
parts.append("c3RldmVuPXtsZW4oc3RldmVuKX0gbGl4aW49e2xlbihsaXhpbil9IHBpbGs9e2xlbihwaWxrKX0iKQogICAgICAgIHByaW50KGYiW0VYUC0wMTNdIHdlaWdodHMge0JBU0VfV30iKQoKICAgIG5fY2VsbHMgPSAwCiAgICBmb3IgYyBpbiBMQUJFTFM6CiAgICAgICAg")
parts.append("bWFzayA9IGRmW2NdLmlzbmEoKS52YWx1ZXMKICAgICAgICB1aWRzID0gZGYubG9jW21hc2ssICJTdHVkeUluc3RhbmNlVUlEIl0KCiAgICAgICAgcGFydHMgPSBbXQogICAgICAgIHdzID0gW10KICAgICAgICBmb3Igc3JjLCBrZXksIHcgaW4gKAogICAgICAgICAg")
parts.append("ICAoc3RldmVuLCBjLCBCQVNFX1dbInN0ZXZlbiJdKSwKICAgICAgICAgICAgKGxpeGluLCBjLCBCQVNFX1dbImxpeGluIl0pLAogICAgICAgICAgICAocGlsaywgTm9uZSwgQkFTRV9XWyJwaWxrIl0pLAogICAgICAgICk6CiAgICAgICAgICAgIGlmIGtleSBpcyBu")
parts.append("b3QgTm9uZToKICAgICAgICAgICAgICAgIGlmIGtleSBub3QgaW4gc3JjLmNvbHVtbnM6CiAgICAgICAgICAgICAgICAgICAgY29udGludWUKICAgICAgICAgICAgICAgIHJhdyA9IHNyY1trZXldLnJlaW5kZXgodWlkcykudG9fbnVtcHkoZHR5cGU9bnAuZmxvYXQ2")
parts.append("NCkKICAgICAgICAgICAgICAgIGNsID0gX2NsZWFuKHJhdykKICAgICAgICAgICAgZWxzZToKICAgICAgICAgICAgICAgIGNsID0gX3BpbGtfc29mdChzcmMsIGMpLnJlaW5kZXgodWlkcykudG9fbnVtcHkoZHR5cGU9bnAuZmxvYXQ2NCkKICAgICAgICAgICAgICAg")
parts.append("IGNsID0gX2NsZWFuKGNsKQogICAgICAgICAgICBwYXJ0cy5hcHBlbmQoY2wpCiAgICAgICAgICAgIHdzLmFwcGVuZCh3KQoKICAgICAgICBzdGFjayA9IG5wLnZzdGFjayhwYXJ0cykgICMgKFMsIE4pCiAgICAgICAgd3ZlYyA9IG5wLmFzYXJyYXkod3MsIGR0eXBl")
parts.append("PW5wLmZsb2F0NjQpWzosIE5vbmVdCiAgICAgICAgdmFsaWQgPSBucC5pc2Zpbml0ZShzdGFjaykKICAgICAgICB3c3VtID0gbnAud2hlcmUodmFsaWQsIHd2ZWMsIDAuMCkuc3VtKGF4aXM9MCkKICAgICAgICBwc3VtID0gbnAud2hlcmUodmFsaWQsIHN0YWNrICog")
parts.append("d3ZlYywgMC4wKS5zdW0oYXhpcz0wKQogICAgICAgIGJsZW5kID0gbnAud2hlcmUod3N1bSA+IDAsIHBzdW0gLyBucC5tYXhpbXVtKHdzdW0sIDFlLTkpLCBucC5uYW4pCgogICAgICAgIGRmLmxvY1ttYXNrLCBjXSA9IGJsZW5kCiAgICAgICAgbl91c2UgPSBpbnQo")
parts.append("bnAuaXNmaW5pdGUoYmxlbmQpLnN1bSgpKQogICAgICAgIG5fY2VsbHMgKz0gbl91c2UKICAgICAgICBpZiB2ZXJib3NlOgogICAgICAgICAgICBhZ3JlZSA9IHZhbGlkLmFsbChheGlzPTApLnN1bSgpIGlmIHZhbGlkLnNpemUgZWxzZSAwCiAgICAgICAgICAgIHBy")
parts.append("aW50KAogICAgICAgICAgICAgICAgZiIgIHtjOjE4c30gZ29sZCB7aW50KHRyYWluW2NdLm5vdG5hKCkuc3VtKCkpOjRkfSIKICAgICAgICAgICAgICAgIGYiICsgbXVsdGkge25fdXNlOjRkfSAoYWxsMz17aW50KGFncmVlKX0pID0ge2ludChkZltjXS5ub3RuYSgp")
parts.append("LnN1bSgpKTo1ZH0iCiAgICAgICAgICAgICkKICAgIGlmIHZlcmJvc2U6CiAgICAgICAgcHJpbnQoZiJbRVhQLTAxM10gbXVsdGktc291cmNlIGNlbGxzOiB7bl9jZWxsc30iKQogICAgICAgICMgc2hvdyBsb3NzLXdlaWdodCBtYXNzCiAgICAgICAgeSA9IGRmW0xB")
parts.append("QkVMU10udG9fbnVtcHkoZHR5cGU9bnAuZmxvYXQ2NCkKICAgICAgICB3ID0gMi4wICogbnAuYWJzKHkgLSAwLjUpCiAgICAgICAgcHJpbnQoZiJbRVhQLTAxM10gbG9zcy13IG1lYW49e25wLm5hbm1lYW4odyk6LjNmfSAiCiAgICAgICAgICAgICAgZiJwMTA9e25w")
parts.append("Lm5hbnF1YW50aWxlKHcsMC4xKTouM2Z9IHA5MD17bnAubmFucXVhbnRpbGUodywwLjkpOi4zZn0iKQogICAgcmV0dXJuIGRmCgoKaWYgX19uYW1lX18gPT0gIl9fbWFpbl9fIjoKICAgIHJvb3QgPSBQYXRoKG9zLmVudmlyb24uZ2V0KCJSU05BX1JPT1QiLCByIkQ6")
parts.append("L+ahjOmdoi9LYWdnbGUvcnNuYV9rbmVlIikpCiAgICBvdXQgPSBidWlsZF9leHAwMTNfbGFiZWxzKHJvb3QgLyAidHJhaW4uY3N2IikKICAgIGRzdCA9IHJvb3QgLyAibGFiZWxzIiAvICJ0cmFpbl9leHAwMTNfbXVsdGlzb3VyY2UuY3N2IgogICAgZHN0LnBhcmVu")
parts.append("dC5ta2RpcihwYXJlbnRzPVRydWUsIGV4aXN0X29rPVRydWUpCiAgICBvdXQudG9fY3N2KGRzdCwgaW5kZXg9RmFsc2UpCiAgICBwcmludCgid3JvdGUiLCBkc3QsIG91dC5zaGFwZSkK")
open('/kaggle/working/exp013_multisource.py','wb').write(base64.b64decode(''.join(parts)))
print('wrote exp013_multisource.py', os.path.getsize('/kaggle/working/exp013_multisource.py'))

import base64, os
# embed sampling_geometry.py
parts=[]
parts.append("IiIiRVhQLTAxNiBwaHlzaWNhbCBnZW9tZXRyeSAvIHNhbXBsaW5nLiBJc29sYXRlZCBmcm9tIHRyYWluaW5nIGxvZ2ljLgoKTGVnYWN5IHBhdGggcmVtYWlucyBieXRlLWNvbXBhdGlibGUgaW4gc2VsZWN0aW9uIG9yZGVyIChVSUQgb3JkZXIgcHJlc2VydmVkKS4K")
parts.append("UGh5c2ljYWwgcGF0aCBvbmx5IGNoYW5nZXM6CiAgMSkgc2xpY2Ugc29ydCBrZXk6IGRvdChJUFAsIHNsaWNlX25vcm1hbCkgd2l0aCBsb2dnZWQgZmFsbGJhY2tzCiAgMikgMi41RCB0cmlwbGV0czogdHJ1ZSBwaHlzaWNhbCBuZWlnaGJvcnMgb24gdGhlIHNvcnRl")
parts.append("ZCBzdGFjawogIDMpIGNsaXAgY2VudGVyczogcGh5c2ljYWwtc3BhbiBiYXNlZCAoc3RpbGwgY2FwcGVkIGJ5IG5fY2xpcHMpCgpObyBsYXRlcmFsaXR5LCBubyBNSUwsIG5vIGxhYmVscywgbm8gbG9zcywgbm8gYmFja2JvbmUgY2hhbmdlcy4KIiIiCmZyb20gX19m")
parts.append("dXR1cmVfXyBpbXBvcnQgYW5ub3RhdGlvbnMKCmltcG9ydCBvcwpmcm9tIGNvbGxlY3Rpb25zIGltcG9ydCBDb3VudGVyCmZyb20gZGF0YWNsYXNzZXMgaW1wb3J0IGRhdGFjbGFzcywgZmllbGQKCmltcG9ydCBudW1weSBhcyBucAoKIyBtb2R1bGUtbGV2ZWwgY291")
parts.append("bnRlcnMgZm9yIHNtb2tlIHRlc3RzIC8gdHJhaW5pbmcgbG9ncwpTVEFUUyA9IHsKICAgICJzb3J0X2lwcF96X2ZhbGxiYWNrIjogMCwKICAgICJzb3J0X25vX2lvcCI6IDAsCiAgICAic29ydF9ub19pcHAiOiAwLAogICAgInNvcnRfcGh5c2ljYWxfb2siOiAwLAog")
parts.append("ICAgInRyaXBsZXRfbWlzc2luZ19uZWlnaGJvciI6IDAsCiAgICAidHJpcGxldF9tYWRlIjogMCwKICAgICJ0cmlwbGV0X2dhcF9yZWxheGVkIjogMCwKICAgICJ0cmlwbGV0X3NraXBwZWQiOiAwLAp9CiMgYWNjdW11bGF0ZWQgZ2VvbWV0cnkgKGZvciBFWFAtMDE2")
parts.append("IEdQVSBydW4gZHVtcCkKR0FQUzogbGlzdFtmbG9hdF0gPSBbXQpTUEFOUzogbGlzdFtmbG9hdF0gPSBbXQpDTElQU19QRVJfU0VSSUVTOiBsaXN0W2ludF0gPSBbXQpVU0VEX1NMSUNFU19QRVJfU0VSSUVTOiBsaXN0W2ludF0gPSBbXQoKCmRlZiByZXNldF9zdGF0")
parts.append("cygpIC0+IE5vbmU6CiAgICBmb3IgayBpbiBTVEFUUzoKICAgICAgICBTVEFUU1trXSA9IDAKICAgIEdBUFMuY2xlYXIoKQogICAgU1BBTlMuY2xlYXIoKQogICAgQ0xJUFNfUEVSX1NFUklFUy5jbGVhcigpCiAgICBVU0VEX1NMSUNFU19QRVJfU0VSSUVTLmNsZWFy")
parts.append("KCkKCgpkZWYgZHVtcF9zdGF0cygpIC0+IGRpY3Q6CiAgICBpbXBvcnQgbnVtcHkgYXMgbnAKCiAgICBnID0gbnAuYXNhcnJheShHQVBTLCBkdHlwZT1ucC5mbG9hdDY0KQogICAgc3AgPSBucC5hc2FycmF5KFNQQU5TLCBkdHlwZT1ucC5mbG9hdDY0KQogICAgY3Mg")
parts.append("PSBucC5hc2FycmF5KENMSVBTX1BFUl9TRVJJRVMsIGR0eXBlPW5wLmZsb2F0NjQpCiAgICBvdXQgPSBkaWN0KFNUQVRTKQogICAgb3V0WyJmYWxsYmFja19yYXRlIl0gPSAoCiAgICAgICAgKFNUQVRTWyJzb3J0X2lwcF96X2ZhbGxiYWNrIl0gKyBTVEFUU1sic29y")
parts.append("dF9ub19pcHAiXSkKICAgICAgICAvIG1heCgKICAgICAgICAgICAgMSwKICAgICAgICAgICAgU1RBVFNbInNvcnRfcGh5c2ljYWxfb2siXQogICAgICAgICAgICArIFNUQVRTWyJzb3J0X2lwcF96X2ZhbGxiYWNrIl0KICAgICAgICAgICAgKyBTVEFUU1sic29ydF9u")
parts.append("b19pcHAiXSwKICAgICAgICApCiAgICApCiAgICBvdXRbImNsaXBzX3Blcl9zZXJpZXMiXSA9IGZsb2F0KG5wLm1lZGlhbihjcykpIGlmIGNzLnNpemUgZWxzZSBOb25lCiAgICBvdXRbImNsaXBzX3RvdGFsIl0gPSBpbnQoY3Muc3VtKCkpIGlmIGNzLnNpemUgZWxz")
parts.append("ZSAwCiAgICBvdXRbInVzZWRfc2xpY2VzX3Blcl9zZXJpZXMiXSA9ICgKICAgICAgICBmbG9hdChucC5tZWRpYW4obnAuYXNhcnJheShVU0VEX1NMSUNFU19QRVJfU0VSSUVTLCBkdHlwZT1ucC5mbG9hdDY0KSkpCiAgICAgICAgaWYgVVNFRF9TTElDRVNfUEVSX1NF")
parts.append("UklFUwogICAgICAgIGVsc2UgTm9uZQogICAgKQogICAgb3V0WyJhZGphY2VudF9nYXBfbWVkaWFuIl0gPSBmbG9hdChucC5tZWRpYW4oZykpIGlmIGcuc2l6ZSBlbHNlIE5vbmUKICAgIG91dFsiYWRqYWNlbnRfZ2FwX1A5MCJdID0gZmxvYXQobnAucGVyY2VudGls")
parts.append("ZShnLCA5MCkpIGlmIGcuc2l6ZSBlbHNlIE5vbmUKICAgIG91dFsiYWRqYWNlbnRfZ2FwX1A5NSJdID0gZmxvYXQobnAucGVyY2VudGlsZShnLCA5NSkpIGlmIGcuc2l6ZSBlbHNlIE5vbmUKICAgIG91dFsicGh5c2ljYWxfc3Bhbl9tZWRpYW4iXSA9IGZsb2F0KG5w")
parts.append("Lm1lZGlhbihzcCkpIGlmIHNwLnNpemUgZWxzZSBOb25lCiAgICBvdXRbInBoeXNpY2FsX3NwYW5fUDkwIl0gPSBmbG9hdChucC5wZXJjZW50aWxlKHNwLCA5MCkpIGlmIHNwLnNpemUgZWxzZSBOb25lCiAgICByZXR1cm4gb3V0CgoKZGVmIHNsaWNlX25vcm1hbF9m")
parts.append("cm9tX2lvcChpb3ApIC0+IG5wLm5kYXJyYXkgfCBOb25lOgogICAgIiIiSU9QOiA2IGZsb2F0cyBbcm93X3gscm93X3kscm93X3osIGNvbF94LGNvbF95LGNvbF96XSAtPiB1bml0IG5vcm1hbC4iIiIKICAgIGlmIGlvcCBpcyBOb25lOgogICAgICAgIHJldHVybiBO")
parts.append("b25lCiAgICB0cnk6CiAgICAgICAgdiA9IG5wLmFzYXJyYXkoaW9wLCBkdHlwZT1ucC5mbG9hdDY0KS5yZXNoYXBlKC0xKQogICAgICAgIGlmIHYuc2l6ZSA8IDY6CiAgICAgICAgICAgIHJldHVybiBOb25lCiAgICAgICAgcm93LCBjb2wgPSB2WzozXSwgdlszOjZd")
parts.append("CiAgICAgICAgbiA9IG5wLmNyb3NzKHJvdywgY29sKQogICAgICAgIG5uID0gZmxvYXQobnAubGluYWxnLm5vcm0obikpCiAgICAgICAgaWYgbm90IG5wLmlzZmluaXRlKG5uKSBvciBubiA8IDFlLTg6CiAgICAgICAgICAgIHJldHVybiBOb25lCiAgICAgICAgcmV0")
parts.append("dXJuIG4gLyBubgogICAgZXhjZXB0IEV4Y2VwdGlvbjoKICAgICAgICByZXR1cm4gTm9uZQoKCmRlZiBwaHlzaWNhbF9wb3NpdGlvbihpcHAsIG5vcm1hbDogbnAubmRhcnJheSB8IE5vbmUpIC0+IHR1cGxlW2Zsb2F0IHwgTm9uZSwgc3RyXToKICAgICIiIlJldHVy")
parts.append("biAocG9zLCBraW5kKSBraW5kIGluIHtwaHlzaWNhbCwgaXBwX3osIG1pc3Npbmd9LiIiIgogICAgaWYgaXBwIGlzIE5vbmU6CiAgICAgICAgcmV0dXJuIE5vbmUsICJtaXNzaW5nIgogICAgdHJ5OgogICAgICAgIHAgPSBucC5hc2FycmF5KGlwcCwgZHR5cGU9bnAu")
parts.append("ZmxvYXQ2NCkucmVzaGFwZSgtMSkKICAgICAgICBpZiBwLnNpemUgPCAzOgogICAgICAgICAgICByZXR1cm4gTm9uZSwgIm1pc3NpbmciCiAgICAgICAgaWYgbm9ybWFsIGlzIG5vdCBOb25lOgogICAgICAgICAgICByZXR1cm4gZmxvYXQobnAuZG90KHBbOjNdLCBu")
parts.append("b3JtYWwpKSwgInBoeXNpY2FsIgogICAgICAgIHJldHVybiBmbG9hdChwWzJdKSwgImlwcF96IgogICAgZXhjZXB0IEV4Y2VwdGlvbjoKICAgICAgICByZXR1cm4gTm9uZSwgIm1pc3NpbmciCgoKZGVmIHNvcnRfc2xpY2VzX3BoeXNpY2FsKGZpbGVzLCBkY21fcmVh")
parts.append("ZGVyKToKICAgICIiIlNvcnQgRElDT00gZmlsZXMgYnkgcGh5c2ljYWwgc3RhY2sgcG9zaXRpb24uCgogICAgZGNtX3JlYWRlcihwYXRoKSAtPiBweWRpY29tIGRhdGFzZXQgb3IgTm9uZSAoaGVhZGVyLW9ubHkgb2spLgogICAgUmV0dXJucyAocm93cywgbWV0YSkg")
parts.append("d2hlcmUgcm93cyA9IFsocGF0aCwgcG9zLCBraW5kKV0sIHNvcnRlZCBhc2NlbmRpbmcuCiAgICBGYWxsYmFja3MgYXJlIGNvdW50ZWQgaW4gU1RBVFMgYW5kIHJlcG9ydGVkIGluIG1ldGEuCiAgICAiIiIKICAgIHJvd3MgPSBbXQogICAgbm9ybWFscyA9IFtdCiAg")
parts.append("ICBmb3IgZiBpbiBmaWxlczoKICAgICAgICBkcyA9IE5vbmUKICAgICAgICB0cnk6CiAgICAgICAgICAgIGRzID0gZGNtX3JlYWRlcihmKQogICAgICAgIGV4Y2VwdCBFeGNlcHRpb246CiAgICAgICAgICAgIGRzID0gTm9uZQogICAgICAgIGlvcCA9IGRzLmdldCgi")
parts.append("SW1hZ2VPcmllbnRhdGlvblBhdGllbnQiKSBpZiBkcyBpcyBub3QgTm9uZSBlbHNlIE5vbmUKICAgICAgICBpcHAgPSBkcy5nZXQoIkltYWdlUG9zaXRpb25QYXRpZW50IikgaWYgZHMgaXMgbm90IE5vbmUgZWxzZSBOb25lCiAgICAgICAgbiA9IHNsaWNlX25vcm1h")
parts.append("bF9mcm9tX2lvcChpb3ApCiAgICAgICAgaWYgbiBpcyBOb25lOgogICAgICAgICAgICBTVEFUU1sic29ydF9ub19pb3AiXSArPSAxCiAgICAgICAgZWxzZToKICAgICAgICAgICAgbm9ybWFscy5hcHBlbmQobikKICAgICAgICBwb3MsIGtpbmQgPSBwaHlzaWNhbF9w")
parts.append("b3NpdGlvbihpcHAsIG4pCiAgICAgICAgaWYga2luZCA9PSAicGh5c2ljYWwiOgogICAgICAgICAgICBTVEFUU1sic29ydF9waHlzaWNhbF9vayJdICs9IDEKICAgICAgICBlbGlmIGtpbmQgPT0gImlwcF96IjoKICAgICAgICAgICAgU1RBVFNbInNvcnRfaXBwX3pf")
parts.append("ZmFsbGJhY2siXSArPSAxCiAgICAgICAgZWxzZToKICAgICAgICAgICAgU1RBVFNbInNvcnRfbm9faXBwIl0gKz0gMQogICAgICAgIHJvd3MuYXBwZW5kKChmLCBwb3MsIGtpbmQpKQoKICAgICMgaWYgbm9ybWFscyBkaXNhZ3JlZSBhIGxvdCwgc3RpbGwgdXNlIHBl")
parts.append("ci1zbGljZSBub3JtYWwgKGxvY2FsKSDigJQgbG9nIGNvdW50CiAgICBtZXRhID0gewogICAgICAgICJuIjogbGVuKGZpbGVzKSwKICAgICAgICAibl9waHlzaWNhbCI6IHN1bSgxIGZvciByIGluIHJvd3MgaWYgclsyXSA9PSAicGh5c2ljYWwiKSwKICAgICAgICAi")
parts.append("bl9pcHBfeiI6IHN1bSgxIGZvciByIGluIHJvd3MgaWYgclsyXSA9PSAiaXBwX3oiKSwKICAgICAgICAibl9taXNzaW5nIjogc3VtKDEgZm9yIHIgaW4gcm93cyBpZiByWzJdID09ICJtaXNzaW5nIiksCiAgICB9CgogICAgZGVmIGtleShyKToKICAgICAgICAjIE5v")
parts.append("bmUgbGFzdCwga2VlcCBzdGFibGUgYW1vbmcgZXF1YWxzCiAgICAgICAgcG9zID0gclsxXQogICAgICAgIHJldHVybiAocG9zIGlzIE5vbmUsIHBvcyBpZiBwb3MgaXMgbm90IE5vbmUgZWxzZSAwLjAsIHJbMF0pCgogICAgcm93cy5zb3J0KGtleT1rZXkpCiAgICBy")
parts.append("ZXR1cm4gcm93cywgbWV0YQoKCmRlZiBtZWRpYW5fc3BhY2luZyhwb3NpdGlvbnMpIC0+IGZsb2F0OgogICAgIiIiTWVkaWFuIHzOlHBvc3wgb2YgY29uc2VjdXRpdmUgZmluaXRlIHBvc2l0aW9ucyAoc3RhY2sgc3BhY2luZykuIiIiCiAgICBwID0gbnAuYXNhcnJh")
parts.append("eSgKICAgICAgICBbeCBmb3IgeCBpbiBwb3NpdGlvbnMgaWYgeCBpcyBub3QgTm9uZSBhbmQgbnAuaXNmaW5pdGUoeCldLCBkdHlwZT1ucC5mbG9hdDY0CiAgICApCiAgICBpZiBwLnNpemUgPCAyOgogICAgICAgIHJldHVybiBmbG9hdCgibmFuIikKICAgIGQgPSBu")
parts.append("cC5hYnMobnAuZGlmZihwKSkKICAgIGQgPSBkW2QgPiAxZS04XQogICAgaWYgZC5zaXplID09IDA6CiAgICAgICAgcmV0dXJuIGZsb2F0KCJuYW4iKQogICAgcmV0dXJuIGZsb2F0KG5wLm1lZGlhbihkKSkKCgpkZWYgc2VsZWN0X3RyaXBsZXRfY2VudGVycyhwb3Np")
parts.append("dGlvbnMsIG5fY2xpcHM6IGludCk6CiAgICAiIiJDaG9vc2UgY2VudGVyIGluZGljZXMgZm9yIDIuNUQgY2xpcHMgdXNpbmcgcGh5c2ljYWwgc3Bhbi4KCiAgICBTdHJhdGVneToKICAgICAgLSByZXF1aXJlIGZpbml0ZSBwb3NpdGlvbnMgZm9yIHNwYW47IGZhbGxi")
parts.append("YWNrIHRvIGV2ZW4gaW5kZXggY2VudGVycwogICAgICAtIGNlbnRlcnMgZXZlbmx5IHNwYWNlZCBpbiAqcGh5c2ljYWwgcG9zaXRpb24qIHdpdGhpbiBbbG8sIGhpXQogICAgICAtIG1hcCBlYWNoIGRlc2lyZWQgcG9zaXRpb24gdG8gbmVhcmVzdCB2YWxpZCBpbnRl")
parts.append("cmlvciBpbmRleAogICAgICAtIGVuZm9yY2UgY2VudGVyIGluIFsxLCBuLTJdIHNvIG5laWdoYm9ycyBleGlzdAogICAgUmV0dXJucyBsaXN0IG9mIGNlbnRlciBpbmRpY2VzIChsZW5ndGggPD0gbl9jbGlwcyksIHVuaXF1ZSwgYXNjZW5kaW5nLgogICAgIiIiCiAg")
parts.append("ICBuID0gbGVuKHBvc2l0aW9ucykKICAgIGlmIG4gPCAzIG9yIG5fY2xpcHMgPD0gMDoKICAgICAgICByZXR1cm4gW10KICAgIHBvcyA9IG5wLmFzYXJyYXkoCiAgICAgICAgW25wLm5hbiBpZiBwIGlzIE5vbmUgZWxzZSBmbG9hdChwKSBmb3IgcCBpbiBwb3NpdGlv")
parts.append("bnNdLCBkdHlwZT1ucC5mbG9hdDY0CiAgICApCiAgICBmaW5pdGUgPSBucC5pc2Zpbml0ZShwb3MpCiAgICBpZiBmaW5pdGUuc3VtKCkgPCAzOgogICAgICAgICMgZmFsbGJhY2s6IGV2ZW4gaW5kZXggY2VudGVycyBpbiBzb3J0ZWQgb3JkZXIKICAgICAgICBzdGVw")
parts.append("ID0gbWF4KDEsIChuIC0gMikgLy8gbl9jbGlwcykKICAgICAgICByZXR1cm4gbGlzdChyYW5nZSgxLCBuIC0gMSwgc3RlcCkpWzpuX2NsaXBzXQoKICAgIGxvID0gZmxvYXQobnAubmFubWluKHBvcykpCiAgICBoaSA9IGZsb2F0KG5wLm5hbm1heChwb3MpKQogICAg")
parts.append("aWYgbm90IG5wLmlzZmluaXRlKGxvKSBvciBub3QgbnAuaXNmaW5pdGUoaGkpIG9yIGhpIC0gbG8gPCAxZS04OgogICAgICAgIHN0ZXAgPSBtYXgoMSwgKG4gLSAyKSAvLyBuX2NsaXBzKQogICAgICAgIHJldHVybiBsaXN0KHJhbmdlKDEsIG4gLSAxLCBzdGVwKSlb")
parts.append("Om5fY2xpcHNdCgogICAgIyBwaHlzaWNhbCB0YXJnZXRzCiAgICBpZiBuX2NsaXBzID09IDE6CiAgICAgICAgdGFyZ2V0cyA9IFswLjUgKiAobG8gKyBoaSldCiAgICBlbHNlOgogICAgICAgIHRhcmdldHMgPSBbbG8gKyAoaGkgLSBsbykgKiAoaSArIDAuNSkgLyBu")
parts.append("X2NsaXBzIGZvciBpIGluIHJhbmdlKG5fY2xpcHMpXQoKICAgIHVzZWQgPSBzZXQoKQogICAgY2VudGVycyA9IFtdCiAgICBmb3IgdCBpbiB0YXJnZXRzOgogICAgICAgICMgbmVhcmVzdCBpbmRleCBhbW9uZyBpbnRlcmlvciBzbGljZXMgd2l0aCBmaW5pdGUgcG9z")
parts.append("CiAgICAgICAgYmVzdF9pLCBiZXN0X2QgPSBOb25lLCBOb25lCiAgICAgICAgZm9yIGkgaW4gcmFuZ2UoMSwgbiAtIDEpOgogICAgICAgICAgICBpZiBub3QgZmluaXRlW2ldOgogICAgICAgICAgICAgICAgY29udGludWUKICAgICAgICAgICAgZCA9IGFicyhwb3Nb")
parts.append("aV0gLSB0KQogICAgICAgICAgICBpZiBiZXN0X2QgaXMgTm9uZSBvciBkIDwgYmVzdF9kOgogICAgICAgICAgICAgICAgYmVzdF9pLCBiZXN0X2QgPSBpLCBkCiAgICAgICAgaWYgYmVzdF9pIGlzIE5vbmU6CiAgICAgICAgICAgIGNvbnRpbnVlCiAgICAgICAgaWYg")
parts.append("YmVzdF9pIGluIHVzZWQ6CiAgICAgICAgICAgICMgdHJ5IG5laWdoYm9ycwogICAgICAgICAgICBmb3IgZGVsdGEgaW4gKDEsIC0xLCAyLCAtMik6CiAgICAgICAgICAgICAgICBqID0gYmVzdF9pICsgZGVsdGEKICAgICAgICAgICAgICAgIGlmIDEgPD0gaiA8PSBu")
parts.append("IC0gMiBhbmQgaiBub3QgaW4gdXNlZCBhbmQgZmluaXRlW2pdOgogICAgICAgICAgICAgICAgICAgIGJlc3RfaSA9IGoKICAgICAgICAgICAgICAgICAgICBicmVhawogICAgICAgICAgICBlbHNlOgogICAgICAgICAgICAgICAgY29udGludWUKICAgICAgICB1c2Vk")
parts.append("LmFkZChiZXN0X2kpCiAgICAgICAgY2VudGVycy5hcHBlbmQoYmVzdF9pKQogICAgcmV0dXJuIHNvcnRlZChjZW50ZXJzKQoKCmRlZiBtYWtlX2NsaXBzX3BoeXNpY2FsKAogICAgc2VyaWVzX3Jvd3MsCiAgICBuX2NsaXBzPTIsCiAgICBpbWdfc2l6ZT0yMjQsCiAg")
parts.append("ICB3aW5kb3dfZm49Tm9uZSwKICAgIGRjbV9yZWFkX2ZuPU5vbmUsCiAgICBtYXhfZ2FwX2ZhY3Rvcj0zLjAsCiAgICBhYnNfZ2FwX2NhcD1Ob25lLAopOgogICAgIiIiQnVpbGQgMi41RCBjbGlwcyB3aXRoIHRydWUgcGh5c2ljYWwgbmVpZ2hib3JzLgoKICAgIHNl")
parts.append("cmllc19yb3dzOiBsaXN0IG9mIChwYXRoLCBwb3MsIGtpbmQpIGZyb20gc29ydF9zbGljZXNfcGh5c2ljYWwKICAgICAgICAgICAgICAgICBvciBsZWdhY3kgKHBhdGgsIHopIHR1cGxlcyDigJQgd2UgYWNjZXB0IGJvdGguCiAgICBSZXR1cm5zIChjbGlwcywgY2xp")
parts.append("cF9tZXRhX2xpc3QpLgogICAgIiIiCiAgICBhc3NlcnQgd2luZG93X2ZuIGlzIG5vdCBOb25lIGFuZCBkY21fcmVhZF9mbiBpcyBub3QgTm9uZQogICAgIyBub3JtYWxpemUgcm93cyB0byAocGF0aCwgcG9zKQogICAgbm9ybSA9IFtdCiAgICBmb3IgciBpbiBzZXJp")
parts.append("ZXNfcm93czoKICAgICAgICBpZiBsZW4ocikgPT0gMzoKICAgICAgICAgICAgbm9ybS5hcHBlbmQoKHJbMF0sIHJbMV0pKQogICAgICAgIGVsc2U6CiAgICAgICAgICAgIG5vcm0uYXBwZW5kKChyWzBdLCByWzFdKSkKICAgIG4gPSBsZW4obm9ybSkKICAgIGlmIG4g")
parts.append("PCAzOgogICAgICAgIHJldHVybiBbXSwgW10KICAgIHBhdGhzID0gW3JbMF0gZm9yIHIgaW4gbm9ybV0KICAgIHBvcyA9IFtyWzFdIGZvciByIGluIG5vcm1dCiAgICBzcGFjaW5nID0gbWVkaWFuX3NwYWNpbmcocG9zKQogICAgY2VudGVycyA9IHNlbGVjdF90cmlw")
parts.append("bGV0X2NlbnRlcnMocG9zLCBuX2NsaXBzKQogICAgY2xpcHMsIG1ldGFzID0gW10sIFtdCiAgICBmb3IgYyBpbiBjZW50ZXJzOgogICAgICAgIGlkeF90cmlwID0gKGMgLSAxLCBjLCBjICsgMSkKICAgICAgICBpZiBpZHhfdHJpcFswXSA8IDAgb3IgaWR4X3RyaXBb")
parts.append("Ml0gPj0gbjoKICAgICAgICAgICAgU1RBVFNbInRyaXBsZXRfc2tpcHBlZCJdICs9IDEKICAgICAgICAgICAgY29udGludWUKICAgICAgICBwcyA9IFtwb3NbaV0gZm9yIGkgaW4gaWR4X3RyaXBdCiAgICAgICAgaWYgYW55KHAgaXMgTm9uZSBvciBub3QgbnAuaXNm")
parts.append("aW5pdGUocCkgZm9yIHAgaW4gcHMpOgogICAgICAgICAgICBTVEFUU1sidHJpcGxldF9taXNzaW5nX25laWdoYm9yIl0gKz0gMQogICAgICAgICAgICAjIHN0aWxsIGFsbG93IGlmIGluZGljZXMgYXJlIGNvbnNlY3V0aXZlIGluIHNvcnRlZCBvcmRlcgogICAgICAg")
parts.append("ICAgICBnYXBzID0gKE5vbmUsIE5vbmUpCiAgICAgICAgICAgIHJlbGF4ZWQgPSBUcnVlCiAgICAgICAgZWxzZToKICAgICAgICAgICAgZ2FwcyA9ICgKICAgICAgICAgICAgICAgIGFicyhwc1sxXSAtIHBzWzBdKSwKICAgICAgICAgICAgICAgIGFicyhwc1syXSAt")
parts.append("IHBzWzFdKSwKICAgICAgICAgICAgKQogICAgICAgICAgICBtYXhfZ2FwID0gTm9uZQogICAgICAgICAgICBpZiBucC5pc2Zpbml0ZShzcGFjaW5nKSBhbmQgc3BhY2luZyA+IDA6CiAgICAgICAgICAgICAgICBtYXhfZ2FwID0gc3BhY2luZyAqIG1heF9nYXBfZmFj")
parts.append("dG9yCiAgICAgICAgICAgIGlmIGFic19nYXBfY2FwIGlzIG5vdCBOb25lOgogICAgICAgICAgICAgICAgbWF4X2dhcCA9IG1pbihtYXhfZ2FwLCBhYnNfZ2FwX2NhcCkgaWYgbWF4X2dhcCBlbHNlIGFic19nYXBfY2FwCiAgICAgICAgICAgIGlmIG1heF9nYXAgaXMg")
parts.append("bm90IE5vbmUgYW5kIChnYXBzWzBdID4gbWF4X2dhcCBvciBnYXBzWzFdID4gbWF4X2dhcCk6CiAgICAgICAgICAgICAgICBTVEFUU1sidHJpcGxldF9nYXBfcmVsYXhlZCJdICs9IDEKICAgICAgICAgICAgICAgIHJlbGF4ZWQgPSBUcnVlCiAgICAgICAgICAgIGVs")
parts.append("c2U6CiAgICAgICAgICAgICAgICByZWxheGVkID0gRmFsc2UKICAgICAgICBjaGFuID0gW10KICAgICAgICBvayA9IFRydWUKICAgICAgICBmb3IgayBpbiBpZHhfdHJpcDoKICAgICAgICAgICAgdHJ5OgogICAgICAgICAgICAgICAgZGNtID0gZGNtX3JlYWRfZm4o")
parts.append("cGF0aHNba10pCiAgICAgICAgICAgICAgICBjaGFuLmFwcGVuZCh3aW5kb3dfZm4oZGNtLCBpbWdfc2l6ZSkpCiAgICAgICAgICAgIGV4Y2VwdCBFeGNlcHRpb246CiAgICAgICAgICAgICAgICBvayA9IEZhbHNlCiAgICAgICAgICAgICAgICBicmVhawogICAgICAg")
parts.append("IGlmIG5vdCBvazoKICAgICAgICAgICAgU1RBVFNbInRyaXBsZXRfc2tpcHBlZCJdICs9IDEKICAgICAgICAgICAgY29udGludWUKICAgICAgICBjbGlwcy5hcHBlbmQobnAuc3RhY2soY2hhbiwgYXhpcz0wKSkKICAgICAgICBTVEFUU1sidHJpcGxldF9tYWRlIl0g")
parts.append("Kz0gMQogICAgICAgIGlmIGdhcHMgYW5kIGdhcHNbMF0gaXMgbm90IE5vbmU6CiAgICAgICAgICAgIEdBUFMuYXBwZW5kKGZsb2F0KGdhcHNbMF0pKQogICAgICAgICAgICBHQVBTLmFwcGVuZChmbG9hdChnYXBzWzFdKSkKICAgICAgICBtZXRhcy5hcHBlbmQoCiAg")
parts.append("ICAgICAgICAgIHsKICAgICAgICAgICAgICAgICJjZW50ZXIiOiBjLAogICAgICAgICAgICAgICAgImlkeCI6IGlkeF90cmlwLAogICAgICAgICAgICAgICAgInBvcyI6IHBzLAogICAgICAgICAgICAgICAgImdhcHMiOiBnYXBzLAogICAgICAgICAgICAgICAgInNw")
parts.append("YWNpbmciOiBzcGFjaW5nLAogICAgICAgICAgICAgICAgInJlbGF4ZWQiOiByZWxheGVkLAogICAgICAgICAgICB9CiAgICAgICAgKQogICAgIyBzZXJpZXMtbGV2ZWwgYWNjdW11bGF0b3JzCiAgICBDTElQU19QRVJfU0VSSUVTLmFwcGVuZChsZW4oY2xpcHMpKQog")
parts.append("ICAgVVNFRF9TTElDRVNfUEVSX1NFUklFUy5hcHBlbmQobGVuKGNsaXBzKSAqIDMpCiAgICBmaW5pdGVfcG9zID0gW3AgZm9yIF8sIHAgaW4gbm9ybSBpZiBwIGlzIG5vdCBOb25lIGFuZCBucC5pc2Zpbml0ZShwKV0KICAgIGlmIGxlbihmaW5pdGVfcG9zKSA+PSAy")
parts.append("OgogICAgICAgIFNQQU5TLmFwcGVuZChmbG9hdChtYXgoZmluaXRlX3BvcykgLSBtaW4oZmluaXRlX3BvcykpKQogICAgcmV0dXJuIGNsaXBzLCBtZXRhcwo=")
open('/kaggle/working/sampling_geometry.py','wb').write(base64.b64decode(''.join(parts)))
print('wrote sampling_geometry.py', os.path.getsize('/kaggle/working/sampling_geometry.py'))
open('/kaggle/working/rsna_report_labels.py','w').write('# stub\n')
print('embeds ready')


In [ ]:
import os, glob
need = ['llm_labels_v4_blend.csv', 'labels_llm_gpt56sol.csv', 'report_labels_v2.csv']
for fn in need:
    hits = glob.glob(f'/kaggle/input/**/{fn}', recursive=True)
    print(fn, hits)
    assert hits, f'missing {fn}'
os.environ['EXP013_MULTISOURCE'] = '1'
print('EXP013_MULTISOURCE=1')


In [ ]:
import subprocess, sys, os, json, time
os.chdir('/kaggle/working')
os.environ['EXP013_MULTISOURCE'] = '1'
# record pre-run config
import hashlib
def sha(p):
    h = hashlib.sha256()
    h.update(open(p,'rb').read())
    return h.hexdigest()
cfg = {
  'experiment_id': 'exp_016_physical_axial',
  'code_sha_rsna_v3': sha('rsna_v3.py'),
  'code_sha_sampling_geometry': sha('sampling_geometry.py'),
  'label_sha': sha('exp013_multisource.py'),
  'sampling_mode': 'physical',
  'seed': 42, 'fold': 0, 'planes': 'axial',
  'n_clips': 4, 'resolution': 224, 'epochs': 3, 'batch': 8,
  'loss': 'conf-weight w=2*|y-0.5|',
  'labels': 'EXP-013 multi-source',
  'control': 'EXP-013 exp_013_multisource_axial_fold0 public 0.716 (legacy)',
}
json.dump(cfg, open('exp016_config.json','w'), indent=2)
print('CONFIG', json.dumps(cfg, indent=2), flush=True)

cmd = [sys.executable, '-u', 'rsna_v3.py',
       '--planes', 'axial',
       '--epochs', '3', '--folds', '5', '--only-fold', '0',
       '--batch', '8', '--n-clips', '4',
       '--sampling-mode', 'physical',
       '--experiment-id', 'exp_016_physical_axial',
       '--verbose']
print('=== EXP-016 physical Axial fold0 ===', flush=True)
print(' '.join(cmd), flush=True)
t0 = time.time()
r = subprocess.run(cmd)
print('returncode', r.returncode, 'wall_sec', time.time()-t0, flush=True)

import numpy as np, pandas as pd
art = '/kaggle/working/artifacts/exp_016_physical_axial/fold0'
ok = all(os.path.isfile(os.path.join(art, n)) for n in ['test_pred.npy','test_uid.csv','meta.json'])
wok = os.path.isfile(os.path.join(art, 'Axial', 'model.pt'))
print('artifact meta', ok, 'weights', wok)
if os.path.isfile(os.path.join(art, 'meta.json')):
    print('meta', json.load(open(os.path.join(art,'meta.json'))))
sp = '/kaggle/working/artifacts/exp_016_physical_axial/sampling_stats.json'
if os.path.isfile(sp):
    print('SAMPLING_STATS', open(sp).read())
else:
    print('SAMPLING_STATS NOT MEASURED')
if os.path.isfile('/kaggle/working/submission.csv'):
    s = pd.read_csv('/kaggle/working/submission.csv')
    print('[sub]', s.shape, float(s.iloc[:,1:].to_numpy().mean()))
print('[exp016] train done — NOT submitting')
